In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# =============================================================================
# ASSUMPTIONS
# =============================================================================
SOLIDS_MASS_FRACTION = 0.50      # assumed 50 wt% solids
NACN_SOLUTION_STRENGTH = 0.30    # assumed 30 wt% NaCN dosing solution
PROCESS_LIQUID_DENSITY = 1.0              # t/m3
SOLIDS_DENSITY = 2.7             # t/m3, assumed dry solids density
N_TANKS = 8
TANK_VOLUME_M3 = 2987
TOTAL_CIRCUIT_VOLUME_M3 = N_TANKS * TANK_VOLUME_M3

# Density-based liquid holdup fraction in the circuit at the assumed slurry wt% solids
SOLIDS_VOLUME_PER_TONNE_DRY = 1.0 / SOLIDS_DENSITY
LIQUID_MASS_PER_TONNE_DRY = (1.0 - SOLIDS_MASS_FRACTION) / SOLIDS_MASS_FRACTION
LIQUID_VOLUME_PER_TONNE_DRY = LIQUID_MASS_PER_TONNE_DRY / PROCESS_LIQUID_DENSITY
SLURRY_VOLUME_PER_TONNE_DRY = SOLIDS_VOLUME_PER_TONNE_DRY + LIQUID_VOLUME_PER_TONNE_DRY

LIQUID_HOLDUP_FRACTION = LIQUID_VOLUME_PER_TONNE_DRY / SLURRY_VOLUME_PER_TONNE_DRY
LIQUID_INVENTORY_M3 = TOTAL_CIRCUIT_VOLUME_M3 * LIQUID_HOLDUP_FRACTION

# =============================================================================
# COLUMN DEFINITIONS
# =============================================================================
op_data_cols = [
	"date",
	"throughput_tpd",
	"au_feed_gpt",
	"ag_feed_gpt",
	"cu_feed_ppm",
	"au_tail_gpt",
	"ag_tail_gpt",
	"nacn_consumption_tpd",
	"specific_nacn_kgpt",
	"tailings_moisture_pct",
	"do_ag1",
	"do_ag2",
	"do_ag3",
	"do_ag4",
	"do_ag5",
	"do_ag6",
	"do_ag7",
	"do_ag8",
]

leach_2025_cols = [
	"date",
	"time",
	"au_ppm_tk_1_e",
	"au_ppm_tk_1_s",
	"au_ppm_tk_6_s",
	"au_ppm_tk_8_s",
	"ag_ppm_tk_1_e",
	"ag_ppm_tk_1_s",
	"ag_ppm_tk_6_s",
	"ag_ppm_tk_8_s",
	"cu_ppm_tk_1_e",
	"cu_ppm_tk_1_s",
	"cu_ppm_tk_6_s",
	"cu_ppm_tk_8_s",
	"zn_ppm_tk_1_e",
	"zn_ppm_tk_1_s",
	"zn_ppm_tk_6_s",
	"zn_ppm_tk_8_s",
	"ph_tk_1_e",
	"ph_tk_1_s",
	"ph_tk_6_s",
	"ph_tk_8_s",
	"free_cn_ppm_tk_1_e",
	"free_cn_ppm_tk_1_s",
	"free_cn_ppm_tk_6_s",
	"free_cn_ppm_tk_8_s",
	"wad_gpl_tk_1_e",
	"wad_gpl_tk_1_s",
	"wad_gpl_tk_6_s",
	"wad_gpl_tk_8_s",
]

leach_2026_cols = [
	"date",
	"time",
	"au_ppm_tk_1_e",
	"au_ppm_tk_1_s",
	"au_ppm_tk_6_s",
	"au_ppm_tk_8_s",
	"ag_ppm_tk_1_e",
	"ag_ppm_tk_1_s",
	"ag_ppm_tk_6_s",
	"ag_ppm_tk_8_s",
	"cu_ppm_tk_1_e",
	"cu_ppm_tk_1_s",
	"cu_ppm_tk_6_s",
	"cu_ppm_tk_8_s",
	"zn_ppm_tk_1_e",
	"zn_ppm_tk_1_s",
	"zn_ppm_tk_6_s",
	"zn_ppm_tk_8_s",
	"pb_ppm_tk_1_e",
	"pb_ppm_tk_8_s",
	"ph_tk_1_e",
	"ph_tk_1_s",
	"ph_tk_6_s",
	"ph_tk_8_s",
	"free_cn_ppm_tk_1_e",
	"free_cn_ppm_tk_1_s",
	"free_cn_ppm_tk_6_s",
	"free_cn_ppm_tk_8_s",
	"wad_gpl_tk_1_e",
	"wad_gpl_tk_1_s",
	"wad_gpl_tk_6_s",
	"wad_gpl_tk_8_s",
]

# =============================================================================
# HELPERS
# =============================================================================
def coerce_numeric(df: pd.DataFrame, exclude=None) -> pd.DataFrame:
	exclude = set(exclude or [])
	for col in df.columns:
		if col not in exclude:
			df[col] = pd.to_numeric(df[col], errors="coerce")
	return df

def clean_operational_data(file_path: Path) -> pd.DataFrame:
	df = pd.read_excel(
		file_path,
		sheet_name="Operational Data",
		skiprows=4,
		usecols="A:R",
		header=None,
	)
	df.columns = op_data_cols
	df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
	df = df.dropna(how="all")
	df = df[df["date"].notna()].copy()
	df = coerce_numeric(df, exclude=["date"])

	numeric_cols = [c for c in df.columns if c != "date"]
	df = (
		df.groupby("date", as_index=False)[numeric_cols]
		.mean()
		.sort_values("date")
		.reset_index(drop=True)
	)
	return df

def clean_leach_sheet(
	file_path: Path,
	sheet_name: str,
	col_names: list[str],
	usecols: str,
) -> pd.DataFrame:
	df = pd.read_excel(
		file_path,
		sheet_name=sheet_name,
		skiprows=2,
		usecols=usecols,
		header=None,
	)
	df.columns = col_names
	df = df.dropna(how="all").copy()

	# Dates are only populated on first row for each day
	df["date"] = pd.to_datetime(df["date"], errors="coerce")
	df["date"] = df["date"].ffill().dt.normalize()

	# Keep time tidy, though we aggregate to daily
	df["time"] = pd.to_datetime(df["time"], errors="coerce").dt.time

	df = df[df["date"].notna()].copy()
	df = coerce_numeric(df, exclude=["date", "time"])

	value_cols = [c for c in df.columns if c not in ["date", "time"]]
	df_daily = (
		df.groupby("date", as_index=False)[value_cols]
		.mean()
		.sort_values("date")
		.reset_index(drop=True)
	)
	return df_daily

# =============================================================================
# LOAD + CLEAN
# =============================================================================
file_path = Path("leachit.xlsx")

df_op_data = clean_operational_data(file_path)

df_leach_2025_daily = clean_leach_sheet(
	file_path=file_path,
	sheet_name="Leaching 2025",
	col_names=leach_2025_cols,
	usecols="A:AD",
)

df_leach_2026_daily = clean_leach_sheet(
	file_path=file_path,
	sheet_name="Leaching 2026",
	col_names=leach_2026_cols,
	usecols="A:AF",
)

# =============================================================================
# COMBINE LEACH SHEETS
# =============================================================================
df_leach_daily = pd.concat(
	[df_leach_2025_daily, df_leach_2026_daily],
	ignore_index=True,
	sort=False,
)

leach_value_cols = [c for c in df_leach_daily.columns if c != "date"]
df_leach_daily = (
	df_leach_daily.groupby("date", as_index=False)[leach_value_cols]
	.mean()
	.sort_values("date")
	.reset_index(drop=True)
)

# =============================================================================
# MERGE TO DAILY MASTER
# =============================================================================
df = (
	df_op_data.merge(df_leach_daily, on="date", how="left")
	.sort_values("date")
	.reset_index(drop=True)
)

# =============================================================================
# DERIVED METRICS
# =============================================================================
df["year"] = df["date"].dt.year

# Recovery
df["recovery_au_pct"] = np.where(
	df["au_feed_gpt"] > 0,
	((df["au_feed_gpt"] - df["au_tail_gpt"]) / df["au_feed_gpt"]) * 100,
	np.nan,
)

# Average DO
do_cols = [c for c in df.columns if c.startswith("do_ag")]
df["do_avg"] = df[do_cols].mean(axis=1)

# Solution averages
for el in ["au", "ag", "cu", "zn"]:
	cols = [c for c in df.columns if c.startswith(f"{el}_ppm_tk_")]
	df[f"{el}_solution_ppm_avg"] = df[cols].mean(axis=1)

free_cols = [c for c in df.columns if c.startswith("free_cn_ppm_tk_")]
wad_cols = [c for c in df.columns if c.startswith("wad_gpl_tk_")]

df["free_cn_ppm_avg"] = df[free_cols].mean(axis=1)
df["wad_gpl_avg"] = df[wad_cols].mean(axis=1)
df["wad_ppm_avg"] = df["wad_gpl_avg"] * 1000

# Approximate complexed cyanide = WAD - free
# Note: free CN is ppm ~= mg/L, so divide by 1000 to convert to g/L
df["complexed_cn_gpl_avg"] = df["wad_gpl_avg"] - (df["free_cn_ppm_avg"] / 1000)

# Operating-only subset
dfo = df[df["throughput_tpd"] > 0].copy()

# =============================================================================
# BASIC SUMMARY
# =============================================================================
summary = pd.Series({
	"n_days_total": len(df),
	"n_days_operating": len(dfo),
	"date_min": df["date"].min(),
	"date_max": df["date"].max(),
	"throughput_tpd_mean": dfo["throughput_tpd"].mean(),
	"nacn_consumption_tpd_mean": dfo["nacn_consumption_tpd"].mean(),
	"specific_nacn_kgpt_mean": dfo["specific_nacn_kgpt"].mean(),
	"cu_feed_ppm_mean": dfo["cu_feed_ppm"].mean(),
	"cu_solution_ppm_avg_mean": dfo["cu_solution_ppm_avg"].mean(),
	"free_cn_ppm_avg_mean": dfo["free_cn_ppm_avg"].mean(),
	"wad_gpl_avg_mean": dfo["wad_gpl_avg"].mean(),
	"complexed_cn_gpl_avg_mean": dfo["complexed_cn_gpl_avg"].mean(),
	"recovery_au_pct_mean": dfo["recovery_au_pct"].mean(),
})
print(summary)

# =============================================================================
# YEAR-ON-YEAR SUMMARY
# =============================================================================
year_summary = dfo.groupby("year").agg(
	days=("date", "count"),
	throughput_tpd_mean=("throughput_tpd", "mean"),
	nacn_tpd_mean=("nacn_consumption_tpd", "mean"),
	specific_nacn_kgpt_mean=("specific_nacn_kgpt", "mean"),
	cu_feed_ppm_mean=("cu_feed_ppm", "mean"),
	cu_solution_ppm_avg_mean=("cu_solution_ppm_avg", "mean"),
	free_cn_ppm_avg_mean=("free_cn_ppm_avg", "mean"),
	wad_gpl_avg_mean=("wad_gpl_avg", "mean"),
	complexed_cn_gpl_avg_mean=("complexed_cn_gpl_avg", "mean"),
	recovery_au_pct_mean=("recovery_au_pct", "mean"),
	ph8_mean=("ph_tk_8_s", "mean"),
).round(2)

print(year_summary)

# =============================================================================
# QUANTILES
# =============================================================================
quantiles = dfo[
	[
		"throughput_tpd",
		"nacn_consumption_tpd",
		"specific_nacn_kgpt",
		"cu_feed_ppm",
		"cu_solution_ppm_avg",
		"free_cn_ppm_avg",
		"wad_gpl_avg",
		"complexed_cn_gpl_avg",
		"recovery_au_pct",
	]
].quantile([0.05, 0.25, 0.50, 0.75, 0.95]).round(2)

print(quantiles)

# =============================================================================
# CORRELATIONS
# =============================================================================
drivers = [
	"cu_feed_ppm",
	"cu_solution_ppm_avg",
	"zn_solution_ppm_avg",
	"ag_feed_gpt",
	"au_feed_gpt",
	"throughput_tpd",
	"do_avg",
	"ph_tk_8_s",
]

targets = [
	"nacn_consumption_tpd",
	"specific_nacn_kgpt",
	"free_cn_ppm_avg",
	"wad_gpl_avg",
	"complexed_cn_gpl_avg",
	"recovery_au_pct",
]

for target in targets:
	corr = (
		dfo[drivers + [target]]
		.corr(numeric_only=True)[target]
		.sort_values(ascending=False)
		.round(3)
	)
	print(f"\n=== Correlations with {target} ===")
	print(corr)

# Year-specific correlations
for yr in sorted(dfo["year"].dropna().unique()):
	sub = dfo[dfo["year"] == yr]
	print(f"\n\n######## YEAR {yr} ########")
	for target in [
		"specific_nacn_kgpt",
		"nacn_consumption_tpd",
		"complexed_cn_gpl_avg",
		"free_cn_ppm_avg",
		"recovery_au_pct",
	]:
		corr = (
			sub[drivers + [target]]
			.corr(numeric_only=True)[target]
			.sort_values(ascending=False)
			.round(3)
		)
		print(f"\n--- {target} ---")
		print(corr)

# =============================================================================
# TANK 1 INLET / OUTLET CHECK
# TK-1 E = entrada (entry/inlet)
# TK-1 S = salida (exit/outlet)
# =============================================================================
for var in ["au_ppm", "ag_ppm", "cu_ppm", "zn_ppm", "ph", "free_cn_ppm", "wad_gpl"]:
	e = f"{var}_tk_1_e"
	s = f"{var}_tk_1_s"
	if e in dfo.columns and s in dfo.columns:
		delta = (dfo[s] - dfo[e]).describe()[["mean", "50%", "min", "max"]].round(3)
		print(f"\nTank 1 change: {s} - {e}")
		print(delta)

# =============================================================================
# MASS BALANCE / CIRCUIT LOADS
# IMPORTANT:
# This is an indicative circuit balance only, not a full plant-closed balance.
# Residence time is estimated from nominal tank volume and density-based slurry
# volumetric flow using assumed wt% solids, solids density, and water density.
# =============================================================================

# Treat throughput_tpd as dry solids throughput, then add process liquid from assumed wt% solids
dfo["solids_tpd"] = dfo["throughput_tpd"]
dfo["liquid_tpd"] = dfo["throughput_tpd"] * (1 - SOLIDS_MASS_FRACTION) / SOLIDS_MASS_FRACTION

# Convert mass flows to volumetric flows using material densities
dfo["solids_flow_m3d"] = dfo["solids_tpd"] / SOLIDS_DENSITY
dfo["liquid_flow_m3d"] = dfo["liquid_tpd"] / PROCESS_LIQUID_DENSITY

# Total slurry volumetric flow
dfo["slurry_flow_m3d"] = dfo["solids_flow_m3d"] + dfo["liquid_flow_m3d"]

# Hydraulic residence time
dfo["rt_days"] = TOTAL_CIRCUIT_VOLUME_M3 / dfo["slurry_flow_m3d"]
dfo["rt_hours"] = dfo["rt_days"] * 24

# NaCN solution required at 30 wt%
dfo["nacn_solution_tpd_30pct"] = dfo["nacn_consumption_tpd"] / NACN_SOLUTION_STRENGTH

# In-circuit inventories based on average sampled concentrations
# free CN ppm = mg/L; inventory_t = ppm * m3 / 1e6
dfo["free_cn_inventory_t"] = dfo["free_cn_ppm_avg"] * LIQUID_INVENTORY_M3 / 1e6

# WAD g/L; inventory_t = g/L * m3 / 1000
dfo["wad_inventory_t"] = dfo["wad_gpl_avg"] * LIQUID_INVENTORY_M3 / 1000

# Estimated complexed inventory
dfo["complexed_inventory_t"] = dfo["wad_inventory_t"] - dfo["free_cn_inventory_t"]

# Apparent outlet loads using Tank 8 solution and estimated liquid flow
# free CN ppm = mg/L -> kg/d = ppm * m3/d / 1000
dfo["free_cn_tk8_kgd"] = dfo["free_cn_ppm_tk_8_s"] * dfo["liquid_flow_m3d"] / 1000.0

# WAD g/L -> kg/d = g/L * m3/d
dfo["wad_tk8_kgd"] = dfo["wad_gpl_tk_8_s"] * dfo["liquid_flow_m3d"]

dfo["complexed_cn_tk8_kgd"] = dfo["wad_tk8_kgd"] - dfo["free_cn_tk8_kgd"]

mass_balance_summary = pd.Series({
	"total_circuit_volume_m3": TOTAL_CIRCUIT_VOLUME_M3,
	"liquid_inventory_m3": LIQUID_INVENTORY_M3,
	"mean_rt_hours": dfo["rt_hours"].mean(),
	"mean_nacn_input_tpd": dfo["nacn_consumption_tpd"].mean(),
	"median_nacn_input_tpd": dfo["nacn_consumption_tpd"].median(),
	"mean_30pct_nacn_solution_tpd": dfo["nacn_solution_tpd_30pct"].mean(),
	"mean_free_cn_inventory_t": dfo["free_cn_inventory_t"].mean(),
	"mean_wad_inventory_t": dfo["wad_inventory_t"].mean(),
	"mean_complexed_inventory_t": dfo["complexed_inventory_t"].mean(),
}).round(3)

print("\nMass balance / circuit-load summary")
print(mass_balance_summary)

# =============================================================================
# COPPER DECILES
# =============================================================================
dfo["cu_sol_decile"] = pd.qcut(dfo["cu_solution_ppm_avg"], 10, duplicates="drop")

cu_deciles = dfo.groupby("cu_sol_decile").agg(
	n=("date", "count"),
	cu_solution_ppm_avg=("cu_solution_ppm_avg", "mean"),
	nacn_consumption_tpd=("nacn_consumption_tpd", "mean"),
	specific_nacn_kgpt=("specific_nacn_kgpt", "mean"),
	complexed_cn_gpl_avg=("complexed_cn_gpl_avg", "mean"),
	free_cn_ppm_avg=("free_cn_ppm_avg", "mean"),
	recovery_au_pct=("recovery_au_pct", "mean"),
).round(2)

print("\nCopper deciles")
print(cu_deciles)

# =============================================================================
# LAG CHECK
# Does feed Cu lead solution Cu / CN consumption by a few days?
# =============================================================================
rows = []
for lag in range(0, 8):
	tmp = dfo.copy()
	tmp[f"cu_feed_lag_{lag}"] = tmp["cu_feed_ppm"].shift(lag)

	for target in [
		"cu_solution_ppm_avg",
		"nacn_consumption_tpd",
		"specific_nacn_kgpt",
		"complexed_cn_gpl_avg",
	]:
		corr = tmp[[f"cu_feed_lag_{lag}", target]].corr().iloc[0, 1]
		rows.append({
			"lag_days": lag,
			"target": target,
			"corr": corr,
		})

lag_df = pd.DataFrame(rows)
lag_pivot = lag_df.pivot(index="lag_days", columns="target", values="corr").round(3)

print("\nLag correlation table")
print(lag_pivot)

# =============================================================================
# PLOTTING + DIAGNOSTIC ANALYSIS
# =============================================================================

# -----------------------------------------------------------------------------
# 0. PREP
# -----------------------------------------------------------------------------
dfo_plot = dfo.copy()

def iqr_filter(df, cols, k=1.5):
    df_filtered = df.copy()

    for col in cols:
        if col not in df_filtered.columns:
            continue

        q1 = df_filtered[col].quantile(0.25)
        q3 = df_filtered[col].quantile(0.75)
        iqr = q3 - q1

        lower = q1 - k * iqr
        upper = q3 + k * iqr

        df_filtered = df_filtered[
            (df_filtered[col] >= lower) & (df_filtered[col] <= upper)
        ]

    return df_filtered

# --- FILTER OUTLIERS ---
filter_cols = [
    "nacn_consumption_tpd",
    "specific_nacn_kgpt",
    "cu_solution_ppm_avg",
    "free_cn_ppm_avg",
    "complexed_cn_gpl_avg",
    "recovery_au_pct",
]

dfo_plot = iqr_filter(dfo_plot, filter_cols, k=1.5)

dfo_plot["date"] = pd.to_datetime(dfo_plot["date"])
dfo_plot = dfo_plot.sort_values("date").reset_index(drop=True)

# Defensive year creation if not already present
if "year" not in dfo_plot.columns:
    dfo_plot["year"] = dfo_plot["date"].dt.year

# Helpful hover fields
hover_cols = [
    c for c in [
        "date",
        "year",
        "nacn_consumption_tpd",
        "specific_nacn_kgpt",
        "cu_feed_ppm",
        "cu_solution_ppm_avg",
        "free_cn_ppm_avg",
        "wad_gpl_avg",
        "complexed_cn_gpl_avg",
        "recovery_au_pct",
    ] if c in dfo_plot.columns
]

# Colour system
COLORS = {
    "nacn": "#1f77b4",
    "cu_feed": "#9467bd",
    "cu_sol": "#ff7f0e",
    "free_cn": "#2ca02c",
    "wad": "#d62728",
    "complexed": "#8c564b",
    "recovery": "#17becf",
    "neutral": "#7f7f7f",
    "bg": "#ffffff",
    "grid": "#e5e7eb",
    "text": "#1f2937",
}

PLOTLY_TEMPLATE = "plotly_white"

def apply_professional_layout(
    fig,
    title,
    height=900,
    legend_orientation="h",
    showlegend=True,
):
    fig.update_layout(
        template=PLOTLY_TEMPLATE,
        title=dict(
            text=title,
            x=0.01,
            xanchor="left",
            font=dict(size=24, color=COLORS["text"]),
        ),
        paper_bgcolor=COLORS["bg"],
        plot_bgcolor=COLORS["bg"],
        font=dict(size=13, color=COLORS["text"]),
        hovermode="x unified",
        height=height,
        margin=dict(l=70, r=30, t=80, b=60),
        legend=dict(
            orientation=legend_orientation,
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1.0,
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="rgba(0,0,0,0.08)",
            borderwidth=1,
            font=dict(size=12),
        ),
        showlegend=showlegend,
    )

    fig.update_xaxes(
        showgrid=True,
        gridcolor=COLORS["grid"],
        zeroline=False,
        showline=True,
        linecolor="rgba(0,0,0,0.15)",
        tickfont=dict(size=12),
    )
    fig.update_yaxes(
        showgrid=True,
        gridcolor=COLORS["grid"],
        zeroline=False,
        showline=True,
        linecolor="rgba(0,0,0,0.15)",
        tickfont=dict(size=12),
    )
    return fig


def add_range_selector(fig, row=None, col=None):
    
    if row is not None and col is not None:
        # subplot xaxis numbering is automatic; for shared x this still works visually
        pass

    fig.update_xaxes(
        rangeslider_visible=False,
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1m", step="month", stepmode="backward"),
                dict(count=3, label="3m", step="month", stepmode="backward"),
                dict(count=6, label="6m", step="month", stepmode="backward"),
                dict(label="YTD", step="year", stepmode="todate"),
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(label="All", step="all"),
            ])
        )
    )
    return fig


def add_regression_line(fig, x, y, name, color):
    """
    Adds a simple OLS line without external sklearn dependency.
    """
    mask = pd.notna(x) & pd.notna(y)
    x_clean = pd.Series(x)[mask].astype(float)
    y_clean = pd.Series(y)[mask].astype(float)

    if len(x_clean) < 2:
        return fig

    slope, intercept = np.polyfit(x_clean, y_clean, 1)
    x_line = np.linspace(x_clean.min(), x_clean.max(), 100)
    y_line = slope * x_line + intercept

    fig.add_trace(
        go.Scatter(
            x=x_line,
            y=y_line,
            mode="lines",
            name=name,
            line=dict(color=color, width=3, dash="solid"),
            hoverinfo="skip",
        )
    )
    return fig

# -----------------------------------------------------------------------------
# 1. TIME SERIES OVERVIEW (INTERACTIVE MULTI-PANEL)
# -----------------------------------------------------------------------------
fig_ts = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=(
        "NaCN Consumption",
        "Copper in Feed vs Solution",
        "Free CN vs WAD CN",
        "Gold Recovery",
    ),
)

# Row 1
fig_ts.add_trace(
    go.Scatter(
        x=dfo_plot["date"],
        y=dfo_plot["nacn_consumption_tpd"],
        mode="lines",
        name="NaCN Consumption (t/d)",
        line=dict(color=COLORS["nacn"], width=2.5),
        hovertemplate="<b>%{x|%d %b %Y}</b><br>NaCN: %{y:.2f} t/d<extra></extra>",
    ),
    row=1, col=1
)

# Optional rolling trend
fig_ts.add_trace(
    go.Scatter(
        x=dfo_plot["date"],
        y=dfo_plot["nacn_consumption_tpd"].rolling(14, min_periods=3).mean(),
        mode="lines",
        name="NaCN 14d rolling",
        line=dict(color=COLORS["nacn"], width=3, dash="dash"),
        opacity=0.7,
        hovertemplate="<b>%{x|%d %b %Y}</b><br>14d rolling: %{y:.2f} t/d<extra></extra>",
    ),
    row=1, col=1
)

# Row 2
fig_ts.add_trace(
    go.Scatter(
        x=dfo_plot["date"],
        y=dfo_plot["cu_feed_ppm"],
        mode="lines",
        name="Feed Cu (ppm)",
        line=dict(color=COLORS["cu_feed"], width=2),
        hovertemplate="<b>%{x|%d %b %Y}</b><br>Feed Cu: %{y:,.0f} ppm<extra></extra>",
    ),
    row=2, col=1
)

fig_ts.add_trace(
    go.Scatter(
        x=dfo_plot["date"],
        y=dfo_plot["cu_solution_ppm_avg"],
        mode="lines",
        name="Solution Cu Avg (ppm)",
        line=dict(color=COLORS["cu_sol"], width=2.5),
        hovertemplate="<b>%{x|%d %b %Y}</b><br>Solution Cu: %{y:,.0f} ppm<extra></extra>",
    ),
    row=2, col=1
)

# Row 3
fig_ts.add_trace(
    go.Scatter(
        x=dfo_plot["date"],
        y=dfo_plot["free_cn_ppm_avg"],
        mode="lines",
        name="Free CN Avg (ppm)",
        line=dict(color=COLORS["free_cn"], width=2.5),
        hovertemplate="<b>%{x|%d %b %Y}</b><br>Free CN: %{y:,.0f} ppm<extra></extra>",
    ),
    row=3, col=1
)

fig_ts.add_trace(
    go.Scatter(
        x=dfo_plot["date"],
        y=dfo_plot["wad_gpl_avg"] * 1000,
        mode="lines",
        name="WAD Avg (ppm equiv.)",
        line=dict(color=COLORS["wad"], width=2.5),
        hovertemplate="<b>%{x|%d %b %Y}</b><br>WAD: %{y:,.0f} ppm equiv.<extra></extra>",
    ),
    row=3, col=1
)

# Row 4
fig_ts.add_trace(
    go.Scatter(
        x=dfo_plot["date"],
        y=dfo_plot["recovery_au_pct"],
        mode="lines",
        name="Au Recovery (%)",
        line=dict(color=COLORS["recovery"], width=2.5),
        hovertemplate="<b>%{x|%d %b %Y}</b><br>Recovery: %{y:.1f}%<extra></extra>",
    ),
    row=4, col=1
)

fig_ts.update_yaxes(title_text="t/d", row=1, col=1)
fig_ts.update_yaxes(title_text="ppm", row=2, col=1)
fig_ts.update_yaxes(title_text="ppm", row=3, col=1)
fig_ts.update_yaxes(title_text="%", row=4, col=1)
fig_ts.update_xaxes(title_text="Date", row=4, col=1)

apply_professional_layout(fig_ts, "Leach Circuit Overview", height=1150)
add_range_selector(fig_ts)
fig_ts.show()


# -----------------------------------------------------------------------------
# 2. COPPER VS CYANIDE CONSUMPTION (INTERACTIVE SCATTER + OLS)
# -----------------------------------------------------------------------------
fig_scatter_1 = make_subplots(
    rows=1,
    cols=3,
    horizontal_spacing=0.08,
    subplot_titles=(
        "Feed Cu vs NaCN Consumption",
        "Solution Cu vs NaCN Consumption",
        "Solution Cu vs Specific NaCN",
    ),
)

# 1
fig_scatter_1.add_trace(
    go.Scatter(
        x=dfo_plot["cu_feed_ppm"],
        y=dfo_plot["nacn_consumption_tpd"],
        mode="markers",
        name="Data",
        marker=dict(
            size=9,
            color=dfo_plot["year"],
            colorscale="Blues",
            showscale=False,
            line=dict(width=0.5, color="rgba(0,0,0,0.2)"),
            opacity=0.75,
        ),
        customdata=dfo_plot[["date", "cu_solution_ppm_avg", "specific_nacn_kgpt"]].values,
        hovertemplate=(
            "<b>%{customdata[0]|%d %b %Y}</b><br>"
            "Feed Cu: %{x:,.0f} ppm<br>"
            "NaCN: %{y:.2f} t/d<br>"
            "Solution Cu: %{customdata[1]:,.0f} ppm<br>"
            "Specific NaCN: %{customdata[2]:.2f} kg/t"
            "<extra></extra>"
        ),
    ),
    row=1, col=1
)
add_regression_line(
    fig_scatter_1,
    dfo_plot["cu_feed_ppm"],
    dfo_plot["nacn_consumption_tpd"],
    "Trend",
    COLORS["nacn"]
)

# 2
fig_scatter_1.add_trace(
    go.Scatter(
        x=dfo_plot["cu_solution_ppm_avg"],
        y=dfo_plot["nacn_consumption_tpd"],
        mode="markers",
        name="Data ",
        marker=dict(
            size=9,
            color=dfo_plot["year"],
            colorscale="Oranges",
            showscale=False,
            line=dict(width=0.5, color="rgba(0,0,0,0.2)"),
            opacity=0.75,
        ),
        customdata=dfo_plot[["date", "cu_feed_ppm", "specific_nacn_kgpt"]].values,
        hovertemplate=(
            "<b>%{customdata[0]|%d %b %Y}</b><br>"
            "Solution Cu: %{x:,.0f} ppm<br>"
            "NaCN: %{y:.2f} t/d<br>"
            "Feed Cu: %{customdata[1]:,.0f} ppm<br>"
            "Specific NaCN: %{customdata[2]:.2f} kg/t"
            "<extra></extra>"
        ),
    ),
    row=1, col=2
)
mask = dfo_plot["cu_solution_ppm_avg"].notna() & dfo_plot["nacn_consumption_tpd"].notna()
x = dfo_plot.loc[mask, "cu_solution_ppm_avg"]
y = dfo_plot.loc[mask, "nacn_consumption_tpd"]
if len(x) > 1:
    slope, intercept = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = slope * x_line + intercept
    fig_scatter_1.add_trace(
        go.Scatter(
            x=x_line, y=y_line, mode="lines",
            name="Trend ",
            line=dict(color=COLORS["cu_sol"], width=3),
            hoverinfo="skip"
        ),
        row=1, col=2
    )

# 3
fig_scatter_1.add_trace(
    go.Scatter(
        x=dfo_plot["cu_solution_ppm_avg"],
        y=dfo_plot["specific_nacn_kgpt"],
        mode="markers",
        name="Data  ",
        marker=dict(
            size=9,
            color=dfo_plot["year"],
            colorscale="Purples",
            showscale=False,
            line=dict(width=0.5, color="rgba(0,0,0,0.2)"),
            opacity=0.75,
        ),
        customdata=dfo_plot[["date", "nacn_consumption_tpd", "cu_feed_ppm"]].values,
        hovertemplate=(
            "<b>%{customdata[0]|%d %b %Y}</b><br>"
            "Solution Cu: %{x:,.0f} ppm<br>"
            "Specific NaCN: %{y:.2f} kg/t<br>"
            "NaCN: %{customdata[1]:.2f} t/d<br>"
            "Feed Cu: %{customdata[2]:,.0f} ppm"
            "<extra></extra>"
        ),
    ),
    row=1, col=3
)
mask = dfo_plot["cu_solution_ppm_avg"].notna() & dfo_plot["specific_nacn_kgpt"].notna()
x = dfo_plot.loc[mask, "cu_solution_ppm_avg"]
y = dfo_plot.loc[mask, "specific_nacn_kgpt"]
if len(x) > 1:
    slope, intercept = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = slope * x_line + intercept
    fig_scatter_1.add_trace(
        go.Scatter(
            x=x_line, y=y_line, mode="lines",
            name="Trend  ",
            line=dict(color="#6f42c1", width=3),
            hoverinfo="skip"
        ),
        row=1, col=3
    )

fig_scatter_1.update_xaxes(title_text="Feed Cu (ppm)", row=1, col=1)
fig_scatter_1.update_xaxes(title_text="Solution Cu Avg (ppm)", row=1, col=2)
fig_scatter_1.update_xaxes(title_text="Solution Cu Avg (ppm)", row=1, col=3)
fig_scatter_1.update_yaxes(title_text="NaCN Consumption (t/d)", row=1, col=1)
fig_scatter_1.update_yaxes(title_text="NaCN Consumption (t/d)", row=1, col=2)
fig_scatter_1.update_yaxes(title_text="Specific NaCN (kg/t)", row=1, col=3)

apply_professional_layout(fig_scatter_1, "Copper Relationship Diagnostics", height=520)
fig_scatter_1.update_layout(showlegend=False)
fig_scatter_1.show()


# -----------------------------------------------------------------------------
# 3. COPPER VS CYANIDE SPECIATION
# -----------------------------------------------------------------------------
fig_scatter_2 = make_subplots(
    rows=1,
    cols=3,
    horizontal_spacing=0.08,
    subplot_titles=(
        "Solution Cu vs Free CN",
        "Solution Cu vs WAD CN",
        "Solution Cu vs Estimated Complexed CN",
    ),
)

# Free CN
fig_scatter_2.add_trace(
    go.Scatter(
        x=dfo_plot["cu_solution_ppm_avg"],
        y=dfo_plot["free_cn_ppm_avg"],
        mode="markers",
        marker=dict(
            size=9,
            color=COLORS["free_cn"],
            line=dict(width=0.5, color="rgba(0,0,0,0.2)"),
            opacity=0.70,
        ),
        customdata=dfo_plot[["date", "wad_gpl_avg", "complexed_cn_gpl_avg"]].values,
        hovertemplate=(
            "<b>%{customdata[0]|%d %b %Y}</b><br>"
            "Solution Cu: %{x:,.0f} ppm<br>"
            "Free CN: %{y:,.0f} ppm<br>"
            "WAD: %{customdata[1]:.2f} g/L<br>"
            "Complexed CN: %{customdata[2]:.2f} g/L"
            "<extra></extra>"
        ),
        name="Free CN",
    ),
    row=1, col=1
)

# WAD
fig_scatter_2.add_trace(
    go.Scatter(
        x=dfo_plot["cu_solution_ppm_avg"],
        y=dfo_plot["wad_gpl_avg"],
        mode="markers",
        marker=dict(
            size=9,
            color=COLORS["wad"],
            line=dict(width=0.5, color="rgba(0,0,0,0.2)"),
            opacity=0.70,
        ),
        customdata=dfo_plot[["date", "free_cn_ppm_avg", "complexed_cn_gpl_avg"]].values,
        hovertemplate=(
            "<b>%{customdata[0]|%d %b %Y}</b><br>"
            "Solution Cu: %{x:,.0f} ppm<br>"
            "WAD CN: %{y:.2f} g/L<br>"
            "Free CN: %{customdata[1]:,.0f} ppm<br>"
            "Complexed CN: %{customdata[2]:.2f} g/L"
            "<extra></extra>"
        ),
        name="WAD CN",
    ),
    row=1, col=2
)

# Complexed CN
fig_scatter_2.add_trace(
    go.Scatter(
        x=dfo_plot["cu_solution_ppm_avg"],
        y=dfo_plot["complexed_cn_gpl_avg"],
        mode="markers",
        marker=dict(
            size=9,
            color=COLORS["complexed"],
            line=dict(width=0.5, color="rgba(0,0,0,0.2)"),
            opacity=0.70,
        ),
        customdata=dfo_plot[["date", "free_cn_ppm_avg", "wad_gpl_avg"]].values,
        hovertemplate=(
            "<b>%{customdata[0]|%d %b %Y}</b><br>"
            "Solution Cu: %{x:,.0f} ppm<br>"
            "Complexed CN: %{y:.2f} g/L<br>"
            "Free CN: %{customdata[1]:,.0f} ppm<br>"
            "WAD: %{customdata[2]:.2f} g/L"
            "<extra></extra>"
        ),
        name="Complexed CN",
    ),
    row=1, col=3
)

# Trend lines
for col_idx, y_col, clr in [
    (1, "free_cn_ppm_avg", COLORS["free_cn"]),
    (2, "wad_gpl_avg", COLORS["wad"]),
    (3, "complexed_cn_gpl_avg", COLORS["complexed"]),
]:
    mask = dfo_plot["cu_solution_ppm_avg"].notna() & dfo_plot[y_col].notna()
    x = dfo_plot.loc[mask, "cu_solution_ppm_avg"]
    y = dfo_plot.loc[mask, y_col]
    if len(x) > 1:
        slope, intercept = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 100)
        y_line = slope * x_line + intercept
        fig_scatter_2.add_trace(
            go.Scatter(
                x=x_line,
                y=y_line,
                mode="lines",
                line=dict(color=clr, width=3),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1, col=col_idx
        )

fig_scatter_2.update_xaxes(title_text="Solution Cu Avg (ppm)", row=1, col=1)
fig_scatter_2.update_xaxes(title_text="Solution Cu Avg (ppm)", row=1, col=2)
fig_scatter_2.update_xaxes(title_text="Solution Cu Avg (ppm)", row=1, col=3)
fig_scatter_2.update_yaxes(title_text="Free CN Avg (ppm)", row=1, col=1)
fig_scatter_2.update_yaxes(title_text="WAD CN Avg (g/L)", row=1, col=2)
fig_scatter_2.update_yaxes(title_text="Complexed CN Avg (g/L)", row=1, col=3)

apply_professional_layout(fig_scatter_2, "Copper vs Cyanide Speciation", height=520)
fig_scatter_2.update_layout(showlegend=False)
fig_scatter_2.show()


# -----------------------------------------------------------------------------
# 4. YEAR-TO-YEAR DISTRIBUTION COMPARISON
# -----------------------------------------------------------------------------
year_metrics = [
    ("nacn_consumption_tpd", "NaCN Consumption (t/d)"),
    ("specific_nacn_kgpt", "Specific NaCN (kg/t)"),
    ("cu_solution_ppm_avg", "Solution Cu (ppm)"),
    ("free_cn_ppm_avg", "Free CN (ppm)"),
    ("complexed_cn_gpl_avg", "Complexed CN (g/L)"),
    ("recovery_au_pct", "Au Recovery (%)"),
]

fig_box = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=[label for _, label in year_metrics],
    horizontal_spacing=0.08,
    vertical_spacing=0.14,
)

positions = [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3)]

for (metric, label), (r, c) in zip(year_metrics, positions):
    for yr in sorted(dfo_plot["year"].dropna().unique()):
        year_data = dfo_plot.loc[dfo_plot["year"] == yr, metric]
        fig_box.add_trace(
            go.Box(
                y=year_data,
                name=str(yr),
                boxmean=True,
                marker_color=COLORS["nacn"] if yr == min(dfo_plot["year"].dropna().unique()) else COLORS["cu_sol"],
                line=dict(width=1.2),
                opacity=0.8,
                showlegend=(metric == year_metrics[0][0]),
                hovertemplate=f"Year: {yr}<br>{label}: %{{y}}<extra></extra>",
            ),
            row=r, col=c
        )

apply_professional_layout(fig_box, "Year-on-Year Distribution Comparison", height=900)
fig_box.update_layout(boxmode="group")
fig_box.show()


# -----------------------------------------------------------------------------
# 5. TANK PROGRESSION CHECK
# -----------------------------------------------------------------------------
tank_progression = pd.DataFrame({
    "Au (ppm)": [
        dfo_plot["au_ppm_tk_1_e"].mean(),
        dfo_plot["au_ppm_tk_1_s"].mean(),
        dfo_plot["au_ppm_tk_6_s"].mean(),
        dfo_plot["au_ppm_tk_8_s"].mean(),
    ],
    "Ag (ppm)": [
        dfo_plot["ag_ppm_tk_1_e"].mean(),
        dfo_plot["ag_ppm_tk_1_s"].mean(),
        dfo_plot["ag_ppm_tk_6_s"].mean(),
        dfo_plot["ag_ppm_tk_8_s"].mean(),
    ],
    "Cu (ppm)": [
        dfo_plot["cu_ppm_tk_1_e"].mean(),
        dfo_plot["cu_ppm_tk_1_s"].mean(),
        dfo_plot["cu_ppm_tk_6_s"].mean(),
        dfo_plot["cu_ppm_tk_8_s"].mean(),
    ],
    "Free CN (ppm)": [
        dfo_plot["free_cn_ppm_tk_1_e"].mean(),
        dfo_plot["free_cn_ppm_tk_1_s"].mean(),
        dfo_plot["free_cn_ppm_tk_6_s"].mean(),
        dfo_plot["free_cn_ppm_tk_8_s"].mean(),
    ],
    "WAD (g/L)": [
        dfo_plot["wad_gpl_tk_1_e"].mean(),
        dfo_plot["wad_gpl_tk_1_s"].mean(),
        dfo_plot["wad_gpl_tk_6_s"].mean(),
        dfo_plot["wad_gpl_tk_8_s"].mean(),
    ],
    "pH": [
        dfo_plot["ph_tk_1_e"].mean(),
        dfo_plot["ph_tk_1_s"].mean(),
        dfo_plot["ph_tk_6_s"].mean(),
        dfo_plot["ph_tk_8_s"].mean(),
    ],
}, index=["TK1-E", "TK1-S", "TK6-S", "TK8-S"]).round(3)

print("Average tank progression values:")
display(tank_progression)

tank_long = (
    tank_progression
    .reset_index()
    .rename(columns={"index": "Sampling Point"})
    .melt(id_vars="Sampling Point", var_name="Metric", value_name="Value")
)

fig_tank = px.line(
    tank_long,
    x="Sampling Point",
    y="Value",
    color="Metric",
    markers=True,
    line_group="Metric",
    facet_col="Metric",
    facet_col_wrap=3,
    title="Through-Circuit Sampling Progression",
    template=PLOTLY_TEMPLATE,
)

fig_tank.update_traces(
    line=dict(width=3),
    marker=dict(size=8),
)

fig_tank.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_tank.update_yaxes(matches=None, showgrid=True, gridcolor=COLORS["grid"])
fig_tank.update_xaxes(showgrid=False)
fig_tank.update_layout(
    height=900,
    title=dict(x=0.01, xanchor="left", font=dict(size=24)),
    margin=dict(l=60, r=30, t=80, b=40),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
    ),
)
fig_tank.show()

# =============================================================================
# DIAGNOSTIC SUMMARY TABLES + POLISHED OUTPUTS
# =============================================================================

try:
    from IPython.display import display, Markdown
    IN_NOTEBOOK = True
except ImportError:
    IN_NOTEBOOK = False


# -----------------------------------------------------------------------------
# 0. CONFIG / LABELS
# -----------------------------------------------------------------------------
output_dir = Path("la_coipa_diagnostics_outputs")
output_dir.mkdir(exist_ok=True)

metric_labels = {
    "throughput_tpd": "Throughput (t/d)",
    "nacn_consumption_tpd": "NaCN consumption (t/d)",
    "specific_nacn_kgpt": "Specific NaCN (kg/t)",
    "cu_feed_ppm": "Cu feed (ppm)",
    "cu_solution_ppm_avg": "Cu in solution (ppm)",
    "free_cn_ppm_avg": "Free CN average (ppm)",
    "wad_gpl_avg": "WAD CN average (g/L)",
    "complexed_cn_gpl_avg": "Estimated complexed CN (g/L)",
    "recovery_au_pct": "Gold recovery (%)",
    "do_avg": "DO average (ppm)",
    "ph_tk_8_s": "pH TK-8",
    "rt_hours": "Residence time (h)",
    "au_feed_gpt": "Au feed (g/t)",
    "ag_feed_gpt": "Ag feed (g/t)",
    "zn_solution_ppm_avg": "Zn in solution (ppm)",
    "ph_tk_1_s": "pH TK-1",
    "tailings_moisture_pct": "Tailings moisture (%)",
}

target_map = {
    "nacn_consumption_tpd": "NaCN consumption (t/d)",
    "specific_nacn_kgpt": "Specific NaCN (kg/t)",
    "free_cn_ppm_avg": "Free CN average (ppm)",
    "wad_gpl_avg": "WAD CN average (g/L)",
    "complexed_cn_gpl_avg": "Estimated complexed CN (g/L)",
    "recovery_au_pct": "Gold recovery (%)",
}

candidate_drivers = [
    "throughput_tpd",
    "au_feed_gpt",
    "ag_feed_gpt",
    "cu_feed_ppm",
    "cu_solution_ppm_avg",
    "zn_solution_ppm_avg",
    "do_avg",
    "ph_tk_1_s",
    "ph_tk_8_s",
    "tailings_moisture_pct",
    "rt_hours",
]

highlight_targets = [
    "nacn_consumption_tpd",
    "specific_nacn_kgpt",
    "free_cn_ppm_avg",
    "wad_gpl_avg",
    "complexed_cn_gpl_avg",
    "recovery_au_pct",
]


# -----------------------------------------------------------------------------
# 1. HELPER FUNCTIONS
# -----------------------------------------------------------------------------
def safe_corr(df_in: pd.DataFrame, x: str, y: str) -> float:
    sub = df_in[[x, y]].dropna()
    if len(sub) < 3:
        return np.nan
    return sub[x].corr(sub[y])

def classify_strength(val: float) -> str:
    if pd.isna(val):
        return "Insufficient data"
    a = abs(val)
    if a >= 0.80:
        return "Very strong"
    elif a >= 0.60:
        return "Strong"
    elif a >= 0.40:
        return "Moderate"
    elif a >= 0.20:
        return "Weak"
    return "Very weak"

def classify_direction(val: float) -> str:
    if pd.isna(val):
        return "Unknown"
    if val > 0:
        return "Positive"
    elif val < 0:
        return "Negative"
    return "Neutral"

def pct_change(new_val: float, old_val: float) -> float:
    if pd.isna(new_val) or pd.isna(old_val) or old_val == 0:
        return np.nan
    return ((new_val - old_val) / old_val) * 100

def build_finding(metric: str, value: float, direction: str, strength: str, context: str) -> str:
    if pd.isna(value):
        return f"{metric}: not enough data to assess."
    return f"{metric}: {direction.lower()} relationship ({strength.lower()}, r={value:.2f}). {context}"

def fmt_num(x, ndp=3):
    return "–" if pd.isna(x) else f"{x:,.{ndp}f}"

def fmt_pct(x, ndp=1):
    return "–" if pd.isna(x) else f"{x:.{ndp}f}%"

def print_section(title: str):
    print("\n" + "=" * 100)
    print(title.upper())
    print("=" * 100)

def style_table(df: pd.DataFrame, caption: str = ""):
    if df.empty:
        return df

    styler = (
        df.style
        .set_caption(caption)
        .set_table_styles([
            {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "16px"), ("font-weight", "bold"), ("text-align", "left"), ("padding", "6px 0")]},
            {"selector": "th", "props": [("background-color", "#0f172a"), ("color", "white"), ("padding", "8px"), ("border", "1px solid #cbd5e1")]},
            {"selector": "td", "props": [("padding", "8px"), ("border", "1px solid #e2e8f0")]},
            {"selector": "table", "props": [("border-collapse", "collapse"), ("font-size", "13px"), ("width", "100%")]},
        ])
    )
    return styler

def make_excel_friendly(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if out[c].dtype.kind in "fc":
            out[c] = out[c].round(3)
    return out


# -----------------------------------------------------------------------------
# 2. YEAR-ON-YEAR COMPARISON TABLE
# -----------------------------------------------------------------------------
if set(dfo["year"].dropna().unique()) >= {2025, 2026}:
    yoy_metrics = [
        "throughput_tpd",
        "nacn_consumption_tpd",
        "specific_nacn_kgpt",
        "cu_feed_ppm",
        "cu_solution_ppm_avg",
        "free_cn_ppm_avg",
        "wad_gpl_avg",
        "complexed_cn_gpl_avg",
        "recovery_au_pct",
        "do_avg",
        "ph_tk_8_s",
        "rt_hours",
    ]

    yoy = (
        dfo.groupby("year")[yoy_metrics]
        .mean()
        .T
        .rename(columns={2025: "mean_2025", 2026: "mean_2026"})
    )

    yoy["mean_2025"] = pd.to_numeric(yoy["mean_2025"], errors="coerce")
    yoy["mean_2026"] = pd.to_numeric(yoy["mean_2026"], errors="coerce")
    yoy["abs_change"] = yoy["mean_2026"] - yoy["mean_2025"]
    yoy["pct_change"] = [
        pct_change(
            float(yoy.at[idx, "mean_2026"]) if pd.notna(yoy.at[idx, "mean_2026"]) else np.nan,
            float(yoy.at[idx, "mean_2025"]) if pd.notna(yoy.at[idx, "mean_2025"]) else np.nan,
        )
        for idx in yoy.index
    ]

    yoy["metric"] = yoy.index.map(lambda x: metric_labels.get(x, x))
    yoy = yoy[["metric", "mean_2025", "mean_2026", "abs_change", "pct_change"]]
    yoy = yoy.sort_values("metric").reset_index(drop=True)

else:
    yoy = pd.DataFrame()
    print("2025 and 2026 not both available, skipping YoY table.")


# -----------------------------------------------------------------------------
# 3. RANKED DRIVER TABLES
# -----------------------------------------------------------------------------
driver_rows = []

for target_col, target_label in target_map.items():
    for driver in candidate_drivers:
        r = safe_corr(dfo, driver, target_col)
        driver_rows.append({
            "target_col": target_col,
            "target": target_label,
            "driver_col": driver,
            "driver": metric_labels.get(driver, driver),
            "correlation_r": r,
            "abs_r": abs(r) if pd.notna(r) else np.nan,
            "direction": classify_direction(r),
            "strength": classify_strength(r),
        })

driver_table = (
    pd.DataFrame(driver_rows)
    .sort_values(["target", "abs_r"], ascending=[True, False])
    .reset_index(drop=True)
)

top_drivers = (
    driver_table.groupby("target", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)


# -----------------------------------------------------------------------------
# 4. FINDINGS TABLE
# -----------------------------------------------------------------------------
findings_rows = []

key_relationships = [
    ("cu_solution_ppm_avg", "nacn_consumption_tpd", "Higher dissolved Cu tends to coincide with higher cyanide consumption."),
    ("cu_solution_ppm_avg", "specific_nacn_kgpt", "Higher dissolved Cu tends to increase cyanide consumption intensity per tonne treated."),
    ("cu_solution_ppm_avg", "free_cn_ppm_avg", "Higher dissolved Cu appears to reduce the amount of free cyanide maintained in solution."),
    ("cu_solution_ppm_avg", "wad_gpl_avg", "Higher dissolved Cu is associated with higher WAD cyanide."),
    ("cu_solution_ppm_avg", "complexed_cn_gpl_avg", "Higher dissolved Cu is associated with more cyanide reporting to complexed/WAD form."),
    ("cu_solution_ppm_avg", "recovery_au_pct", "If strongly negative, this suggests high dissolved Cu periods may also coincide with weaker recovery."),
    ("cu_feed_ppm", "nacn_consumption_tpd", "This tests whether feed copper alone explains cyanide demand."),
]

for driver, target, context in key_relationships:
    r = safe_corr(dfo, driver, target)
    findings_rows.append({
        "theme": "Copper and cyanide",
        "driver": metric_labels.get(driver, driver),
        "target": metric_labels.get(target, target),
        "r": round(r, 3) if pd.notna(r) else np.nan,
        "direction": classify_direction(r),
        "strength": classify_strength(r),
        "finding": build_finding(
            metric=f"{metric_labels.get(driver, driver)} vs {metric_labels.get(target, target)}",
            value=r,
            direction=classify_direction(r),
            strength=classify_strength(r),
            context=context,
        )
    })

tank_metrics = [
    ("cu_ppm_tk_1_s", "cu_ppm_tk_1_e", "Cu through Tank 1"),
    ("free_cn_ppm_tk_1_s", "free_cn_ppm_tk_1_e", "Free CN through Tank 1"),
    ("wad_gpl_tk_1_s", "wad_gpl_tk_1_e", "WAD through Tank 1"),
    ("au_ppm_tk_1_s", "au_ppm_tk_1_e", "Dissolved Au through Tank 1"),
    ("ag_ppm_tk_1_s", "ag_ppm_tk_1_e", "Dissolved Ag through Tank 1"),
]

tank_progression_rows = []

for inlet_col, outlet_col, label in tank_metrics:
    if inlet_col in dfo.columns and outlet_col in dfo.columns:
        delta = (dfo[outlet_col] - dfo[inlet_col]).mean()
        tank_progression_rows.append({
            "metric": label,
            "inlet_col": inlet_col,
            "outlet_col": outlet_col,
            "mean_inlet": dfo[inlet_col].mean(),
            "mean_outlet": dfo[outlet_col].mean(),
            "mean_delta": delta,
            "direction": "Increase" if delta > 0 else "Decrease" if delta < 0 else "No change",
        })
        findings_rows.append({
            "theme": "Tank 1 progression",
            "driver": inlet_col,
            "target": outlet_col,
            "r": np.nan,
            "direction": "Increase" if delta > 0 else "Decrease" if delta < 0 else "No change",
            "strength": "Mean delta",
            "finding": (
                f"{label}: mean change across Tank 1 = {delta:.3f}. "
                "This supports the interpretation that Tank 1 start/end samples represent progression through the tank."
            )
        })

tank_progression = pd.DataFrame(tank_progression_rows)

if not yoy.empty:
    yoy_lookup = yoy.set_index("metric")
    yoy_raw_lookup = (
        dfo.groupby("year")[list(target_map.keys())]
        .mean()
        .T
        .rename(columns={2025: "mean_2025", 2026: "mean_2026"})
    )
    yoy_raw_lookup["abs_change"] = yoy_raw_lookup["mean_2026"] - yoy_raw_lookup["mean_2025"]
    yoy_raw_lookup["pct_change"] = [
        pct_change(yoy_raw_lookup.at[idx, "mean_2026"], yoy_raw_lookup.at[idx, "mean_2025"])
        for idx in yoy_raw_lookup.index
    ]

    for metric in target_map.keys():
        findings_rows.append({
            "theme": "Year-on-year shift",
            "driver": "2025 vs 2026",
            "target": metric_labels.get(metric, metric),
            "r": np.nan,
            "direction": "Increase" if yoy_raw_lookup.loc[metric, "abs_change"] > 0 else "Decrease",
            "strength": "Mean shift",
            "finding": (
                f"{metric_labels.get(metric, metric)}: "
                f"2025 mean = {yoy_raw_lookup.loc[metric, 'mean_2025']:.3f}, "
                f"2026 mean = {yoy_raw_lookup.loc[metric, 'mean_2026']:.3f}, "
                f"change = {yoy_raw_lookup.loc[metric, 'abs_change']:.3f} "
                f"({yoy_raw_lookup.loc[metric, 'pct_change']:.1f}%)."
            )
        })

mass_balance_findings = {
    "Mean residence time (h)": dfo["rt_hours"].mean() if "rt_hours" in dfo.columns else np.nan,
    "Mean NaCN input (t/d)": dfo["nacn_consumption_tpd"].mean() if "nacn_consumption_tpd" in dfo.columns else np.nan,
    "Mean 30% NaCN solution required (t/d)": dfo["nacn_solution_tpd_30pct"].mean() if "nacn_solution_tpd_30pct" in dfo.columns else np.nan,
    "Mean free CN inventory (t)": dfo["free_cn_inventory_t"].mean() if "free_cn_inventory_t" in dfo.columns else np.nan,
    "Mean WAD inventory (t)": dfo["wad_inventory_t"].mean() if "wad_inventory_t" in dfo.columns else np.nan,
    "Mean complexed CN inventory (t)": dfo["complexed_inventory_t"].mean() if "complexed_inventory_t" in dfo.columns else np.nan,
}

for label, value in mass_balance_findings.items():
    findings_rows.append({
        "theme": "Indicative mass balance",
        "driver": "Circuit inventory / flow assumptions",
        "target": label,
        "r": np.nan,
        "direction": "Magnitude check",
        "strength": "Indicative",
        "finding": f"{label}: {value:.3f}" if pd.notna(value) else f"{label}: not available"
    })

findings_table = pd.DataFrame(findings_rows)


# -----------------------------------------------------------------------------
# 5. EXECUTIVE SUMMARY TABLE
# -----------------------------------------------------------------------------
executive_rows = []

r_sol_cu_nacn = safe_corr(dfo, "cu_solution_ppm_avg", "nacn_consumption_tpd")
r_feed_cu_nacn = safe_corr(dfo, "cu_feed_ppm", "nacn_consumption_tpd")
r_sol_cu_complexed = safe_corr(dfo, "cu_solution_ppm_avg", "complexed_cn_gpl_avg")
r_sol_cu_free = safe_corr(dfo, "cu_solution_ppm_avg", "free_cn_ppm_avg")

executive_rows.append({
    "finding": (
        "Dissolved copper is a stronger indicator of cyanide demand than feed copper."
        if abs(r_sol_cu_nacn) > abs(r_feed_cu_nacn)
        else "Feed copper is at least as strong as dissolved copper in explaining cyanide demand."
    ),
    "evidence": f"r(solution Cu, NaCN t/d) = {r_sol_cu_nacn:.2f}; r(feed Cu, NaCN t/d) = {r_feed_cu_nacn:.2f}",
    "implication": "Circuit chemistry appears more sensitive to dissolved copper than to feed assay alone. Solution chemistry should be monitored directly where possible."
})

executive_rows.append({
    "finding": "Higher dissolved copper is strongly associated with more cyanide tied up in WAD/complexed form.",
    "evidence": f"r(solution Cu, complexed CN) = {r_sol_cu_complexed:.2f}",
    "implication": "A large share of added cyanide may be reporting to copper-related complexes rather than remaining available as free cyanide."
})

executive_rows.append({
    "finding": (
        "Higher dissolved copper tends to coincide with lower free cyanide."
        if r_sol_cu_free < 0
        else "Higher dissolved copper does not appear to reduce free cyanide in this dataset."
    ),
    "evidence": f"r(solution Cu, free CN) = {r_sol_cu_free:.2f}",
    "implication": "High-copper periods are likely to require materially higher NaCN addition to maintain the same free CN operating window."
})

if not yoy.empty:
    yoy_raw = (
        dfo.groupby("year")[["cu_solution_ppm_avg", "complexed_cn_gpl_avg", "free_cn_ppm_avg", "recovery_au_pct"]]
        .mean()
        .T
        .rename(columns={2025: "mean_2025", 2026: "mean_2026"})
    )
    executive_rows.append({
        "finding": (
            "2026 appears materially worse than 2025 on cyanide chemistry."
            if yoy_raw.loc["complexed_cn_gpl_avg", "mean_2026"] > yoy_raw.loc["complexed_cn_gpl_avg", "mean_2025"]
            else "2026 does not appear worse than 2025 on cyanide chemistry."
        ),
        "evidence": (
            f"Solution Cu: {yoy_raw.loc['cu_solution_ppm_avg', 'mean_2025']:.1f} -> {yoy_raw.loc['cu_solution_ppm_avg', 'mean_2026']:.1f}; "
            f"Complexed CN: {yoy_raw.loc['complexed_cn_gpl_avg', 'mean_2025']:.2f} -> {yoy_raw.loc['complexed_cn_gpl_avg', 'mean_2026']:.2f}; "
            f"Free CN: {yoy_raw.loc['free_cn_ppm_avg', 'mean_2025']:.1f} -> {yoy_raw.loc['free_cn_ppm_avg', 'mean_2026']:.1f}; "
            f"Recovery: {yoy_raw.loc['recovery_au_pct', 'mean_2025']:.1f} -> {yoy_raw.loc['recovery_au_pct', 'mean_2026']:.1f}"
        ),
        "implication": "This later period should be investigated for feed change, soluble copper mineralogy, recycle chemistry, residence time, and operating strategy shifts."
    })

if {"free_cn_inventory_t", "complexed_inventory_t"}.issubset(dfo.columns):
    executive_rows.append({
        "finding": "The circuit appears to carry a large WAD/complexed cyanide inventory relative to free cyanide.",
        "evidence": (
            f"Mean free CN inventory = {dfo['free_cn_inventory_t'].mean():.2f} t; "
            f"mean complexed CN inventory = {dfo['complexed_inventory_t'].mean():.2f} t"
        ),
        "implication": "The cyanide problem is unlikely to be explained by free cyanide alone; complexation load appears to be a major component."
    })

executive_summary = pd.DataFrame(executive_rows)


# -----------------------------------------------------------------------------
# 6. KPI SNAPSHOT TABLE
# -----------------------------------------------------------------------------
kpi_rows = []

if not yoy.empty:
    yoy_raw = (
        dfo.groupby("year")[["cu_solution_ppm_avg", "free_cn_ppm_avg", "complexed_cn_gpl_avg", "specific_nacn_kgpt", "recovery_au_pct", "rt_hours"]]
        .mean()
        .T
        .rename(columns={2025: "mean_2025", 2026: "mean_2026"})
    )
    yoy_raw["abs_change"] = yoy_raw["mean_2026"] - yoy_raw["mean_2025"]
    yoy_raw["pct_change"] = [
        pct_change(yoy_raw.at[idx, "mean_2026"], yoy_raw.at[idx, "mean_2025"]) for idx in yoy_raw.index
    ]

    for idx in yoy_raw.index:
        kpi_rows.append({
            "metric": metric_labels.get(idx, idx),
            "2025 mean": yoy_raw.loc[idx, "mean_2025"],
            "2026 mean": yoy_raw.loc[idx, "mean_2026"],
            "change": yoy_raw.loc[idx, "abs_change"],
            "change %": yoy_raw.loc[idx, "pct_change"],
        })

kpi_snapshot = pd.DataFrame(kpi_rows)


# -----------------------------------------------------------------------------
# 7. NOTEBOOK DISPLAY
# -----------------------------------------------------------------------------
print_section("Executive summary")
print(executive_summary.to_string(index=False))

if IN_NOTEBOOK:
    display(style_table(executive_summary, "Executive summary"))

print_section("Year-on-year comparison")
if yoy.empty:
    print("2025 and 2026 data not both available.")
else:
    yoy_display = yoy.copy()
    for c in ["mean_2025", "mean_2026", "abs_change"]:
        yoy_display[c] = yoy_display[c].map(lambda x: fmt_num(x, 3))
    yoy_display["pct_change"] = yoy_display["pct_change"].map(lambda x: fmt_pct(x, 1))
    print(yoy_display.to_string(index=False))
    if IN_NOTEBOOK:
        display(style_table(yoy_display, "Year-on-year comparison"))

print_section("Top drivers per target")
top_drivers_display = top_drivers[["target", "driver", "correlation_r", "direction", "strength"]].copy()
top_drivers_display["correlation_r"] = top_drivers_display["correlation_r"].round(3)
print(top_drivers_display.to_string(index=False))
if IN_NOTEBOOK:
    display(style_table(top_drivers_display, "Top 5 drivers per target"))

print_section("Findings")
print(findings_table[["theme", "finding"]].to_string(index=False, max_colwidth=140))
if IN_NOTEBOOK:
    display(style_table(findings_table[["theme", "finding"]], "Findings table"))

if not tank_progression.empty:
    print_section("Tank 1 progression")
    print(tank_progression.round(3).to_string(index=False))
    if IN_NOTEBOOK:
        display(style_table(tank_progression.round(3), "Tank 1 progression"))


# -----------------------------------------------------------------------------
# 8. POLISHED CHARTS
# -----------------------------------------------------------------------------
# 8A. Year-on-year comparison chart
if not yoy.empty:
    yoy_plot = yoy[yoy["metric"].isin([
        "Cu in solution (ppm)",
        "Free CN average (ppm)",
        "Estimated complexed CN (g/L)",
        "Specific NaCN (kg/t)",
        "Gold recovery (%)",
        "Residence time (h)",
    ])].copy()

    yoy_long = yoy_plot.melt(
        id_vars="metric",
        value_vars=["mean_2025", "mean_2026"],
        var_name="year",
        value_name="value"
    )
    yoy_long["year"] = yoy_long["year"].str.replace("mean_", "", regex=False)

    fig_yoy = px.bar(
        yoy_long,
        x="metric",
        y="value",
        color="year",
        barmode="group",
        text="value",
        title="Year-on-year comparison: key operating and chemistry indicators",
        labels={"metric": "", "value": "Mean value", "year": "Year"},
    )
    fig_yoy.update_traces(texttemplate="%{text:.2f}", textposition="outside")
    fig_yoy.update_layout(
        template="plotly_white",
        height=520,
        legend_title_text="",
        title_x=0.02,
        xaxis_tickangle=-25,
        margin=dict(l=40, r=30, t=70, b=120),
    )
    fig_yoy.show()

# 8B. Driver heatmap
heatmap_df = driver_table.copy()
heatmap_pivot = heatmap_df.pivot(index="driver", columns="target", values="correlation_r")
heatmap_pivot = heatmap_pivot.reindex(
    index=[metric_labels.get(x, x) for x in candidate_drivers]
)

fig_heat = px.imshow(
    heatmap_pivot,
    text_auto=".2f",
    aspect="auto",
    title="Driver correlation heatmap",
    labels=dict(x="", y="", color="r"),
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
)
fig_heat.update_layout(
    template="plotly_white",
    height=520,
    title_x=0.02,
    margin=dict(l=40, r=30, t=70, b=40),
)
fig_heat.show()

# 8C. Copper relationship dashboard
copper_targets = [
    ("nacn_consumption_tpd", "NaCN consumption (t/d)"),
    ("specific_nacn_kgpt", "Specific NaCN (kg/t)"),
    ("free_cn_ppm_avg", "Free CN average (ppm)"),
    ("complexed_cn_gpl_avg", "Estimated complexed CN (g/L)"),
]

fig_scatter = make_subplots(
    rows=2, cols=2,
    subplot_titles=[label for _, label in copper_targets],
    horizontal_spacing=0.10,
    vertical_spacing=0.16
)

positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

for (target_col, target_label), (r, c) in zip(copper_targets, positions):
    sub = dfo[["cu_solution_ppm_avg", target_col]].dropna().copy()
    corr_val = safe_corr(dfo, "cu_solution_ppm_avg", target_col)

    fig_scatter.add_trace(
        go.Scatter(
            x=sub["cu_solution_ppm_avg"],
            y=sub[target_col],
            mode="markers",
            name=target_label,
            showlegend=False,
            marker=dict(size=7, opacity=0.70),
            hovertemplate=(
                "Cu in solution: %{x:,.1f}<br>"
                f"{target_label}: "+"%{y:,.3f}<extra></extra>"
            ),
        ),
        row=r, col=c
    )

    if len(sub) >= 2:
        z = np.polyfit(sub["cu_solution_ppm_avg"], sub[target_col], 1)
        xline = np.linspace(sub["cu_solution_ppm_avg"].min(), sub["cu_solution_ppm_avg"].max(), 100)
        yline = z[0] * xline + z[1]

        fig_scatter.add_trace(
            go.Scatter(
                x=xline,
                y=yline,
                mode="lines",
                name=f"Trend ({target_label})",
                showlegend=False,
                hoverinfo="skip",
                line=dict(width=2),
            ),
            row=r, col=c
        )

    fig_scatter.update_xaxes(title_text="Cu in solution (ppm)", row=r, col=c)
    fig_scatter.update_yaxes(title_text=target_label, row=r, col=c)

    fig_scatter.add_annotation(
        x=0.98, y=0.95,
        xref=f"x{'' if (r, c) == (1, 1) else (2 if (r, c) == (1, 2) else 3 if (r, c) == (2, 1) else 4)} domain",
        yref=f"y{'' if (r, c) == (1, 1) else (2 if (r, c) == (1, 2) else 3 if (r, c) == (2, 1) else 4)} domain",
        text=f"r = {corr_val:.2f}" if pd.notna(corr_val) else "r = n/a",
        showarrow=False,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="rgba(100,100,100,0.35)",
        font=dict(size=11),
        row=r, col=c
    )

fig_scatter.update_layout(
    template="plotly_white",
    height=760,
    title="Copper relationship dashboard",
    title_x=0.02,
    margin=dict(l=50, r=30, t=80, b=50),
)
fig_scatter.show()

# 8D. Cyanide inventory comparison
inventory_cols = {
    "Free CN inventory (t)": "free_cn_inventory_t",
    "WAD inventory (t)": "wad_inventory_t",
    "Complexed CN inventory (t)": "complexed_inventory_t",
}
inventory_rows = []
for label, col in inventory_cols.items():
    if col in dfo.columns:
        inventory_rows.append({"inventory_type": label, "mean_tonnes": dfo[col].mean()})

inventory_df = pd.DataFrame(inventory_rows)

if not inventory_df.empty:
    fig_inventory = px.bar(
        inventory_df,
        x="inventory_type",
        y="mean_tonnes",
        text="mean_tonnes",
        title="Indicative cyanide inventory comparison",
        labels={"inventory_type": "", "mean_tonnes": "Mean inventory (t)"},
    )
    fig_inventory.update_traces(texttemplate="%{text:.2f}", textposition="outside")
    fig_inventory.update_layout(
        template="plotly_white",
        height=460,
        title_x=0.02,
        margin=dict(l=40, r=30, t=70, b=60),
    )
    fig_inventory.show()


# -----------------------------------------------------------------------------
# 9. EXPORT TABLES
# -----------------------------------------------------------------------------
driver_table_export = make_excel_friendly(driver_table)
top_drivers_export = make_excel_friendly(top_drivers)
findings_table_export = make_excel_friendly(findings_table)
executive_summary_export = executive_summary.copy()
tank_progression_export = make_excel_friendly(tank_progression)
yoy_export = make_excel_friendly(yoy)
kpi_snapshot_export = make_excel_friendly(kpi_snapshot)

driver_table_export.to_csv(output_dir / "ranked_driver_table.csv", index=False)
top_drivers_export.to_csv(output_dir / "top_5_drivers_per_target.csv", index=False)
findings_table_export.to_csv(output_dir / "findings_table.csv", index=False)
executive_summary_export.to_csv(output_dir / "executive_summary.csv", index=False)
kpi_snapshot_export.to_csv(output_dir / "kpi_snapshot.csv", index=False)

if not yoy_export.empty:
    yoy_export.to_csv(output_dir / "year_on_year_summary.csv", index=False)

if not tank_progression_export.empty:
    tank_progression_export.to_csv(output_dir / "tank_progression_summary.csv", index=False)

# formatted Excel pack
with pd.ExcelWriter(output_dir / "diagnostic_summary_pack.xlsx", engine="openpyxl") as writer:
    executive_summary_export.to_excel(writer, sheet_name="Executive Summary", index=False)
    if not yoy_export.empty:
        yoy_export.to_excel(writer, sheet_name="YoY Summary", index=False)
    kpi_snapshot_export.to_excel(writer, sheet_name="KPI Snapshot", index=False)
    driver_table_export.to_excel(writer, sheet_name="Driver Ranking", index=False)
    top_drivers_export.to_excel(writer, sheet_name="Top Drivers", index=False)
    findings_table_export.to_excel(writer, sheet_name="Findings", index=False)
    if not tank_progression_export.empty:
        tank_progression_export.to_excel(writer, sheet_name="Tank 1 Progression", index=False)

print(f"\nSaved outputs to: {output_dir.resolve()}")


# -----------------------------------------------------------------------------
# 10. CLEAN NARRATIVE SUMMARY
# -----------------------------------------------------------------------------

if IN_NOTEBOOK:
    display(Markdown("### Narrative summary"))
    narrative_md = []
    for i, row in executive_summary.iterrows():
        narrative_md.append(
            f"**{i+1}. {row['finding']}**  \n"
            f"Evidence: {row['evidence']}  \n"
            f"Implication: {row['implication']}"
        )
    display(Markdown("\n\n".join(narrative_md)))


# =============================================================================
# MASS-BALANCE RECONCILIATION — POLISHED DIAGNOSTIC PACK
# =============================================================================

try:
    from IPython.display import display, Markdown
    IN_NOTEBOOK = True
except ImportError:
    IN_NOTEBOOK = False


# -----------------------------------------------------------------------------
# 0. CONFIG / HELPERS
# -----------------------------------------------------------------------------
output_dir = Path("la_coipa_diagnostics_outputs")
output_dir.mkdir(exist_ok=True)

def to_excel_ready(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if out[c].dtype.kind in "fc":
            out[c] = out[c].round(3)
    return out


# -----------------------------------------------------------------------------
# 1. MASS BALANCE BASIS
# -----------------------------------------------------------------------------
# Assumptions expected to already exist:
# - SOLIDS_MASS_FRACTION
# - PROCESS_LIQUID_DENSITY
# - SOLIDS_DENSITY
# - TOTAL_CIRCUIT_VOLUME_M3
# - NACN_SOLUTION_STRENGTH

dfo = dfo.copy()
dfo["date"] = pd.to_datetime(dfo["date"], errors="coerce")

# Dry solids throughput plus density-based slurry volume estimate
dfo["solids_tpd_est"] = dfo["throughput_tpd"]
dfo["water_tpd_est"] = dfo["throughput_tpd"] * (1 - SOLIDS_MASS_FRACTION) / SOLIDS_MASS_FRACTION
dfo["water_m3d_est"] = dfo["water_tpd_est"] / PROCESS_LIQUID_DENSITY
dfo["solids_m3d_est"] = dfo["solids_tpd_est"] / SOLIDS_DENSITY
dfo["slurry_tpd_est"] = dfo["solids_tpd_est"] + dfo["water_tpd_est"]
dfo["slurry_m3d_est"] = dfo["solids_m3d_est"] + dfo["water_m3d_est"]

dfo["rt_days_est"] = safe_divide(TOTAL_CIRCUIT_VOLUME_M3, dfo["slurry_m3d_est"])
dfo["rt_hours_est"] = dfo["rt_days_est"] * 24
dfo["rt_hours_est_plot"] = dfo["rt_hours_est"].where(dfo["throughput_tpd"] >= 500, np.nan)


# -----------------------------------------------------------------------------
# 2. CYANIDE MASS BALANCE
# -----------------------------------------------------------------------------
# Pure NaCN input from operational data
dfo["nacn_input_tpd"] = dfo["nacn_consumption_tpd"]
dfo["nacn_input_kgd"] = dfo["nacn_input_tpd"] * 1000

# Equivalent 30% solution dosing rate
dfo["nacn_solution_tpd_30pct"] = safe_divide(dfo["nacn_input_tpd"], NACN_SOLUTION_STRENGTH)
dfo["nacn_solution_kgd_30pct"] = dfo["nacn_solution_tpd_30pct"] * 1000

# Outlet cyanide estimates at Tank 8
# free_cn_ppm_tk_8_s : mg/L == g/m3
# wad_gpl_tk_8_s     : g/L  == kg/m3
dfo["free_cn_out_kgd_est"] = dfo["free_cn_ppm_tk_8_s"] * dfo["water_m3d_est"] / 1000.0
dfo["wad_cn_out_kgd_est"] = dfo["wad_gpl_tk_8_s"] * dfo["water_m3d_est"]
dfo["complexed_cn_out_kgd_est"] = dfo["wad_cn_out_kgd_est"] - dfo["free_cn_out_kgd_est"]

# Accountability ratios
dfo["free_cn_accountability_pct"] = np.where(
    dfo["nacn_input_kgd"] > 0,
    100 * dfo["free_cn_out_kgd_est"] / dfo["nacn_input_kgd"],
    np.nan,
)
dfo["wad_cn_accountability_pct"] = np.where(
    dfo["nacn_input_kgd"] > 0,
    100 * dfo["wad_cn_out_kgd_est"] / dfo["nacn_input_kgd"],
    np.nan,
)
dfo["complexed_cn_accountability_pct"] = np.where(
    dfo["nacn_input_kgd"] > 0,
    100 * dfo["complexed_cn_out_kgd_est"] / dfo["nacn_input_kgd"],
    np.nan,
)

# Indicative liquid-phase inventory estimate
LIQUID_INVENTORY_M3 = TOTAL_CIRCUIT_VOLUME_M3 * (1 - SOLIDS_MASS_FRACTION)

dfo["free_cn_inventory_t_est"] = dfo["free_cn_ppm_avg"] * LIQUID_INVENTORY_M3 / 1e6
dfo["wad_cn_inventory_t_est"] = dfo["wad_gpl_avg"] * LIQUID_INVENTORY_M3 / 1000.0
dfo["complexed_cn_inventory_t_est"] = dfo["wad_cn_inventory_t_est"] - dfo["free_cn_inventory_t_est"]


# -----------------------------------------------------------------------------
# 3. GOLD / SILVER / COPPER MASS FLOWS
# -----------------------------------------------------------------------------
# Gold
dfo["au_feed_gpd"] = dfo["au_feed_gpt"] * dfo["throughput_tpd"]
dfo["au_tail_gpd"] = dfo["au_tail_gpt"] * dfo["throughput_tpd"]
dfo["au_extracted_gpd"] = dfo["au_feed_gpd"] - dfo["au_tail_gpd"]
dfo["au_recovery_calc_pct"] = np.where(
    dfo["au_feed_gpd"] > 0,
    100 * dfo["au_extracted_gpd"] / dfo["au_feed_gpd"],
    np.nan,
)

# Silver
dfo["ag_feed_gpd"] = dfo["ag_feed_gpt"] * dfo["throughput_tpd"]
dfo["ag_tail_gpd"] = dfo["ag_tail_gpt"] * dfo["throughput_tpd"]
dfo["ag_extracted_gpd"] = dfo["ag_feed_gpd"] - dfo["ag_tail_gpd"]
dfo["ag_recovery_calc_pct"] = np.where(
    dfo["ag_feed_gpd"] > 0,
    100 * dfo["ag_extracted_gpd"] / dfo["ag_feed_gpd"],
    np.nan,
)

# Copper
dfo["cu_feed_gpd"] = dfo["cu_feed_ppm"] * dfo["throughput_tpd"]
dfo["cu_sol_tk8_kgd_est"] = dfo["cu_ppm_tk_8_s"] * dfo["water_m3d_est"] / 1000.0
dfo["cu_sol_tk8_gpd_est"] = dfo["cu_sol_tk8_kgd_est"] * 1000
dfo["cu_solution_fraction_pct_est"] = np.where(
    dfo["cu_feed_gpd"] > 0,
    100 * dfo["cu_sol_tk8_gpd_est"] / dfo["cu_feed_gpd"],
    np.nan,
)


# -----------------------------------------------------------------------------
# 4. DAILY RECONCILIATION TABLE
# -----------------------------------------------------------------------------
reconciliation_daily = dfo[[
    "date",
    "throughput_tpd",
    "solids_tpd_est",
    "water_m3d_est",
    "slurry_m3d_est",
    "rt_hours_est",
    "nacn_input_tpd",
    "nacn_solution_tpd_30pct",

    # Cu concentration / mass-flow views
    "cu_ppm_tk_8_s",
    "cu_sol_tk8_gpd_est",
    "cu_solution_fraction_pct_est",

    # CN outflow / accountability
    "free_cn_out_kgd_est",
    "wad_cn_out_kgd_est",
    "complexed_cn_out_kgd_est",
    "free_cn_accountability_pct",
    "wad_cn_accountability_pct",
    "complexed_cn_accountability_pct",

    # CN inventory
    "free_cn_inventory_t_est",
    "wad_cn_inventory_t_est",
    "complexed_cn_inventory_t_est",

    # Au / Ag
    "au_feed_gpd",
    "au_tail_gpd",
    "au_extracted_gpd",
    "au_recovery_calc_pct",
    "ag_feed_gpd",
    "ag_tail_gpd",
    "ag_extracted_gpd",
    "ag_recovery_calc_pct",

    # Cu feed
    "cu_feed_gpd",
]].copy()


# -----------------------------------------------------------------------------
# 5. SUMMARY TABLE
# -----------------------------------------------------------------------------
reconciliation_summary = pd.DataFrame([
    {"metric": "Mean throughput", "value": dfo["throughput_tpd"].mean(), "unit": "t/d"},
    {"metric": "Mean water flow", "value": dfo["water_m3d_est"].mean(), "unit": "m3/d"},
    {"metric": "Mean slurry flow", "value": dfo["slurry_m3d_est"].mean(), "unit": "m3/d"},
    {"metric": "Mean estimated residence time", "value": dfo["rt_hours_est"].mean(), "unit": "h"},
    {"metric": "Mean NaCN input", "value": dfo["nacn_input_tpd"].mean(), "unit": "t/d"},
    {"metric": "Mean 30% NaCN solution rate", "value": dfo["nacn_solution_tpd_30pct"].mean(), "unit": "t/d"},
    {"metric": "Mean free CN outflow", "value": dfo["free_cn_out_kgd_est"].mean(), "unit": "kg/d"},
    {"metric": "Mean WAD CN outflow", "value": dfo["wad_cn_out_kgd_est"].mean(), "unit": "kg/d"},
    {"metric": "Mean complexed CN outflow", "value": dfo["complexed_cn_out_kgd_est"].mean(), "unit": "kg/d"},
    {"metric": "Mean free CN inventory", "value": dfo["free_cn_inventory_t_est"].mean(), "unit": "t"},
    {"metric": "Mean WAD CN inventory", "value": dfo["wad_cn_inventory_t_est"].mean(), "unit": "t"},
    {"metric": "Mean complexed CN inventory", "value": dfo["complexed_cn_inventory_t_est"].mean(), "unit": "t"},
    {"metric": "Mean Au feed", "value": dfo["au_feed_gpd"].mean(), "unit": "g/d"},
    {"metric": "Mean Au extracted", "value": dfo["au_extracted_gpd"].mean(), "unit": "g/d"},
    {"metric": "Mean Au recovery (calculated)", "value": dfo["au_recovery_calc_pct"].mean(), "unit": "%"},
    {"metric": "Mean Ag feed", "value": dfo["ag_feed_gpd"].mean(), "unit": "g/d"},
    {"metric": "Mean Ag extracted", "value": dfo["ag_extracted_gpd"].mean(), "unit": "g/d"},
    {"metric": "Mean Ag recovery (calculated)", "value": dfo["ag_recovery_calc_pct"].mean(), "unit": "%"},
    {"metric": "Mean Cu in feed", "value": dfo["cu_feed_gpd"].mean(), "unit": "g/d"},
    {"metric": "Mean Cu in TK-8 solution", "value": dfo["cu_sol_tk8_gpd_est"].mean(), "unit": "g/d"},
    {"metric": "Mean Cu solution fraction", "value": dfo["cu_solution_fraction_pct_est"].mean(), "unit": "%"},
]).round(3)


# -----------------------------------------------------------------------------
# 6. MONTHLY RECONCILIATION
# -----------------------------------------------------------------------------
reconciliation_monthly = (
    reconciliation_daily
    .set_index("date")
    .resample("ME")
    .mean(numeric_only=True)
    .round(3)
    .reset_index()
)


# -----------------------------------------------------------------------------
# 7. FLAGS / UNUSUAL DAYS
# -----------------------------------------------------------------------------
high_specific_thresh = dfo["specific_nacn_kgpt"].quantile(0.90)
high_rt_thresh = reconciliation_daily["rt_hours_est"].quantile(0.90)
low_rt_thresh = reconciliation_daily["rt_hours_est"].quantile(0.10)

reconciliation_daily["flag_high_cu_solution_fraction"] = reconciliation_daily["cu_solution_fraction_pct_est"] > 50
reconciliation_daily["flag_high_wad_accountability"] = reconciliation_daily["wad_cn_accountability_pct"] > 100
reconciliation_daily["flag_high_specific_nacn"] = dfo["specific_nacn_kgpt"] > high_specific_thresh
reconciliation_daily["flag_high_rt"] = reconciliation_daily["rt_hours_est"] > high_rt_thresh
reconciliation_daily["flag_low_rt"] = reconciliation_daily["rt_hours_est"] < low_rt_thresh

flag_cols = [
    "flag_high_cu_solution_fraction",
    "flag_high_wad_accountability",
    "flag_high_specific_nacn",
    "flag_high_rt",
    "flag_low_rt",
]

reconciliation_daily["flag_count"] = reconciliation_daily[flag_cols].sum(axis=1)

flag_days = reconciliation_daily.loc[
    reconciliation_daily["flag_count"] > 0,
    ["date", "throughput_tpd", "rt_hours_est", "nacn_input_tpd",
     "wad_cn_accountability_pct", "cu_solution_fraction_pct_est", "flag_count"] + flag_cols
].copy()

flag_summary = pd.DataFrame({
    "flag": [
        "High Cu solution fraction",
        "High WAD accountability",
        "High specific NaCN",
        "High residence time",
        "Low residence time",
    ],
    "days_flagged": [
        reconciliation_daily["flag_high_cu_solution_fraction"].sum(),
        reconciliation_daily["flag_high_wad_accountability"].sum(),
        reconciliation_daily["flag_high_specific_nacn"].sum(),
        reconciliation_daily["flag_high_rt"].sum(),
        reconciliation_daily["flag_low_rt"].sum(),
    ]
})


# -----------------------------------------------------------------------------
# 8. NOTEBOOK DISPLAY
# -----------------------------------------------------------------------------
print_section("Daily reconciliation preview")
print(reconciliation_daily.head().round(3).to_string(index=False))
if IN_NOTEBOOK:
    display(style_table(reconciliation_daily.head().round(3), "Daily reconciliation preview"))

print_section("Reconciliation summary")
print(reconciliation_summary.to_string(index=False))
if IN_NOTEBOOK:
    display(style_table(reconciliation_summary, "Reconciliation summary"))

print_section("Monthly reconciliation summary")
print(reconciliation_monthly.tail(12).round(3).to_string(index=False))
if IN_NOTEBOOK:
    display(style_table(reconciliation_monthly.tail(12).round(3), "Monthly reconciliation summary"))

print_section("Flagged days preview")
print(flag_days.head(20).round(3).to_string(index=False))
if IN_NOTEBOOK:
    display(style_table(flag_days.head(20).round(3), "Flagged days preview"))

print_section("Flag summary")
print(flag_summary.to_string(index=False))
if IN_NOTEBOOK:
    display(style_table(flag_summary, "Flag summary"))


# -----------------------------------------------------------------------------
# 9. POLISHED CHARTS (WITH IQR FILTERING FOR VISUALISATION ONLY)
# -----------------------------------------------------------------------------
# IMPORTANT:
# - Raw calculations, summaries, and exports remain unchanged.
# - IQR filtering is applied to plotting datasets only, so visuals are clearer
#   without altering the underlying reconciliation logic.

def get_iqr_mask(series: pd.Series, multiplier: float = 1.5) -> pd.Series:
    """
    Returns a boolean mask where True means the value is within IQR bounds.
    Uses an existing IQR helper if one was defined earlier; otherwise falls
    back to a local implementation.
    """
    s = pd.to_numeric(series, errors="coerce")

    # Try to use an earlier user-defined helper if present
    # Supported patterns:
    #   1) iqr_mask(series, multiplier=1.5)
    #   2) iqr_filter(series, multiplier=1.5)
    #   3) iqr_bounds(series, multiplier=1.5) -> (lower, upper)
    if "iqr_mask" in globals():
        try:
            mask = iqr_mask(s, multiplier=multiplier)
            return pd.Series(mask, index=series.index).fillna(False)
        except Exception:
            pass

    if "iqr_filter" in globals():
        try:
            mask = iqr_filter(s, multiplier=multiplier)
            return pd.Series(mask, index=series.index).fillna(False)
        except Exception:
            pass

    if "iqr_bounds" in globals():
        try:
            lower, upper = iqr_bounds(s, multiplier=multiplier)
            return ((s >= lower) & (s <= upper)).fillna(False)
        except Exception:
            pass

    # Local fallback
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1

    if pd.isna(iqr) or iqr == 0:
        return s.notna()

    lower = q1 - multiplier * iqr
    upper = q3 + multiplier * iqr
    return ((s >= lower) & (s <= upper)).fillna(False)


def iqr_filter_df(df: pd.DataFrame, cols: list[str], multiplier: float = 1.5) -> pd.DataFrame:
    """
    Row-wise IQR filter across one or more numeric columns.
    Keeps rows that are within IQR bounds for all listed columns.
    """
    if df.empty:
        return df.copy()

    mask = pd.Series(True, index=df.index)
    for col in cols:
        if col in df.columns:
            mask &= get_iqr_mask(df[col], multiplier=multiplier)
    return df.loc[mask].copy()


def summarise_plot_filtering(raw_df: pd.DataFrame, filtered_df: pd.DataFrame, label: str):
    removed = len(raw_df) - len(filtered_df)
    pct_removed = (removed / len(raw_df) * 100) if len(raw_df) > 0 else 0
    print(
        f"{label}: plotting filter kept {len(filtered_df):,} / {len(raw_df):,} rows "
        f"(removed {removed:,}, {pct_removed:.1f}%)."
    )


# Optional tuning:
# Use slightly looser filtering for monthly data and standard filtering for daily scatter.
MONTHLY_IQR_MULTIPLIER = 1.5
SCATTER_IQR_MULTIPLIER = 1.5


# 9A. Monthly operations + reconciliation trend panel
monthly_plot_raw = reconciliation_monthly.copy()

monthly_plot = iqr_filter_df(
    monthly_plot_raw,
    cols=[
        "throughput_tpd",
        "rt_hours_est",
        "nacn_input_tpd",
        "nacn_solution_tpd_30pct",
        "wad_cn_accountability_pct",
    ],
    multiplier=MONTHLY_IQR_MULTIPLIER,
)

summarise_plot_filtering(monthly_plot_raw, monthly_plot, "Monthly trend panel")

fig_monthly = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Throughput",
        "Estimated residence time",
        "NaCN input vs 30% solution rate",
        "WAD accountability"
    ),
    horizontal_spacing=0.10,
    vertical_spacing=0.16
)

fig_monthly.add_trace(
    go.Scatter(
        x=monthly_plot["date"],
        y=monthly_plot["throughput_tpd"],
        mode="lines+markers",
        name="Throughput (t/d)",
        hovertemplate="%{x|%b %Y}<br>Throughput: %{y:,.0f} t/d<extra></extra>",
    ),
    row=1, col=1
)

fig_monthly.add_trace(
    go.Scatter(
        x=monthly_plot["date"],
        y=monthly_plot["rt_hours_est"],
        mode="lines+markers",
        name="RT (h)",
        hovertemplate="%{x|%b %Y}<br>RT: %{y:,.1f} h<extra></extra>",
    ),
    row=1, col=2
)

fig_monthly.add_trace(
    go.Scatter(
        x=monthly_plot["date"],
        y=monthly_plot["nacn_input_tpd"],
        mode="lines+markers",
        name="NaCN input (t/d)",
        hovertemplate="%{x|%b %Y}<br>NaCN input: %{y:,.2f} t/d<extra></extra>",
    ),
    row=2, col=1
)

fig_monthly.add_trace(
    go.Scatter(
        x=monthly_plot["date"],
        y=monthly_plot["nacn_solution_tpd_30pct"],
        mode="lines+markers",
        name="30% NaCN solution (t/d)",
        hovertemplate="%{x|%b %Y}<br>30% solution: %{y:,.2f} t/d<extra></extra>",
    ),
    row=2, col=1
)

fig_monthly.add_trace(
    go.Scatter(
        x=monthly_plot["date"],
        y=monthly_plot["wad_cn_accountability_pct"],
        mode="lines+markers",
        name="WAD accountability (%)",
        hovertemplate="%{x|%b %Y}<br>WAD accountability: %{y:,.1f}%<extra></extra>",
    ),
    row=2, col=2
)

fig_monthly.update_yaxes(title_text="t/d", row=1, col=1)
fig_monthly.update_yaxes(title_text="h", row=1, col=2)
fig_monthly.update_yaxes(title_text="t/d", row=2, col=1)
fig_monthly.update_yaxes(title_text="%", row=2, col=2)

fig_monthly.update_xaxes(showgrid=True, row=1, col=1)
fig_monthly.update_xaxes(showgrid=True, row=1, col=2)
fig_monthly.update_xaxes(showgrid=True, row=2, col=1)
fig_monthly.update_xaxes(showgrid=True, row=2, col=2)

fig_monthly.update_layout(
    template="plotly_white",
    height=760,
    title="Monthly mass-balance reconciliation trends",
    title_x=0.02,
    legend_title_text="",
    margin=dict(l=50, r=30, t=80, b=40),
)

fig_monthly.show()


# 9B. Mean inventory comparison
inventory_df = pd.DataFrame({
    "inventory_type": [
        "Free CN inventory",
        "WAD CN inventory",
        "Complexed CN inventory",
    ],
    "mean_tonnes": [
        dfo["free_cn_inventory_t_est"].mean(),
        dfo["wad_cn_inventory_t_est"].mean(),
        dfo["complexed_cn_inventory_t_est"].mean(),
    ]
})

fig_inventory = px.bar(
    inventory_df,
    x="inventory_type",
    y="mean_tonnes",
    text="mean_tonnes",
    title="Indicative cyanide inventory comparison",
    labels={"inventory_type": "", "mean_tonnes": "Mean inventory (t)"},
)

fig_inventory.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside",
    hovertemplate="%{x}<br>Mean inventory: %{y:,.2f} t<extra></extra>",
)

fig_inventory.update_layout(
    template="plotly_white",
    height=460,
    title_x=0.02,
    margin=dict(l=40, r=30, t=70, b=60),
)

fig_inventory.show()


# 9C. Flag counts chart
fig_flags = px.bar(
    flag_summary,
    x="flag",
    y="days_flagged",
    text="days_flagged",
    title="Flagged-day summary",
    labels={"flag": "", "days_flagged": "Days flagged"},
)

fig_flags.update_traces(
    textposition="outside",
    hovertemplate="%{x}<br>Days flagged: %{y}<extra></extra>",
)

fig_flags.update_layout(
    template="plotly_white",
    height=460,
    title_x=0.02,
    xaxis_tickangle=-20,
    margin=dict(l=40, r=30, t=70, b=100),
)

fig_flags.show()


# 9D. Copper accountability scatter
# Apply IQR filtering to BOTH axes used in the scatter.
scatter_df_raw = reconciliation_daily[[
    "cu_solution_fraction_pct_est",
    "wad_cn_accountability_pct",
    "nacn_input_tpd",
    "throughput_tpd",
    "date"
]].dropna().copy()

scatter_df = iqr_filter_df(
    scatter_df_raw,
    cols=[
        "cu_solution_fraction_pct_est",
        "wad_cn_accountability_pct",
    ],
    multiplier=SCATTER_IQR_MULTIPLIER,
)

summarise_plot_filtering(scatter_df_raw, scatter_df, "Copper vs WAD scatter")

if not scatter_df.empty:
    fig_cu = px.scatter(
        scatter_df,
        x="cu_solution_fraction_pct_est",
        y="wad_cn_accountability_pct",
        size="nacn_input_tpd",
        hover_data=["date", "throughput_tpd"],
        title="Copper solution fraction vs WAD accountability",
        labels={
            "cu_solution_fraction_pct_est": "Cu solution fraction (%)",
            "wad_cn_accountability_pct": "WAD accountability (%)",
            "nacn_input_tpd": "NaCN input (t/d)",
        },
    )

    fig_cu.update_traces(
        marker=dict(opacity=0.70, line=dict(width=0.5, color="white")),
        hovertemplate=(
            "Date: %{customdata[0]|%Y-%m-%d}<br>"
            "Cu solution fraction: %{x:,.1f}%<br>"
            "WAD accountability: %{y:,.1f}%<br>"
            "Throughput: %{customdata[1]:,.0f} t/d<br>"
            "NaCN input: %{marker.size:,.2f} t/d<extra></extra>"
        ),
    )

    fig_cu.update_layout(
        template="plotly_white",
        height=500,
        title_x=0.02,
        margin=dict(l=50, r=30, t=70, b=50),
    )

    fig_cu.show()


# 9E. Optional diagnostic companion: show excluded scatter outliers separately
excluded_scatter = scatter_df_raw.loc[~scatter_df_raw.index.isin(scatter_df.index)].copy()

if not excluded_scatter.empty:
    print("\nExcluded scatter outliers (for review only):")
    print(
        excluded_scatter[
            ["date", "cu_solution_fraction_pct_est", "wad_cn_accountability_pct", "nacn_input_tpd", "throughput_tpd"]
        ]
        .sort_values(["cu_solution_fraction_pct_est", "wad_cn_accountability_pct"], ascending=False)
        .head(20)
        .round(3)
        .to_string(index=False)
    )


# -----------------------------------------------------------------------------
# 10. NARRATIVE SUMMARY
# -----------------------------------------------------------------------------
narrative_rows = [
    {
        "finding": "The reconciliation suggests a large cyanide inventory is carried in WAD and complexed form relative to free cyanide.",
        "evidence": (
            f"Mean free CN inventory = {dfo['free_cn_inventory_t_est'].mean():.2f} t; "
            f"mean WAD inventory = {dfo['wad_cn_inventory_t_est'].mean():.2f} t; "
            f"mean complexed inventory = {dfo['complexed_cn_inventory_t_est'].mean():.2f} t."
        ),
        "implication": "A high share of cyanide appears tied up in non-free forms, so reagent demand cannot be interpreted from free CN alone."
    },
    {
        "finding": "Estimated WAD accountability should be treated as an indicative diagnostic, not a closed mass balance.",
        "evidence": (
            f"Mean WAD accountability = {dfo['wad_cn_accountability_pct'].mean():.1f}%; "
            f"days above 100% = {int(reconciliation_daily['flag_high_wad_accountability'].sum())}."
        ),
        "implication": "Values above 100% indicate that simplifying assumptions, sample representativeness, recycle effects, or timing mismatches are material."
    },
    {
        "finding": "The circuit shows meaningful day-to-day variability in estimated residence time.",
        "evidence": (
            f"Estimated RT mean = {dfo['rt_hours_est'].mean():.1f} h; "
            f"P10 = {reconciliation_daily['rt_hours_est'].quantile(0.10):.1f} h; "
            f"P90 = {reconciliation_daily['rt_hours_est'].quantile(0.90):.1f} h."
        ),
        "implication": "Residence time variability should be considered when interpreting same-day chemistry and recovery relationships."
    },
    {
        "finding": "Copper appearing in solution at Tank 8 is non-trivial relative to feed copper on some days.",
        "evidence": (
            f"Mean estimated Cu solution fraction = {dfo['cu_solution_fraction_pct_est'].mean():.1f}%; "
            f"days above 50% = {int(reconciliation_daily['flag_high_cu_solution_fraction'].sum())}."
        ),
        "implication": "This supports the idea that soluble copper load is an important consumer of cyanide and should be tracked alongside feed metrics."
    },
]

reconciliation_narrative = pd.DataFrame(narrative_rows)

print_section("Narrative summary")
for i, row in reconciliation_narrative.iterrows():
    print(f"\n{i+1}. {row['finding']}")
    print(f"   Evidence: {row['evidence']}")
    print(f"   Implication: {row['implication']}")

if IN_NOTEBOOK:
    display(style_table(reconciliation_narrative, "Narrative summary"))


# -----------------------------------------------------------------------------
# 11. EXPORT
# -----------------------------------------------------------------------------
reconciliation_daily_export = to_excel_ready(reconciliation_daily)
reconciliation_summary_export = to_excel_ready(reconciliation_summary)
reconciliation_monthly_export = to_excel_ready(reconciliation_monthly)
flag_days_export = to_excel_ready(flag_days)
flag_summary_export = to_excel_ready(flag_summary)
reconciliation_narrative_export = reconciliation_narrative.copy()

reconciliation_daily_export.to_csv(output_dir / "mass_balance_reconciliation_daily.csv", index=False)
reconciliation_monthly_export.to_csv(output_dir / "mass_balance_reconciliation_monthly.csv", index=False)
reconciliation_summary_export.to_csv(output_dir / "mass_balance_reconciliation_summary.csv", index=False)
flag_days_export.to_csv(output_dir / "mass_balance_flagged_days.csv", index=False)
flag_summary_export.to_csv(output_dir / "mass_balance_flag_summary.csv", index=False)
reconciliation_narrative_export.to_csv(output_dir / "mass_balance_narrative_summary.csv", index=False)

with pd.ExcelWriter(output_dir / "mass_balance_reconciliation_pack.xlsx", engine="openpyxl") as writer:
    reconciliation_summary_export.to_excel(writer, sheet_name="Summary", index=False)
    reconciliation_monthly_export.to_excel(writer, sheet_name="Monthly", index=False)
    flag_summary_export.to_excel(writer, sheet_name="Flag Summary", index=False)
    flag_days_export.to_excel(writer, sheet_name="Flagged Days", index=False)
    reconciliation_narrative_export.to_excel(writer, sheet_name="Narrative", index=False)
    reconciliation_daily_export.to_excel(writer, sheet_name="Daily", index=False)

print(f"\nMass-balance outputs saved to: {output_dir.resolve()}")

# =============================================================================
# SITE QUESTIONS / DATA GAPS TABLE
# =============================================================================

questions_rows = [
    {
        "theme": "Sampling definitions",
        "question": "Can you confirm whether TK-1 E means Tank 1 inlet (Entrada) and TK-1 S means Tank 1 outlet (Salida)?",
        "why_it_matters": "This affects interpretation of tank-by-tank dissolution and cyanide consumption behaviour.",
        "priority": "High",
    },
    {
        "theme": "Sampling locations",
        "question": "Are Tanks 1, 6 and 8 the only routine sampling points, or are intermediate tank samples available?",
        "why_it_matters": "More intermediate samples would allow a better cyanide and metal progression profile through the circuit.",
        "priority": "High",
    },
    {
        "theme": "Percent solids",
        "question": "Can you provide actual leach feed % solids, and does it vary materially over time?",
        "why_it_matters": "This is required for a more reliable water and cyanide mass balance.",
        "priority": "High",
    },
    {
        "theme": "Slurry density",
        "question": "Can you provide slurry density or pulp density through the leach train?",
        "why_it_matters": "This improves residence time and solution flow estimates.",
        "priority": "High",
    },
    {
        "theme": "Cyanide dosing",
        "question": "Is all cyanide added at Tank 1, or are there additional dosing points further downstream?",
        "why_it_matters": "The current balance assumes all NaCN enters at Tank 1.",
        "priority": "High",
    },
    {
        "theme": "Cyanide solution strength",
        "question": "Can you confirm that the NaCN dosing solution is consistently 30%, and whether this varies operationally?",
        "why_it_matters": "This affects conversion between pure NaCN consumption and dosing solution flow.",
        "priority": "Medium",
    },
    {
        "theme": "Water balance",
        "question": "Can you provide process water, reclaim water, barren solution, wash water, and any recycle flowrates?",
        "why_it_matters": "A proper plant water and cyanide balance cannot be closed without these streams.",
        "priority": "High",
    },
    {
        "theme": "Tailings solution",
        "question": "Do you have tailings solution flowrate and tailings solution assays, not just tailings moisture?",
        "why_it_matters": "This is needed to estimate cyanide and dissolved copper losses leaving the circuit.",
        "priority": "High",
    },
    {
        "theme": "Merrill-Crowe circuit",
        "question": "Can you provide pregnant solution, clarified solution, barren return, and precipitate circuit flow and assay data?",
        "why_it_matters": "Merrill-Crowe recycle streams may materially affect copper and cyanide chemistry.",
        "priority": "High",
    },
    {
        "theme": "Copper mineralogy",
        "question": "Is the reported feed Cu total copper, soluble copper, acid-soluble copper, or cyanide-soluble copper?",
        "why_it_matters": "Total Cu may not explain cyanide demand as well as soluble/reactive Cu.",
        "priority": "High",
    },
    {
        "theme": "Metallurgy",
        "question": "Do you have mineralogy, sequential copper assays, or diagnostic leach/testwork showing what copper species are present?",
        "why_it_matters": "Different copper minerals consume cyanide very differently.",
        "priority": "High",
    },
    {
        "theme": "Lab methods",
        "question": "Can you confirm the lab methods for free CN and WAD CN, including units and reporting basis?",
        "why_it_matters": "The interpretation of free vs WAD vs complexed CN depends on method and unit consistency.",
        "priority": "High",
    },
    {
        "theme": "Lead data",
        "question": "In 2026 there are Pb columns. Are lead concentrations operationally important, and are 2025 Pb data available elsewhere?",
        "why_it_matters": "Pb may be relevant to solution chemistry and to consistency of the yearly dataset.",
        "priority": "Low",
    },
    {
        "theme": "Operating changes",
        "question": "Were there any material operating changes in 2026, such as ore source, blending, grind size, pH setpoint, oxygen strategy, or cyanide control strategy?",
        "why_it_matters": "The 2026 data appear materially different from 2025 and may reflect a step change in operation.",
        "priority": "High",
    },
    {
        "theme": "Ore blending",
        "question": "Can you provide ore source / pit / blend information by day or week?",
        "why_it_matters": "This may explain periods of high Cu, high WAD cyanide, and lower recovery.",
        "priority": "High",
    },
    {
        "theme": "Particle size",
        "question": "Do you have grind size or P80 data for the same dates?",
        "why_it_matters": "Particle size affects both dissolution kinetics and apparent reagent demand.",
        "priority": "Medium",
    },
    {
        "theme": "Residence time realism",
        "question": "Do you have actual tank operating volumes or level measurements?",
        "why_it_matters": "The current residence time uses nominal tank volume and may over- or under-estimate true RT.",
        "priority": "Medium",
    },
]

# -----------------------------------------------------------------------------
# Build table
# -----------------------------------------------------------------------------
questions_table = pd.DataFrame(questions_rows)

priority_order = pd.CategoricalDtype(
    categories=["High", "Medium", "Low"],
    ordered=True
)
questions_table["priority"] = questions_table["priority"].astype(priority_order)

questions_table = (
    questions_table
    .sort_values(["priority", "theme"], ascending=[True, True])
    .reset_index(drop=True)
)

questions_table.index = questions_table.index + 1
questions_table.index.name = "No."

display(Markdown("## Site Questions and Data Gaps"))
display(Markdown(
    "This table summarises the main follow-up questions and data gaps identified during the initial review of the La Coipa leaching dataset. "
    "Items have been prioritised based on likely impact on mass balance reliability, cyanide accounting, and interpretation of process behaviour."
))

# -----------------------------------------------------------------------------
# Notebook summary
# -----------------------------------------------------------------------------
priority_summary = (
    questions_table["priority"]
    .value_counts()
    .reindex(["High", "Medium", "Low"])
    .fillna(0)
    .astype(int)
)

theme_summary = questions_table["theme"].nunique()

display(Markdown(
    f"**Summary:** {len(questions_table)} questions identified across **{theme_summary} themes** "
    f"({priority_summary['High']} High, {priority_summary['Medium']} Medium, {priority_summary['Low']} Low priority)."
))

# -----------------------------------------------------------------------------
# Display-friendly copy
# -----------------------------------------------------------------------------
display_table = questions_table.rename(
    columns={
        "theme": "Theme",
        "question": "Question",
        "why_it_matters": "Why it matters",
        "priority": "Priority",
    }
)

def priority_colour(val):
    if val == "High":
        return "background-color: #fde2e1; color: #9f1239; font-weight: 600;"
    if val == "Medium":
        return "background-color: #fff4db; color: #92400e; font-weight: 600;"
    if val == "Low":
        return "background-color: #e8f3e8; color: #166534; font-weight: 600;"
    return ""

styled_questions = (
    display_table.style
    .applymap(priority_colour, subset=["Priority"])
    .set_properties(subset=["Theme", "Priority"], **{"text-align": "left", "vertical-align": "top"})
    .set_properties(subset=["Question", "Why it matters"], **{
        "text-align": "left",
        "white-space": "normal",
        "vertical-align": "top"
    })
    .set_table_styles([
        {"selector": "th", "props": [
            ("background-color", "#1f2937"),
            ("color", "white"),
            ("font-weight", "bold"),
            ("text-align", "left"),
            ("padding", "8px"),
            ("border", "1px solid #d1d5db")
        ]},
        {"selector": "td", "props": [
            ("padding", "8px"),
            ("border", "1px solid #e5e7eb")
        ]},
        {"selector": "caption", "props": [
            ("caption-side", "top"),
            ("font-size", "14px"),
            ("font-weight", "600"),
            ("text-align", "left"),
            ("padding", "0 0 8px 0")
        ]},
    ])
    .set_caption("Table: Site questions and data gaps identified from the current dataset review")
)

display(styled_questions)

# -----------------------------------------------------------------------------
# Export outputs
# -----------------------------------------------------------------------------
output_dir = Path("la_coipa_diagnostics_outputs")
output_dir.mkdir(exist_ok=True)

raw_csv_path = output_dir / "site_questions_data_gaps.csv"
xlsx_path = output_dir / "site_questions_data_gaps.xlsx"
html_path = output_dir / "site_questions_data_gaps.html"

display_table.to_csv(raw_csv_path, index=True)
display_table.to_excel(xlsx_path, index=True)
styled_questions.to_html(html_path)

display(Markdown(
    f"**Saved outputs:**\n"
    f"- CSV: `{raw_csv_path}`  \n"
    f"- Excel: `{xlsx_path}`  \n"
    f"- HTML: `{html_path}`"
))


# =============================================================================
# DASHBOARD
# =============================================================================

plot_df = dfo.copy().sort_values("date")

# -----------------------------------------------------------------------------
# 1. TIME SERIES DASHBOARD
# -----------------------------------------------------------------------------
fig = make_subplots(
	rows=4,
	cols=1,
	shared_xaxes=True,
	vertical_spacing=0.05,
	subplot_titles=(
		"NaCN Consumption and Specific Consumption",
		"Copper Behaviour",
		"Free CN / WAD / Complexed CN",
		"Gold Recovery and Residence Time",
	),
	specs=[[{"secondary_y": True}],
		   [{"secondary_y": False}],
		   [{"secondary_y": False}],
		   [{"secondary_y": True}]]
)

# Row 1
fig.add_trace(
	go.Scatter(
		x=plot_df["date"],
		y=plot_df["nacn_consumption_tpd"],
		mode="lines",
		name="NaCN Consumption (t/d)",
		hovertemplate="Date=%{x}<br>NaCN=%{y:.2f} t/d<extra></extra>",
	),
	row=1, col=1, secondary_y=False
)

fig.add_trace(
	go.Scatter(
		x=plot_df["date"],
		y=plot_df["specific_nacn_kgpt"],
		mode="lines",
		name="Specific NaCN (kg/t)",
		hovertemplate="Date=%{x}<br>Specific NaCN=%{y:.2f} kg/t<extra></extra>",
	),
	row=1, col=1, secondary_y=True
)

# Row 2
fig.add_trace(
	go.Scatter(
		x=plot_df["date"],
		y=plot_df["cu_feed_ppm"],
		mode="lines",
		name="Feed Cu (ppm)",
		hovertemplate="Date=%{x}<br>Feed Cu=%{y:.1f} ppm<extra></extra>",
	),
	row=2, col=1
)

fig.add_trace(
	go.Scatter(
		x=plot_df["date"],
		y=plot_df["cu_solution_ppm_avg"],
		mode="lines",
		name="Solution Cu Avg (ppm)",
		hovertemplate="Date=%{x}<br>Solution Cu=%{y:.1f} ppm<extra></extra>",
	),
	row=2, col=1
)

# Row 3
fig.add_trace(
	go.Scatter(
		x=plot_df["date"],
		y=plot_df["free_cn_ppm_avg"],
		mode="lines",
		name="Free CN Avg (ppm)",
		hovertemplate="Date=%{x}<br>Free CN=%{y:.1f} ppm<extra></extra>",
	),
	row=3, col=1
)

fig.add_trace(
	go.Scatter(
		x=plot_df["date"],
		y=plot_df["wad_gpl_avg"],
		mode="lines",
		name="WAD Avg (g/L)",
		hovertemplate="Date=%{x}<br>WAD=%{y:.2f} g/L<extra></extra>",
	),
	row=3, col=1
)

fig.add_trace(
	go.Scatter(
		x=plot_df["date"],
		y=plot_df["complexed_cn_gpl_avg"],
		mode="lines",
		name="Complexed CN Avg (g/L)",
		hovertemplate="Date=%{x}<br>Complexed CN=%{y:.2f} g/L<extra></extra>",
	),
	row=3, col=1
)

# Row 4
fig.add_trace(
	go.Scatter(
		x=plot_df["date"],
		y=plot_df["recovery_au_pct"],
		mode="lines",
		name="Au Recovery (%)",
		hovertemplate="Date=%{x}<br>Recovery=%{y:.2f}%<extra></extra>",
	),
	row=4, col=1, secondary_y=False
)

fig.add_trace(
	go.Scatter(
		x=plot_df["date"],
		y=plot_df["rt_hours_est_plot"],
		mode="lines",
		name="rt (hours, est.)",
		hovertemplate="Date=%{x}<br>rt=%{y:.1f} h<extra></extra>",
	),
	row=4, col=1, secondary_y=True
)

fig.update_yaxes(title_text="NaCN (t/d)", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="kg/t", row=1, col=1, secondary_y=True)
fig.update_yaxes(title_text="ppm", row=2, col=1)
fig.update_yaxes(title_text="CN concentration", row=3, col=1)
fig.update_yaxes(title_text="Recovery (%)", row=4, col=1, secondary_y=False)
fig.update_yaxes(title_text="Hours", row=4, col=1, secondary_y=True)

fig.update_layout(
	height=1200,
	width=1400,
	title="La Coipa Leach Circuit Diagnostic Dashboard",
	hovermode="x unified",
	legend=dict(
		orientation="h",
		yanchor="bottom",
		y=1.02,
		xanchor="left",
		x=0
	),
)

fig.show()

# -----------------------------------------------------------------------------
# 2. SCATTER DIAGNOSTICS
# -----------------------------------------------------------------------------

# Filter outliers using IQR
def iqr_mask(series, multiplier=1.5):
    s = pd.to_numeric(series, errors="coerce")
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1

    if pd.isna(iqr) or iqr == 0:
        return s.notna()

    lower = q1 - multiplier * iqr
    upper = q3 + multiplier * iqr
    return (s >= lower) & (s <= upper)


def iqr_filter_xy(df, x_col, y_col, multiplier=1.5):
    mask = iqr_mask(df[x_col], multiplier) & iqr_mask(df[y_col], multiplier)
    return df.loc[mask].copy()


def add_trendline(fig, df, x_col, y_col, row, col):
    if len(df) < 3:
        return

    x = df[x_col].values
    y = df[y_col].values

    coeffs = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = coeffs[0] * x_line + coeffs[1]

    fig.add_trace(
        go.Scatter(
            x=x_line,
            y=y_line,
            mode="lines",
            line=dict(width=2, dash="dash"),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=row, col=col
    )


def annotate_corr(fig, df, x_col, y_col, row, col):
    if len(df) < 3:
        return

    r = df[[x_col, y_col]].corr().iloc[0, 1]

    fig.add_annotation(
        x=0.98,
        y=0.95,
        xref="x domain",
        yref="y domain",
        text=f"r = {r:.2f}",
        showarrow=False,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.75)",
        bordercolor="rgba(100,100,100,0.35)",
        borderwidth=1,
        font=dict(size=11),
        row=row,
        col=col,
    )


scatter_config = [
    ("nacn_consumption_tpd", "NaCN (t/d)", "Solution Cu vs NaCN Consumption"),
    ("specific_nacn_kgpt", "Specific NaCN (kg/t)", "Solution Cu vs Specific NaCN"),
    ("free_cn_ppm_avg", "Free CN (ppm)", "Solution Cu vs Free CN"),
    ("complexed_cn_gpl_avg", "Complexed CN (g/L)", "Solution Cu vs Complexed CN"),
]

scatter_fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[c[2] for c in scatter_config],
)

positions = [(1,1), (1,2), (2,1), (2,2)]

for (y_col, y_label, title), (r, c) in zip(scatter_config, positions):

    raw_df = plot_df[["cu_solution_ppm_avg", y_col, "date", "throughput_tpd"]].dropna()
    filt_df = iqr_filter_xy(raw_df, "cu_solution_ppm_avg", y_col, multiplier=1.5)

    removed = len(raw_df) - len(filt_df)
    print(f"{title}: removed {removed} outliers ({removed/len(raw_df)*100:.1f}%)")

    scatter_fig.add_trace(
        go.Scatter(
            x=filt_df["cu_solution_ppm_avg"],
            y=filt_df[y_col],
            mode="markers",
            marker=dict(
                size=8,
                opacity=0.7,
                line=dict(width=0.5, color="white"),
            ),
            showlegend=False,
            text=filt_df["date"].astype(str),
            hovertemplate=(
                "Date=%{text}<br>"
                "Solution Cu=%{x:.1f} ppm<br>"
                f"{y_label}=%{{y:.2f}}<extra></extra>"
            ),
        ),
        row=r, col=c
    )

    # Trendline + correlation
    add_trendline(scatter_fig, filt_df, "cu_solution_ppm_avg", y_col, r, c)
    annotate_corr(scatter_fig, filt_df, "cu_solution_ppm_avg", y_col, r, c)

    scatter_fig.update_xaxes(title_text="Solution Cu Avg (ppm)", row=r, col=c)
    scatter_fig.update_yaxes(title_text=y_label, row=r, col=c)


scatter_fig.update_layout(
    height=900,
    width=1200,
    title="Copper–Cyanide Relationship Diagnostics (IQR-filtered)",
    template="plotly_white",
    margin=dict(l=50, r=30, t=70, b=50),
)

scatter_fig.show()


# -----------------------------------------------------------------------------
# OUTLIERS
# -----------------------------------------------------------------------------
outlier_df = raw_df.loc[~raw_df.index.isin(filt_df.index)]

if not outlier_df.empty:
    print("\nTop outliers (by Cu or response):")
    print(
        outlier_df.sort_values(
            ["cu_solution_ppm_avg", y_col],
            ascending=False
        ).head(10)
    )

# -----------------------------------------------------------------------------
# 3. TANK 1 PROGRESSION DIAGNOSTICS
# -----------------------------------------------------------------------------

tank_progression_plot = tank_progression.copy()

# Basic validation
required_cols = ["metric", "mean_inlet", "mean_outlet", "mean_delta", "direction"]
missing_cols = [c for c in required_cols if c not in tank_progression_plot.columns]
if missing_cols:
    raise ValueError(f"tank_progression is missing required columns: {missing_cols}")

# Clean labels / ordering
tank_progression_plot["metric"] = tank_progression_plot["metric"].astype(str)
tank_progression_plot["abs_delta"] = tank_progression_plot["mean_delta"].abs()
tank_progression_plot = tank_progression_plot.sort_values("abs_delta", ascending=True).reset_index(drop=True)

# Optional normalised change for cross-metric comparison
tank_progression_plot["pct_change_from_inlet"] = np.where(
    tank_progression_plot["mean_inlet"].abs() > 1e-12,
    100 * tank_progression_plot["mean_delta"] / tank_progression_plot["mean_inlet"],
    np.nan,
)

print("\nTank 1 progression summary:")
print(
    tank_progression_plot[
        ["metric", "mean_inlet", "mean_outlet", "mean_delta", "pct_change_from_inlet", "direction"]
    ].round(3).to_string(index=False)
)

# -------------------------------------------------------------------------
# 3A. DUMBBELL PLOT — mean inlet vs mean outlet
# -------------------------------------------------------------------------
fig_dumbbell = go.Figure()

# connector lines
for _, row in tank_progression_plot.iterrows():
    fig_dumbbell.add_trace(
        go.Scatter(
            x=[row["mean_inlet"], row["mean_outlet"]],
            y=[row["metric"], row["metric"]],
            mode="lines",
            line=dict(width=3, color="rgba(120,120,120,0.45)"),
            hoverinfo="skip",
            showlegend=False,
        )
    )

# inlet markers
fig_dumbbell.add_trace(
    go.Scatter(
        x=tank_progression_plot["mean_inlet"],
        y=tank_progression_plot["metric"],
        mode="markers",
        name="Mean inlet",
        marker=dict(
            size=11,
            symbol="circle",
            line=dict(width=1, color="white"),
        ),
        customdata=np.stack(
            [
                tank_progression_plot["mean_outlet"],
                tank_progression_plot["mean_delta"],
                tank_progression_plot["pct_change_from_inlet"],
            ],
            axis=1,
        ),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Mean inlet: %{x:.3f}<br>"
            "Mean outlet: %{customdata[0]:.3f}<br>"
            "Mean delta: %{customdata[1]:+.3f}<br>"
            "Change from inlet: %{customdata[2]:+.1f}%<extra></extra>"
        ),
    )
)

# outlet markers
fig_dumbbell.add_trace(
    go.Scatter(
        x=tank_progression_plot["mean_outlet"],
        y=tank_progression_plot["metric"],
        mode="markers",
        name="Mean outlet",
        marker=dict(
            size=11,
            symbol="diamond",
            line=dict(width=1, color="white"),
        ),
        customdata=np.stack(
            [
                tank_progression_plot["mean_inlet"],
                tank_progression_plot["mean_delta"],
                tank_progression_plot["pct_change_from_inlet"],
            ],
            axis=1,
        ),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Mean outlet: %{x:.3f}<br>"
            "Mean inlet: %{customdata[0]:.3f}<br>"
            "Mean delta: %{customdata[1]:+.3f}<br>"
            "Change from inlet: %{customdata[2]:+.1f}%<extra></extra>"
        ),
    )
)

fig_dumbbell.update_layout(
    title="Tank 1 progression: mean inlet vs mean outlet",
    template="plotly_white",
    height=max(420, 110 * len(tank_progression_plot)),
    margin=dict(l=70, r=40, t=70, b=50),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0,
    ),
)

fig_dumbbell.update_xaxes(title_text="Mean value")
fig_dumbbell.update_yaxes(title_text="Metric", automargin=True)

fig_dumbbell.show()


# -------------------------------------------------------------------------
# 3B. DELTA BAR CHART — average change across Tank 1
# -------------------------------------------------------------------------
delta_colors = np.where(
    tank_progression_plot["mean_delta"] >= 0,
    "Increase",
    "Decrease",
)

delta_plot_df = tank_progression_plot.copy()
delta_plot_df["delta_sign"] = delta_colors

fig_delta = px.bar(
    delta_plot_df,
    x="mean_delta",
    y="metric",
    color="delta_sign",
    orientation="h",
    text="mean_delta",
    title="Tank 1 progression: average change across the tank",
    labels={
        "mean_delta": "Mean outlet - inlet",
        "metric": "",
        "delta_sign": "Direction",
    },
)

fig_delta.update_traces(
    texttemplate="%{text:+.3f}",
    textposition="outside",
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Mean delta: %{x:+.3f}<extra></extra>"
    ),
)

fig_delta.update_layout(
    template="plotly_white",
    height=max(420, 110 * len(delta_plot_df)),
    margin=dict(l=70, r=40, t=70, b=50),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0,
    ),
)

fig_delta.update_yaxes(automargin=True)
fig_delta.show()


# -------------------------------------------------------------------------
# 3C. OPTIONAL NORMALISED VIEW — percent change from inlet
# -------------------------------------------------------------------------
pct_plot_df = tank_progression_plot.dropna(subset=["pct_change_from_inlet"]).copy()

if not pct_plot_df.empty:
    pct_plot_df["pct_sign"] = np.where(
        pct_plot_df["pct_change_from_inlet"] >= 0,
        "Increase",
        "Decrease",
    )

    fig_pct = px.bar(
        pct_plot_df,
        x="pct_change_from_inlet",
        y="metric",
        color="pct_sign",
        orientation="h",
        text="pct_change_from_inlet",
        title="Tank 1 progression: normalised change from inlet",
        labels={
            "pct_change_from_inlet": "Change from inlet (%)",
            "metric": "",
            "pct_sign": "Direction",
        },
    )

    fig_pct.update_traces(
        texttemplate="%{text:+.1f}%",
        textposition="outside",
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Change from inlet: %{x:+.1f}%<extra></extra>"
        ),
    )

    fig_pct.update_layout(
        template="plotly_white",
        height=max(420, 110 * len(pct_plot_df)),
        margin=dict(l=70, r=40, t=70, b=50),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0,
        ),
    )

    fig_pct.update_yaxes(automargin=True)
    fig_pct.show()


# -------------------------------------------------------------------------
# 3D. OPTIONAL QA/QC TABLE
# -------------------------------------------------------------------------
tank_progression_display = tank_progression_plot[
    ["metric", "mean_inlet", "mean_outlet", "mean_delta", "pct_change_from_inlet", "direction"]
].copy()

tank_progression_display = tank_progression_display.round({
    "mean_inlet": 3,
    "mean_outlet": 3,
    "mean_delta": 3,
    "pct_change_from_inlet": 1,
})

print("\nTank 1 progression QA/QC table:")
print(tank_progression_display.to_string(index=False))

try:
    from IPython.display import display
    display(tank_progression_display)
except Exception:
    pass

# -----------------------------------------------------------------------------
# 4. MONTHLY RECONCILIATION DASHBOARD (POLISHED / PROFESSIONAL)
# -----------------------------------------------------------------------------

monthly_plot = reconciliation_monthly.copy()

if "index" in monthly_plot.columns and "date" not in monthly_plot.columns:
    monthly_plot = monthly_plot.rename(columns={"index": "date"})

monthly_plot["date"] = pd.to_datetime(monthly_plot["date"], errors="coerce")
monthly_plot = monthly_plot.sort_values("date").reset_index(drop=True)

# Optional: light visual-only filtering for extreme monthly spikes if helper exists
def _safe_iqr_mask(series, multiplier=1.5):
    try:
        return get_iqr_mask(series, multiplier=multiplier)
    except Exception:
        s = pd.to_numeric(series, errors="coerce")
        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        if pd.isna(iqr) or iqr == 0:
            return s.notna()
        lower = q1 - multiplier * iqr
        upper = q3 + multiplier * iqr
        return ((s >= lower) & (s <= upper)).fillna(False)

monthly_plot_viz = monthly_plot.copy()

# Only filter clearly distortion-prone monthly fields for plotting, not the base table
for col in ["free_cn_accountability_pct", "wad_cn_accountability_pct"]:
    if col in monthly_plot_viz.columns:
        monthly_plot_viz[col] = monthly_plot_viz[col].where(_safe_iqr_mask(monthly_plot_viz[col], multiplier=1.5), np.nan)

# Clean colour system
COLORS = {
    "nacn": "#4F6BED",
    "cu": "#E4572E",
    "free_acc": "#17B890",
    "wad_acc": "#9B5DE5",
    "au": "#F29E4C",
    "ag": "#12B5E5",
    "grid": "rgba(160,174,192,0.20)",
    "zero": "rgba(100,116,139,0.35)",
    "text": "#334E75",
}

def add_end_label(fig, x, y, text, color, row, col, secondary_y=None, yshift=0):
    if len(x) == 0 or len(y) == 0:
        return
    valid = pd.notna(pd.Series(y))
    if valid.sum() == 0:
        return

    x_last = pd.Series(x)[valid].iloc[-1]
    y_last = pd.Series(y)[valid].iloc[-1]

    fig.add_annotation(
        x=x_last,
        y=y_last,
        text=text,
        showarrow=False,
        xanchor="left",
        yanchor="middle",
        xshift=10,
        yshift=yshift,
        font=dict(size=11, color=color),
        bgcolor="rgba(255,255,255,0.75)",
        bordercolor="rgba(100,100,100,0.18)",
        borderwidth=1,
        row=row,
        col=col,
        secondary_y=secondary_y,
    )

def line_trace(x, y, name, color, hover_label, dash=None):
    return go.Scatter(
        x=x,
        y=y,
        mode="lines+markers",
        name=name,
        line=dict(color=color, width=2.8, dash=dash or "solid"),
        marker=dict(size=6, color=color, line=dict(width=0.8, color="white")),
        hovertemplate=hover_label + "<extra></extra>",
    )

monthly_fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.09,
    subplot_titles=(
        "Monthly NaCN input and copper in solution",
        "Monthly cyanide accountability",
        "Monthly gold and silver extracted",
    ),
    specs=[
        [{"secondary_y": True}],
        [{"secondary_y": False}],
        [{"secondary_y": False}],
    ],
)

# -------------------------------------------------------------------------
# Row 1: NaCN + Cu
# -------------------------------------------------------------------------
monthly_fig.add_trace(
    line_trace(
        monthly_plot_viz["date"],
        monthly_plot_viz["nacn_input_tpd"],
        "NaCN input (t/d)",
        COLORS["nacn"],
        "Month=%{x|%b %Y}<br>NaCN input=%{y:,.2f} t/d",
    ),
    row=1, col=1, secondary_y=False
)

monthly_fig.add_trace(
    line_trace(
        monthly_plot_viz["date"],
        monthly_plot_viz["cu_ppm_tk_8_s"],
        "Cu in TK8 solution (ppm)",
        COLORS["cu"],
        "Month=%{x|%b %Y}<br>Cu in TK8 solution=%{y:,.0f} ppm",
    ),
    row=1, col=1, secondary_y=True
)

# -------------------------------------------------------------------------
# Row 2: Accountability
# -------------------------------------------------------------------------
monthly_fig.add_trace(
    line_trace(
        monthly_plot_viz["date"],
        monthly_plot_viz["free_cn_accountability_pct"],
        "Free CN accountability (%)",
        COLORS["free_acc"],
        "Month=%{x|%b %Y}<br>Free CN accountability=%{y:,.1f}%",
    ),
    row=2, col=1
)

monthly_fig.add_trace(
    line_trace(
        monthly_plot_viz["date"],
        monthly_plot_viz["wad_cn_accountability_pct"],
        "WAD accountability (%)",
        COLORS["wad_acc"],
        "Month=%{x|%b %Y}<br>WAD accountability=%{y:,.1f}%",
    ),
    row=2, col=1
)

# Optional reference line at 100%
monthly_fig.add_hline(
    y=100,
    line_width=1.2,
    line_dash="dot",
    line_color=COLORS["zero"],
    row=2,
    col=1,
)

# -------------------------------------------------------------------------
# Row 3: Au / Ag extraction
# -------------------------------------------------------------------------
monthly_fig.add_trace(
    line_trace(
        monthly_plot_viz["date"],
        monthly_plot_viz["au_extracted_gpd"],
        "Au extracted (g/d)",
        COLORS["au"],
        "Month=%{x|%b %Y}<br>Au extracted=%{y:,.0f} g/d",
    ),
    row=3, col=1
)

monthly_fig.add_trace(
    line_trace(
        monthly_plot_viz["date"],
        monthly_plot_viz["ag_extracted_gpd"],
        "Ag extracted (g/d)",
        COLORS["ag"],
        "Month=%{x|%b %Y}<br>Ag extracted=%{y:,.0f} g/d",
    ),
    row=3, col=1
)

# -------------------------------------------------------------------------
# Axes
# -------------------------------------------------------------------------
monthly_fig.update_yaxes(
    title_text="NaCN input (t/d)",
    row=1, col=1, secondary_y=False,
    showgrid=True, gridcolor=COLORS["grid"],
    zeroline=False,
    tickformat=",.0f"
)

monthly_fig.update_yaxes(
    title_text="Cu in TK8 solution (ppm)",
    row=1, col=1, secondary_y=True,
    showgrid=False,
    zeroline=False,
    tickformat=",.0f"
)

monthly_fig.update_yaxes(
    title_text="Accountability (%)",
    row=2, col=1,
    showgrid=True, gridcolor=COLORS["grid"],
    zeroline=False,
    tickformat=",.0f"
)

monthly_fig.update_yaxes(
    title_text="Extracted metal (g/d)",
    row=3, col=1,
    showgrid=True, gridcolor=COLORS["grid"],
    zeroline=False,
    tickformat="~s"
)

monthly_fig.update_xaxes(
    showgrid=False,
    tickformat="%b %Y",
    ticks="outside",
    tickangle=0,
)

# -------------------------------------------------------------------------
# End-of-line labels
# -------------------------------------------------------------------------
add_end_label(
    monthly_fig,
    monthly_plot_viz["date"],
    monthly_plot_viz["nacn_input_tpd"],
    "NaCN input",
    COLORS["nacn"],
    row=1, col=1, secondary_y=False, yshift=-10
)

add_end_label(
    monthly_fig,
    monthly_plot_viz["date"],
    monthly_plot_viz["cu_ppm_tk_8_s"],
    "Cu in solution",
    COLORS["cu"],
    row=1, col=1, secondary_y=True, yshift=10
)

add_end_label(
    monthly_fig,
    monthly_plot_viz["date"],
    monthly_plot_viz["free_cn_accountability_pct"],
    "Free CN",
    COLORS["free_acc"],
    row=2, col=1, yshift=-10
)

add_end_label(
    monthly_fig,
    monthly_plot_viz["date"],
    monthly_plot_viz["wad_cn_accountability_pct"],
    "WAD",
    COLORS["wad_acc"],
    row=2, col=1, yshift=10
)

add_end_label(
    monthly_fig,
    monthly_plot_viz["date"],
    monthly_plot_viz["au_extracted_gpd"],
    "Au extracted",
    COLORS["au"],
    row=3, col=1, yshift=-10
)

add_end_label(
    monthly_fig,
    monthly_plot_viz["date"],
    monthly_plot_viz["ag_extracted_gpd"],
    "Ag extracted",
    COLORS["ag"],
    row=3, col=1, yshift=10
)

# -------------------------------------------------------------------------
# Layout polish
# -------------------------------------------------------------------------
monthly_fig.update_layout(
    title=dict(
        text="Monthly Reconciliation Dashboard",
        x=0.02,
        xanchor="left",
        font=dict(size=24, color=COLORS["text"]),
    ),
    template="plotly_white",
    height=980,
    width=1400,
    hovermode="x unified",
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=80, r=140, t=90, b=60),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0,
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="rgba(100,100,100,0.15)",
        borderwidth=1,
        font=dict(size=11),
        tracegroupgap=10,
    ),
    font=dict(
        family="Arial, sans-serif",
        size=12,
        color=COLORS["text"],
    ),
)

# Light subplot title polish
for ann in monthly_fig.layout.annotations:
    ann.font = dict(size=16, color=COLORS["text"])

# Optional range slider for easier monthly navigation
monthly_fig.update_xaxes(
    rangeslider_visible=False
)

monthly_fig.show()


# =============================================================================
# ADVANCED ANALYSIS REPORTING PACK
# =============================================================================

try:
    from IPython.display import display, Markdown
    IN_NOTEBOOK = True
except ImportError:
    IN_NOTEBOOK = False

output_dir = Path("la_coipa_diagnostics_outputs")
output_dir.mkdir(exist_ok=True)

# -----------------------------------------------------------------------------
# 1. HELPERS
# -----------------------------------------------------------------------------

def safe_show_table(df: pd.DataFrame, caption: str):
    if df is None:
        df = pd.DataFrame()
    print(df.to_string(index=False))
    if IN_NOTEBOOK:
        display(style_table(df, caption))

def classify_model_quality(r2):
    if pd.isna(r2):
        return "Unknown"
    if r2 >= 0.60:
        return "Strong"
    if r2 >= 0.30:
        return "Moderate"
    if r2 >= 0.10:
        return "Weak"
    return "Poor"

def narrative_flag(text, ok):
    return "OK" if ok else f"CHECK: {text}"

def empty_df(columns):
    return pd.DataFrame(columns=columns)

def ensure_columns(df: pd.DataFrame, columns):
    if df is None or df.empty:
        return pd.DataFrame(columns=columns)
    for c in columns:
        if c not in df.columns:
            df[c] = np.nan
    return df[columns].copy()

def rounded_copy(df: pd.DataFrame, round_map: dict):
    out = df.copy()
    for c, ndp in round_map.items():
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").round(ndp)
    return out

def has_cols(df: pd.DataFrame, cols):
    return df is not None and all(c in df.columns for c in cols)

def write_excel_sheet(writer, df: pd.DataFrame, sheet_name: str):
    if df is None:
        df = pd.DataFrame()
    df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

# -----------------------------------------------------------------------------
# 2. BASE DATA PREP
# -----------------------------------------------------------------------------
if "dfo" not in globals():
    raise NameError("This block expects a DataFrame called 'dfo' to already exist.")

dfo = dfo.copy()

# Ensure a date column exists where possible
if "date" in dfo.columns:
    dfo["date"] = pd.to_datetime(dfo["date"], errors="coerce")

# Basic derived metrics if missing
if "cn_efficiency_index" not in dfo.columns:
    if "recovery_au_pct" in dfo.columns and "specific_nacn_kgpt" in dfo.columns:
        dfo["cn_efficiency_index"] = (
            pd.to_numeric(dfo["recovery_au_pct"], errors="coerce") /
            pd.to_numeric(dfo["specific_nacn_kgpt"], errors="coerce").replace(0, np.nan)
        )
    else:
        dfo["cn_efficiency_index"] = np.nan

if "do_avg" not in dfo.columns:
    do_cols = [c for c in dfo.columns if c.lower().startswith("do_")]
    if do_cols:
        dfo["do_avg"] = dfo[do_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
    else:
        dfo["do_avg"] = np.nan

# -----------------------------------------------------------------------------
# 3. MODELLING
# -----------------------------------------------------------------------------
features = [
    "cu_solution_ppm_avg",
    "throughput_tpd",
    "do_avg",
    "ph_tk_8_s",
]
target = "specific_nacn_kgpt"

available_features = [c for c in features if c in dfo.columns]
required_for_model = available_features + ([target] if target in dfo.columns else [])

if len(available_features) >= 2 and target in dfo.columns:
    model_df = dfo[required_for_model].apply(pd.to_numeric, errors="coerce").dropna().copy()
else:
    model_df = pd.DataFrame(columns=required_for_model)

lin_model = None
rf_model = None
lin_r2 = np.nan
lin_mae = np.nan
rf_r2 = np.nan
rf_mae = np.nan
coef_table = empty_df(["feature", "coefficient"])
rf_importance = empty_df(["feature", "importance"])
X_train = pd.DataFrame()
X_test = pd.DataFrame()
y_train = pd.Series(dtype=float)
y_test = pd.Series(dtype=float)

if len(model_df) >= 10 and len(available_features) >= 2:
    X = model_df[available_features]
    y = model_df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Linear model
    lin_model = LinearRegression()
    lin_model.fit(X_train, y_train)
    y_pred_lin = lin_model.predict(X_test)

    lin_r2 = r2_score(y_test, y_pred_lin)
    lin_mae = mean_absolute_error(y_test, y_pred_lin)

    coef_table = pd.DataFrame({
        "feature": available_features,
        "coefficient": lin_model.coef_,
    }).sort_values("coefficient", key=lambda s: s.abs(), ascending=False).reset_index(drop=True)

    # Random forest
    rf_model = RandomForestRegressor(
        n_estimators=200,
        random_state=42
    )
    rf_model.fit(X_train, y_train)
    y_pred_rf = rf_model.predict(X_test)

    rf_r2 = r2_score(y_test, y_pred_rf)
    rf_mae = mean_absolute_error(y_test, y_pred_rf)

    rf_importance = pd.DataFrame({
        "feature": available_features,
        "importance": rf_model.feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

print(f"Linear: R²={fmt_num(lin_r2)}, MAE={fmt_num(lin_mae, 2)}")
print(f"RF:     R²={fmt_num(rf_r2)}, MAE={fmt_num(rf_mae, 2)}")

# -----------------------------------------------------------------------------
# 4. SUPPORT OBJECTS / DEFENSIVE DEFAULTS
# -----------------------------------------------------------------------------
# Lead-lag table
if "best_lags" not in globals() or best_lags is None:
    best_lags = empty_df(["driver", "target", "lag_days", "corr", "abs_corr"])

# Build a simple lead-lag table if absent / empty
if best_lags.empty:
    lag_pairs = [
        ("cu_solution_ppm_avg", "specific_nacn_kgpt"),
        ("cu_solution_ppm_avg", "free_cn_ppm_avg"),
        ("cu_solution_ppm_avg", "complexed_cn_gpl_avg"),
        ("cu_solution_ppm_avg", "recovery_au_pct"),
        ("wad_gpl_avg", "complexed_cn_gpl_avg"),
        ("free_cn_ppm_avg", "recovery_au_pct"),
        ("throughput_tpd", "specific_nacn_kgpt"),
    ]

    lag_results = []
    max_lag_days = 7

    for driver, target_col in lag_pairs:
        if driver not in dfo.columns or target_col not in dfo.columns:
            continue

        pair_df = dfo[[driver, target_col]].apply(pd.to_numeric, errors="coerce").dropna().copy()
        if len(pair_df) < 10:
            continue

        for lag in range(0, max_lag_days + 1):
            shifted_driver = pair_df[driver].shift(lag)
            aligned = pd.DataFrame({
                "driver_vals": shifted_driver,
                "target_vals": pair_df[target_col],
            }).dropna()

            if len(aligned) < 10:
                continue

            corr = aligned["driver_vals"].corr(aligned["target_vals"])
            if pd.notna(corr):
                lag_results.append({
                    "driver": driver,
                    "target": target_col,
                    "lag_days": lag,
                    "corr": corr,
                    "abs_corr": abs(corr),
                })

    if lag_results:
        lag_df = pd.DataFrame(lag_results)
        best_lags = (
            lag_df.sort_values(["driver", "target", "abs_corr"], ascending=[True, True, False])
            .groupby(["driver", "target"], as_index=False)
            .first()
            .sort_values("abs_corr", ascending=False)
            .reset_index(drop=True)
        )

# Regime counts
if "cu_regime" in dfo.columns:
    regime_counts = dfo["cu_regime"].value_counts(dropna=False)
else:
    regime_counts = pd.Series(dtype="int64")

# Regime summary
if "regime_summary" not in globals() or regime_summary is None or regime_summary.empty:
    if "cu_regime" in dfo.columns:
        summary_cols = [
            c for c in [
                "specific_nacn_kgpt",
                "free_cn_ppm_avg",
                "complexed_cn_gpl_avg",
                "wad_gpl_avg",
                "recovery_au_pct",
                "cu_solution_ppm_avg",
                "cn_efficiency_index",
            ] if c in dfo.columns
        ]
        if summary_cols:
            regime_summary = (
                dfo.groupby("cu_regime")[summary_cols]
                .median(numeric_only=True)
                .reset_index()
            )
        else:
            regime_summary = pd.DataFrame()
    else:
        regime_summary = pd.DataFrame()

# Guidance bands
if "guidance_bands" not in globals() or guidance_bands is None or guidance_bands.empty:
    if "cu_regime" in dfo.columns:
        band_source_cols = [
            c for c in [
                "specific_nacn_kgpt",
                "free_cn_ppm_avg",
                "complexed_cn_gpl_avg",
                "wad_gpl_avg",
                "recovery_au_pct",
            ] if c in dfo.columns
        ]

        if band_source_cols:
            gb_list = []
            for regime, grp in dfo.groupby("cu_regime"):
                row = {"cu_regime": regime}
                for col in band_source_cols:
                    series = pd.to_numeric(grp[col], errors="coerce").dropna()
                    row[f"{col}_p25"] = series.quantile(0.25) if len(series) else np.nan
                    row[f"{col}_p50"] = series.quantile(0.50) if len(series) else np.nan
                    row[f"{col}_p75"] = series.quantile(0.75) if len(series) else np.nan
                gb_list.append(row)
            guidance_bands = pd.DataFrame(gb_list).sort_values("cu_regime").reset_index(drop=True)
        else:
            guidance_bands = pd.DataFrame()
    else:
        guidance_bands = pd.DataFrame()

# Recommendation table
if "recommendation_table" not in globals() or recommendation_table is None:
    recommendation_table = pd.DataFrame()

if recommendation_table.empty and not guidance_bands.empty:
    rec_rows = []
    for _, row in guidance_bands.iterrows():
        rec_rows.append({
            "cu_regime": row.get("cu_regime", np.nan),
            "specific_nacn_guidance_p50": row.get("specific_nacn_kgpt_p50", np.nan),
            "specific_nacn_guidance_p25": row.get("specific_nacn_kgpt_p25", np.nan),
            "specific_nacn_guidance_p75": row.get("specific_nacn_kgpt_p75", np.nan),
            "free_cn_guidance_p50": row.get("free_cn_ppm_avg_p50", np.nan),
            "recovery_reference_p50": row.get("recovery_au_pct_p50", np.nan),
        })
    recommendation_table = pd.DataFrame(rec_rows)

# Efficiency source
if "efficiency_df" not in globals() or efficiency_df is None or efficiency_df.empty:
    efficiency_df = dfo.copy()

# Best / worst days based on efficiency, excluding zero-dose artefacts
efficiency_base = dfo.copy()
if "specific_nacn_kgpt" in efficiency_base.columns:
    efficiency_base["specific_nacn_kgpt"] = pd.to_numeric(efficiency_base["specific_nacn_kgpt"], errors="coerce")
if "recovery_au_pct" in efficiency_base.columns:
    efficiency_base["recovery_au_pct"] = pd.to_numeric(efficiency_base["recovery_au_pct"], errors="coerce")
if "cn_efficiency_index" in efficiency_base.columns:
    efficiency_base["cn_efficiency_index"] = pd.to_numeric(efficiency_base["cn_efficiency_index"], errors="coerce")

credible_eff_mask = pd.Series(True, index=efficiency_base.index)
if "specific_nacn_kgpt" in efficiency_base.columns:
    credible_eff_mask &= efficiency_base["specific_nacn_kgpt"].gt(0)
if "cn_efficiency_index" in efficiency_base.columns:
    credible_eff_mask &= efficiency_base["cn_efficiency_index"].notna()

credible_eff_df = efficiency_base.loc[credible_eff_mask].copy()

if len(credible_eff_df):
    best_days = credible_eff_df.sort_values("cn_efficiency_index", ascending=False).head(20).copy()
    worst_days = credible_eff_df.sort_values("cn_efficiency_index", ascending=True).head(20).copy()
else:
    best_days = pd.DataFrame()
    worst_days = pd.DataFrame()

# -----------------------------------------------------------------------------
# 5. QA / INTERPRETATION FLAGS
# -----------------------------------------------------------------------------
qa_rows = []

qa_rows.append({
    "area": "Linear model",
    "status": narrative_flag(
        "Linear model has negative R² and should not be used operationally.",
        pd.notna(lin_r2) and lin_r2 >= 0
    ),
    "detail": f"R² = {fmt_num(lin_r2)}, MAE = {fmt_num(lin_mae, 2)}"
})

qa_rows.append({
    "area": "Random forest",
    "status": narrative_flag(
        "Random forest has weak explanatory power; treat as exploratory only.",
        pd.notna(rf_r2) and rf_r2 >= 0.20
    ),
    "detail": f"R² = {fmt_num(rf_r2)}, MAE = {fmt_num(rf_mae, 2)}"
})

singleton_regimes = int((regime_counts <= 3).sum()) if len(regime_counts) else 0
qa_rows.append({
    "area": "Regime analysis",
    "status": narrative_flag(
        "One or more regimes are too small to interpret reliably.",
        singleton_regimes == 0
    ),
    "detail": f"Regime counts = {dict(regime_counts.sort_index())}"
})

zero_specific_days = 0
if "specific_nacn_kgpt" in dfo.columns:
    zero_specific_days = int(pd.to_numeric(dfo["specific_nacn_kgpt"], errors="coerce").fillna(0).le(0).sum())

qa_rows.append({
    "area": "Efficiency index",
    "status": narrative_flag(
        "Some days have zero or near-zero specific NaCN values and should be excluded from efficiency ranking.",
        zero_specific_days == 0
    ),
    "detail": f"Days with specific NaCN <= 0: {zero_specific_days}"
})

qa_summary = pd.DataFrame(qa_rows)

print_section("QA / interpretation checks")
safe_show_table(qa_summary, "QA / interpretation checks")

# -----------------------------------------------------------------------------
# 6. EXECUTIVE SUMMARY TABLE
# -----------------------------------------------------------------------------
top_rf_features = rf_importance["feature"].head(5).tolist() if not rf_importance.empty else []

best_lag_nontrivial = best_lags.copy()
if not best_lags.empty and has_cols(best_lags, ["driver", "target"]):
    best_lag_nontrivial = best_lags[
        ~(
            ((best_lags["driver"] == "wad_gpl_avg") & (best_lags["target"] == "complexed_cn_gpl_avg")) |
            ((best_lags["driver"] == "cu_solution_ppm_avg") & (best_lags["target"] == "complexed_cn_gpl_avg"))
        )
    ].copy()

top_leadlag_text = (
    f"{best_lag_nontrivial.iloc[0]['driver']} -> {best_lag_nontrivial.iloc[0]['target']} "
    f"(lag {int(best_lag_nontrivial.iloc[0]['lag_days'])} d, r={best_lag_nontrivial.iloc[0]['corr']:.2f})"
    if not best_lag_nontrivial.empty and has_cols(best_lag_nontrivial, ["driver", "target", "lag_days", "corr"])
    else "No robust non-trivial lead-lag relationship identified."
)

rf_feature_text = ", ".join(top_rf_features) if top_rf_features else "No RF feature importance available."

exec_rows = [
    {
        "finding": "Copper remains the dominant chemistry signal in the dataset.",
        "evidence": f"Top RF features: {rf_feature_text}",
        "implication": "Any guidance logic should continue to centre on dissolved copper regime and free cyanide response."
    },
    {
        "finding": "The current predictive models are not yet strong enough for operational forecasting.",
        "evidence": f"Linear R² = {fmt_num(lin_r2)}; RF R² = {fmt_num(rf_r2)}.",
        "implication": "This section should be framed as exploratory guidance and diagnostic ranking, not final production forecasting."
    },
    {
        "finding": "The regime analysis needs refinement before being treated as a formal operating-mode classifier.",
        "evidence": f"Regime counts = {dict(regime_counts.sort_index())}.",
        "implication": "Reduce or rebalance the cluster structure before presenting regimes as stable operating states."
    },
    {
        "finding": "The strongest practical guidance currently comes from empirical Cu regime bands rather than from predictive models.",
        "evidence": "Guidance bands by Cu regime show practical step changes in reagent demand and performance where sufficient data exists.",
        "implication": "A rules-based or analogue-style guidance system is more defensible at this stage than a pure ML forecaster."
    },
    {
        "finding": "Lead-lag results are currently more useful for diagnostics than for control logic.",
        "evidence": top_leadlag_text,
        "implication": "Use these relationships to support interpretation and monitoring, not as a standalone operating rule."
    },
    {
        "finding": "Efficiency rankings should exclude zero-dose artefacts.",
        "evidence": f"Days with specific NaCN <= 0: {zero_specific_days}.",
        "implication": "Guard the efficiency metric before using it in recommendations or benchmarking."
    },
]

executive_summary_advanced = pd.DataFrame(exec_rows)

print_section("Advanced analysis executive summary")
safe_show_table(executive_summary_advanced, "Advanced analysis executive summary")

# -----------------------------------------------------------------------------
# 7. MODEL PERFORMANCE SUMMARY TABLE
# -----------------------------------------------------------------------------
model_perf = pd.DataFrame([
    {
        "model": "Linear regression",
        "rows_used": len(model_df),
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "r2": round(lin_r2, 3) if pd.notna(lin_r2) else np.nan,
        "mae": round(lin_mae, 3) if pd.notna(lin_mae) else np.nan,
        "quality": classify_model_quality(lin_r2),
    },
    {
        "model": "Random forest",
        "rows_used": len(model_df),
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "r2": round(rf_r2, 3) if pd.notna(rf_r2) else np.nan,
        "mae": round(rf_mae, 3) if pd.notna(rf_mae) else np.nan,
        "quality": classify_model_quality(rf_r2),
    },
])

print_section("Model performance")
safe_show_table(model_perf, "Model performance summary")

# -----------------------------------------------------------------------------
# 8. CLEANED TABLES FOR DISPLAY
# -----------------------------------------------------------------------------
best_lags_display = rounded_copy(
    ensure_columns(best_lags, ["driver", "target", "lag_days", "corr", "abs_corr"]),
    {"corr": 3, "abs_corr": 3}
)

coef_display = rounded_copy(
    ensure_columns(coef_table, ["feature", "coefficient"]),
    {"coefficient": 4}
)

rf_importance_display = rounded_copy(
    ensure_columns(rf_importance.head(15), ["feature", "importance"]),
    {"importance": 4}
)

regime_counts_display = regime_counts.reset_index()
if not regime_counts_display.empty:
    regime_counts_display.columns = ["regime", "count"]
else:
    regime_counts_display = pd.DataFrame(columns=["regime", "count"])

regime_summary_display = regime_summary.reset_index(drop=True) if regime_summary is not None else pd.DataFrame()
guidance_bands_display = guidance_bands.reset_index(drop=True) if guidance_bands is not None else pd.DataFrame()

display_cols = [
    "date",
    "cu_solution_ppm_avg",
    "specific_nacn_kgpt",
    "free_cn_ppm_avg",
    "complexed_cn_gpl_avg",
    "recovery_au_pct",
    "cn_efficiency_index",
]

best_days_display = ensure_columns(best_days, [c for c in display_cols if c in best_days.columns])
worst_days_display = ensure_columns(worst_days, [c for c in display_cols if c in worst_days.columns])

if "date" in best_days_display.columns:
    best_days_display["date"] = pd.to_datetime(best_days_display["date"], errors="coerce").dt.strftime("%Y-%m-%d")
if "date" in worst_days_display.columns:
    worst_days_display["date"] = pd.to_datetime(worst_days_display["date"], errors="coerce").dt.strftime("%Y-%m-%d")

best_days_display = rounded_copy(
    best_days_display.head(15),
    {
        "cu_solution_ppm_avg": 3,
        "specific_nacn_kgpt": 3,
        "free_cn_ppm_avg": 3,
        "complexed_cn_gpl_avg": 3,
        "recovery_au_pct": 3,
        "cn_efficiency_index": 3,
    }
)

worst_days_display = rounded_copy(
    worst_days_display.head(15),
    {
        "cu_solution_ppm_avg": 3,
        "specific_nacn_kgpt": 3,
        "free_cn_ppm_avg": 3,
        "complexed_cn_gpl_avg": 3,
        "recovery_au_pct": 3,
        "cn_efficiency_index": 3,
    }
)

print_section("Best lead-lag relationships")
safe_show_table(best_lags_display, "Best lead-lag relationships")

print_section("Linear coefficients")
safe_show_table(coef_display, "Linear model coefficients")

print_section("Top random forest features")
safe_show_table(rf_importance_display, "Top random forest features")

print_section("Operating regime counts")
safe_show_table(regime_counts_display, "Operating regime counts")

print_section("Operating regime summary")
safe_show_table(regime_summary_display, "Operating regime summary")

print_section("Guidance bands by Cu regime")
safe_show_table(guidance_bands_display, "Guidance bands by Cu regime")

print_section("Most efficient days")
safe_show_table(best_days_display, "Most efficient days")

print_section("Least efficient days")
safe_show_table(worst_days_display, "Least efficient days")

print_section("Recommendation table")
safe_show_table(recommendation_table, "Recommendation table")

# -----------------------------------------------------------------------------
# 9. CHARTS
# -----------------------------------------------------------------------------
# 9A. Lead-lag chart
if not best_lags_display.empty and has_cols(best_lags_display, ["driver", "target", "abs_corr", "lag_days"]):
    leadlag_heat = best_lags_display.copy()
    leadlag_heat["pair"] = leadlag_heat["driver"].astype(str) + " → " + leadlag_heat["target"].astype(str)

    fig_leadlag = px.bar(
        leadlag_heat.sort_values("abs_corr", ascending=True),
        x="abs_corr",
        y="pair",
        color="lag_days",
        orientation="h",
        title="Best lead-lag relationships by driver-target pair",
        labels={"abs_corr": "|Correlation|", "pair": "", "lag_days": "Lag (days)"},
        hover_data={"corr": True, "lag_days": True, "driver": True, "target": True, "abs_corr": False},
    )
    fig_leadlag.update_layout(
        template="plotly_white",
        height=620,
        title_x=0.02,
        margin=dict(l=80, r=30, t=70, b=50),
    )
    fig_leadlag.show()

# 9B. Model performance
if not model_perf.empty and "r2" in model_perf.columns:
    fig_perf = px.bar(
        model_perf,
        x="model",
        y="r2",
        color="quality",
        text="r2",
        title="Model performance comparison (R²)",
        labels={"model": "", "r2": "R²"},
    )
    fig_perf.update_traces(texttemplate="%{text}", textposition="outside")
    fig_perf.update_layout(
        template="plotly_white",
        height=420,
        title_x=0.02,
        margin=dict(l=40, r=30, t=70, b=40),
    )
    fig_perf.add_hline(y=0, line_dash="dot", line_color="rgba(100,100,100,0.5)")
    fig_perf.show()

# 9C. Top RF features
if not rf_importance_display.empty and has_cols(rf_importance_display, ["feature", "importance"]):
    rf_plot = rf_importance_display.head(12).sort_values("importance", ascending=True)
    fig_rf = px.bar(
        rf_plot,
        x="importance",
        y="feature",
        orientation="h",
        title="Top random forest features",
        labels={"importance": "Importance", "feature": ""},
    )
    fig_rf.update_layout(
        template="plotly_white",
        height=520,
        title_x=0.02,
        margin=dict(l=80, r=30, t=70, b=50),
    )
    fig_rf.show()

# 9D. Guidance bands by Cu regime
guidance_required = [
    "cu_regime",
    "specific_nacn_kgpt_p25", "specific_nacn_kgpt_p50", "specific_nacn_kgpt_p75",
    "free_cn_ppm_avg_p25", "free_cn_ppm_avg_p50", "free_cn_ppm_avg_p75",
    "recovery_au_pct_p50"
]
if has_cols(guidance_bands, guidance_required):
    guidance_plot = guidance_bands[guidance_required].copy()

    fig_guidance = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Specific NaCN guidance band", "Free CN and recovery by Cu regime"),
        horizontal_spacing=0.12,
        specs=[[{"secondary_y": False}, {"secondary_y": True}]]
    )

    fig_guidance.add_trace(
        go.Scatter(
            x=guidance_plot["cu_regime"],
            y=guidance_plot["specific_nacn_kgpt_p50"],
            mode="lines+markers",
            name="Specific NaCN p50",
            error_y=dict(
                type="data",
                symmetric=False,
                array=guidance_plot["specific_nacn_kgpt_p75"] - guidance_plot["specific_nacn_kgpt_p50"],
                arrayminus=guidance_plot["specific_nacn_kgpt_p50"] - guidance_plot["specific_nacn_kgpt_p25"],
            ),
            hovertemplate="Regime=%{x}<br>Specific NaCN p50=%{y:.2f} kg/t<extra></extra>",
        ),
        row=1, col=1
    )

    fig_guidance.add_trace(
        go.Scatter(
            x=guidance_plot["cu_regime"],
            y=guidance_plot["free_cn_ppm_avg_p50"],
            mode="lines+markers",
            name="Free CN p50",
            error_y=dict(
                type="data",
                symmetric=False,
                array=guidance_plot["free_cn_ppm_avg_p75"] - guidance_plot["free_cn_ppm_avg_p50"],
                arrayminus=guidance_plot["free_cn_ppm_avg_p50"] - guidance_plot["free_cn_ppm_avg_p25"],
            ),
            hovertemplate="Regime=%{x}<br>Free CN p50=%{y:.0f} ppm<extra></extra>",
        ),
        row=1, col=2, secondary_y=False
    )

    fig_guidance.add_trace(
        go.Scatter(
            x=guidance_plot["cu_regime"],
            y=guidance_plot["recovery_au_pct_p50"],
            mode="lines+markers",
            name="Recovery p50",
            hovertemplate="Regime=%{x}<br>Recovery p50=%{y:.1f}%<extra></extra>",
        ),
        row=1, col=2, secondary_y=True
    )

    fig_guidance.update_yaxes(title_text="Specific NaCN (kg/t)", row=1, col=1)
    fig_guidance.update_yaxes(title_text="Free CN (ppm)", row=1, col=2, secondary_y=False)
    fig_guidance.update_yaxes(title_text="Recovery (%)", row=1, col=2, secondary_y=True)

    fig_guidance.update_layout(
        template="plotly_white",
        height=480,
        title="Guidance profile by copper regime",
        title_x=0.02,
        margin=dict(l=50, r=40, t=70, b=40),
    )
    fig_guidance.show()

# 9E. Efficiency frontier view
eff_required = ["specific_nacn_kgpt", "recovery_au_pct", "cu_solution_ppm_avg"]
if has_cols(efficiency_df, eff_required):
    eff_plot = efficiency_df.copy()

    # Force numeric types for plotted fields
    for c in ["specific_nacn_kgpt", "recovery_au_pct", "cu_solution_ppm_avg", "cn_efficiency_index"]:
        if c in eff_plot.columns:
            eff_plot[c] = pd.to_numeric(eff_plot[c], errors="coerce")

    eff_plot["eff_group"] = "Middle"

    if not best_days.empty:
        eff_plot.loc[eff_plot.index.isin(best_days.head(15).index), "eff_group"] = "Most efficient"
    if not worst_days.empty:
        eff_plot.loc[eff_plot.index.isin(worst_days.head(15).index), "eff_group"] = "Least efficient"

    # Keep only rows that can actually be plotted
    eff_plot = eff_plot.dropna(subset=["specific_nacn_kgpt", "recovery_au_pct"]).copy()

    # Plotly cannot accept NaN marker sizes.
    # Use a safe fallback size when Cu is missing or non-positive.
    eff_plot["cu_solution_ppm_avg_size"] = eff_plot["cu_solution_ppm_avg"].copy()
    eff_plot["cu_solution_ppm_avg_size"] = eff_plot["cu_solution_ppm_avg_size"].replace([np.inf, -np.inf], np.nan)

    if eff_plot["cu_solution_ppm_avg_size"].notna().any():
        fallback_size = max(eff_plot["cu_solution_ppm_avg_size"].median(skipna=True), 1.0)
    else:
        fallback_size = 1.0

    eff_plot["cu_solution_ppm_avg_size"] = eff_plot["cu_solution_ppm_avg_size"].fillna(fallback_size)
    eff_plot.loc[eff_plot["cu_solution_ppm_avg_size"] <= 0, "cu_solution_ppm_avg_size"] = fallback_size

    hover_cols = [c for c in ["date", "free_cn_ppm_avg", "complexed_cn_gpl_avg", "cn_efficiency_index"] if c in eff_plot.columns]

    if not eff_plot.empty:
        fig_eff = px.scatter(
            eff_plot,
            x="specific_nacn_kgpt",
            y="recovery_au_pct",
            color="eff_group",
            size="cu_solution_ppm_avg_size",
            hover_data=hover_cols + ["cu_solution_ppm_avg"],
            title="Efficiency view: recovery vs specific NaCN",
            labels={
                "specific_nacn_kgpt": "Specific NaCN (kg/t)",
                "recovery_au_pct": "Recovery (%)",
                "cu_solution_ppm_avg_size": "Solution Cu (ppm)",
                "cu_solution_ppm_avg": "Solution Cu (ppm)",
            },
        )
        fig_eff.update_layout(
            template="plotly_white",
            height=520,
            title_x=0.02,
            margin=dict(l=50, r=30, t=70, b=50),
        )
        fig_eff.show()

# -----------------------------------------------------------------------------
# 10. NARRATIVE SUMMARY
# -----------------------------------------------------------------------------
narrative_rows = [
    {
        "finding": "The current modelling results are better suited to ranking drivers than to forecasting NaCN input accurately.",
        "evidence": f"Linear R² = {fmt_num(lin_r2)}; RF R² = {fmt_num(rf_r2)}.",
        "implication": "Use this section to support guidance logic and variable prioritisation rather than direct prediction claims."
    },
    {
        "finding": "Dissolved copper remains the most operationally meaningful regime variable.",
        "evidence": "Guidance bands and feature-importance outputs both point to Cu as a primary differentiator where data is available.",
        "implication": "A Cu-regime-based guidance approach remains the most defensible practical pathway at this stage."
    },
    {
        "finding": "The current regime clustering is not yet stable enough to stand alone.",
        "evidence": f"Observed regime counts = {dict(regime_counts.sort_index())}.",
        "implication": "Refine clustering or reduce cluster count before using these labels in a customer-facing operating-mode narrative."
    },
    {
        "finding": "The efficiency ranking needs a guardrail against zero-dose artefacts.",
        "evidence": f"Days with specific NaCN <= 0: {zero_specific_days}.",
        "implication": "Exclude zero-dose or non-credible specific NaCN days before using efficiency rankings in formal recommendations."
    },
]
advanced_narrative = pd.DataFrame(narrative_rows)

print_section("Narrative summary")
safe_show_table(advanced_narrative, "Narrative summary")

# -----------------------------------------------------------------------------
# 11. EXPORT
# -----------------------------------------------------------------------------
model_perf.to_csv(output_dir / "advanced_model_performance.csv", index=False)
best_lags_display.to_csv(output_dir / "advanced_best_lags.csv", index=False)
coef_display.to_csv(output_dir / "advanced_linear_coefficients.csv", index=False)
rf_importance_display.to_csv(output_dir / "advanced_rf_importance.csv", index=False)
regime_counts_display.to_csv(output_dir / "advanced_regime_counts.csv", index=False)
regime_summary_display.to_csv(output_dir / "advanced_regime_summary.csv", index=False)
guidance_bands_display.to_csv(output_dir / "advanced_guidance_bands.csv", index=False)
best_days_display.to_csv(output_dir / "advanced_best_days.csv", index=False)
worst_days_display.to_csv(output_dir / "advanced_worst_days.csv", index=False)
recommendation_table.to_csv(output_dir / "advanced_recommendation_table.csv", index=False)
qa_summary.to_csv(output_dir / "advanced_qa_summary.csv", index=False)
advanced_narrative.to_csv(output_dir / "advanced_narrative_summary.csv", index=False)
executive_summary_advanced.to_csv(output_dir / "advanced_executive_summary.csv", index=False)

with pd.ExcelWriter(output_dir / "advanced_analysis_pack.xlsx", engine="openpyxl") as writer:
    write_excel_sheet(writer, executive_summary_advanced, "Executive Summary")
    write_excel_sheet(writer, qa_summary, "QA Checks")
    write_excel_sheet(writer, model_perf, "Model Performance")
    write_excel_sheet(writer, best_lags_display, "Best Lags")
    write_excel_sheet(writer, coef_display, "Linear Coefficients")
    write_excel_sheet(writer, rf_importance_display, "RF Importance")
    write_excel_sheet(writer, regime_counts_display, "Regime Counts")
    write_excel_sheet(writer, regime_summary_display, "Regime Summary")
    write_excel_sheet(writer, guidance_bands_display, "Guidance Bands")
    write_excel_sheet(writer, best_days_display, "Best Days")
    write_excel_sheet(writer, worst_days_display, "Worst Days")
    write_excel_sheet(writer, recommendation_table, "Recommendations")
    write_excel_sheet(writer, advanced_narrative, "Narrative")

print(f"\nAdvanced analysis outputs saved to: {output_dir.resolve()}")


# =============================================================================
# DAILY REAGENT GUIDANCE TABLE
# =============================================================================

# -----------------------------------------------------------------------------
# 1. PREPARE WORKING DATASET
# -----------------------------------------------------------------------------

if "dfo" not in globals():
    raise RuntimeError(
        "dfo is not defined. Run the upstream data preparation cells first."
    )

guide_df = dfo.copy().sort_values("date").reset_index(drop=True)

# Rebuild / confirm Cu regimes
guide_df["cu_regime"] = pd.qcut(
	guide_df["cu_solution_ppm_avg"],
	q=3,
	labels=["Low Cu", "Medium Cu", "High Cu"],
	duplicates="drop"
)

# Rolling context features
for col in ["cu_solution_ppm_avg", "free_cn_ppm_avg", "wad_gpl_avg", "nacn_consumption_tpd", "specific_nacn_kgpt"]:
	guide_df[f"{col}_roll3"] = guide_df[col].rolling(3, min_periods=1).mean()

# Simple risk flags
guide_df["flag_high_cu"] = guide_df["cu_regime"] == "High Cu"
guide_df["flag_low_free_cn"] = guide_df["free_cn_ppm_avg"] < guide_df["free_cn_ppm_avg"].quantile(0.25)
guide_df["flag_high_complexed_cn"] = guide_df["complexed_cn_gpl_avg"] > guide_df["complexed_cn_gpl_avg"].quantile(0.75)
guide_df["flag_high_specific_nacn"] = guide_df["specific_nacn_kgpt"] > guide_df["specific_nacn_kgpt"].quantile(0.75)
guide_df["flag_low_recovery"] = guide_df["recovery_au_pct"] < guide_df["recovery_au_pct"].quantile(0.25)

# -----------------------------------------------------------------------------
# 2. REGIME-LEVEL GUIDANCE BANDS
# -----------------------------------------------------------------------------
regime_bands = (
	guide_df.groupby("cu_regime", observed=False)
	.agg(
		n_days=("date", "count"),
		mean_solution_cu_ppm=("cu_solution_ppm_avg", "mean"),

		nacn_tpd_p25=("nacn_consumption_tpd", lambda x: x.quantile(0.25)),
		nacn_tpd_p50=("nacn_consumption_tpd", "median"),
		nacn_tpd_p75=("nacn_consumption_tpd", lambda x: x.quantile(0.75)),

		specific_nacn_p25=("specific_nacn_kgpt", lambda x: x.quantile(0.25)),
		specific_nacn_p50=("specific_nacn_kgpt", "median"),
		specific_nacn_p75=("specific_nacn_kgpt", lambda x: x.quantile(0.75)),

		free_cn_p25=("free_cn_ppm_avg", lambda x: x.quantile(0.25)),
		free_cn_p50=("free_cn_ppm_avg", "median"),
		free_cn_p75=("free_cn_ppm_avg", lambda x: x.quantile(0.75)),

		wad_p25=("wad_gpl_avg", lambda x: x.quantile(0.25)),
		wad_p50=("wad_gpl_avg", "median"),
		wad_p75=("wad_gpl_avg", lambda x: x.quantile(0.75)),

		complexed_p25=("complexed_cn_gpl_avg", lambda x: x.quantile(0.25)),
		complexed_p50=("complexed_cn_gpl_avg", "median"),
		complexed_p75=("complexed_cn_gpl_avg", lambda x: x.quantile(0.75)),

		recovery_p25=("recovery_au_pct", lambda x: x.quantile(0.25)),
		recovery_p50=("recovery_au_pct", "median"),
		recovery_p75=("recovery_au_pct", lambda x: x.quantile(0.75)),
	)
	.round(2)
)

print("Regime bands:")
print(regime_bands)

# -----------------------------------------------------------------------------
# 3. PERFORMANCE-WEIGHTED HISTORICAL ANALOGUE ENGINE
# -----------------------------------------------------------------------------
analogue_features = [
    "cu_solution_ppm_avg",
    "throughput_tpd",
    "free_cn_ppm_avg",
    "wad_gpl_avg",
    "ph_tk_8_s",
    "do_avg",
]

base_cols = [
    "date",
    "cu_regime",
    "nacn_consumption_tpd",
    "specific_nacn_kgpt",
    "complexed_cn_gpl_avg",
    "recovery_au_pct",
    "free_cn_ppm_avg",
]

# remove duplicates while preserving order
pool_cols = list(dict.fromkeys(base_cols + analogue_features))

analogue_pool = guide_df[pool_cols].dropna().copy()

# optional debug check
dupes = analogue_pool.columns[analogue_pool.columns.duplicated()].tolist()
print("Duplicate columns:", dupes)

# -------------------------------------------------------------------------
# Derived performance metrics
# -------------------------------------------------------------------------
# Recovery delivered per unit cyanide
analogue_pool["cn_efficiency_score"] = np.where(
    analogue_pool["specific_nacn_kgpt"] > 0,
    analogue_pool["recovery_au_pct"] / analogue_pool["specific_nacn_kgpt"],
    np.nan
)

# Complexation burden relative to free CN
analogue_pool["complexation_ratio"] = np.where(
    analogue_pool["free_cn_ppm_avg"] > 0,
    analogue_pool["complexed_cn_gpl_avg"] / (analogue_pool["free_cn_ppm_avg"] / 1000.0),
    np.nan
)

# Standardise analogue input features for distance calculation
feature_means = analogue_pool[analogue_features].mean()
feature_stds = analogue_pool[analogue_features].std().replace(0, np.nan)

for c in analogue_features:
    analogue_pool[f"{c}_z"] = (
        analogue_pool[c] - feature_means[c]
    ) / feature_stds[c]

# Standardise performance variables as well
perf_cols = [
    "recovery_au_pct",
    "specific_nacn_kgpt",
    "complexed_cn_gpl_avg",
    "cn_efficiency_score",
]

perf_means = analogue_pool[perf_cols].mean()
perf_stds = analogue_pool[perf_cols].std().replace(0, np.nan)

for c in perf_cols:
    analogue_pool[f"{c}_z"] = (
        analogue_pool[c] - perf_means[c]
    ) / perf_stds[c]

# Composite "quality" score:
# higher recovery and efficiency are good
# higher specific NaCN and complexed CN are bad
analogue_pool["quality_score"] = (
    + analogue_pool["recovery_au_pct_z"].fillna(0)
    + analogue_pool["cn_efficiency_score_z"].fillna(0)
    - analogue_pool["specific_nacn_kgpt_z"].fillna(0)
    - analogue_pool["complexed_cn_gpl_avg_z"].fillna(0)
)

def get_recommendation_from_analogues(row, pool, n_analogues=30, top_fraction=0.4):
    """
    Find similar prior operating days, then recommend based on the better-performing
    subset of those analogues.

    Step 1: find nearest analogues by standardised Euclidean distance
    Step 2: rank those analogues by quality_score
    Step 3: derive recommendation band from the top-performing subset
    """
    sub = pool[pool["date"] < row["date"]].copy()
    if len(sub) < max(15, n_analogues):
        sub = pool.copy()

    # Prefer same regime where enough history exists
    same_regime = sub[sub["cu_regime"] == row["cu_regime"]].copy()
    if len(same_regime) >= max(15, n_analogues):
        sub = same_regime

    # Cannot score if current row missing key values
    current = {}
    for c in analogue_features:
        if pd.isna(row[c]) or pd.isna(feature_stds[c]) or feature_stds[c] == 0:
            return {
                "analogue_count": 0,
                "recommended_analogue_count": 0,
                "analog_nacn_tpd_p25": np.nan,
                "analog_nacn_tpd_p50": np.nan,
                "analog_nacn_tpd_p75": np.nan,
                "analog_specific_p25": np.nan,
                "analog_specific_p50": np.nan,
                "analog_specific_p75": np.nan,
                "analog_free_cn_p25": np.nan,
                "analog_free_cn_p50": np.nan,
                "analog_free_cn_p75": np.nan,
                "analog_recovery_p50": np.nan,
                "recommended_nacn_tpd_p25": np.nan,
                "recommended_nacn_tpd_p50": np.nan,
                "recommended_nacn_tpd_p75": np.nan,
                "recommended_specific_p25": np.nan,
                "recommended_specific_p50": np.nan,
                "recommended_specific_p75": np.nan,
                "recommended_free_cn_p25": np.nan,
                "recommended_free_cn_p50": np.nan,
                "recommended_free_cn_p75": np.nan,
                "recommended_recovery_p50": np.nan,
                "recommended_complexed_p50": np.nan,
                "recommended_quality_score_p50": np.nan,
            }
        current[c] = (row[c] - feature_means[c]) / feature_stds[c]

    # Distance on operating conditions
    dist = np.zeros(len(sub))
    for c in analogue_features:
        dist += (sub[f"{c}_z"].values - current[c]) ** 2

    sub = sub.copy()
    sub["distance"] = np.sqrt(dist)

    # First cut = most similar days
    nearest = sub.sort_values("distance").head(n_analogues).copy()

    # Rank nearest analogues by combined quality:
    # better quality and closer similarity both matter
    nearest["distance_rank_score"] = 1 - (
        nearest["distance"].rank(method="average", pct=True)
    )
    nearest["quality_rank_score"] = nearest["quality_score"].rank(method="average", pct=True)

    nearest["recommendation_score"] = (
        0.65 * nearest["quality_rank_score"]
        + 0.35 * nearest["distance_rank_score"]
    )

    n_top = max(8, int(np.ceil(len(nearest) * top_fraction)))
    recommended = nearest.sort_values("recommendation_score", ascending=False).head(n_top).copy()

    return {
        "analogue_count": len(nearest),
        "recommended_analogue_count": len(recommended),

        # descriptive analogue range
        "analog_nacn_tpd_p25": nearest["nacn_consumption_tpd"].quantile(0.25),
        "analog_nacn_tpd_p50": nearest["nacn_consumption_tpd"].median(),
        "analog_nacn_tpd_p75": nearest["nacn_consumption_tpd"].quantile(0.75),
        "analog_specific_p25": nearest["specific_nacn_kgpt"].quantile(0.25),
        "analog_specific_p50": nearest["specific_nacn_kgpt"].median(),
        "analog_specific_p75": nearest["specific_nacn_kgpt"].quantile(0.75),
        "analog_free_cn_p25": nearest["free_cn_ppm_avg"].quantile(0.25),
        "analog_free_cn_p50": nearest["free_cn_ppm_avg"].median(),
        "analog_free_cn_p75": nearest["free_cn_ppm_avg"].quantile(0.75),
        "analog_recovery_p50": nearest["recovery_au_pct"].median(),

        # prescriptive recommendation range from best-performing similar days
        "recommended_nacn_tpd_p25": recommended["nacn_consumption_tpd"].quantile(0.25),
        "recommended_nacn_tpd_p50": recommended["nacn_consumption_tpd"].median(),
        "recommended_nacn_tpd_p75": recommended["nacn_consumption_tpd"].quantile(0.75),
        "recommended_specific_p25": recommended["specific_nacn_kgpt"].quantile(0.25),
        "recommended_specific_p50": recommended["specific_nacn_kgpt"].median(),
        "recommended_specific_p75": recommended["specific_nacn_kgpt"].quantile(0.75),
        "recommended_free_cn_p25": recommended["free_cn_ppm_avg"].quantile(0.25),
        "recommended_free_cn_p50": recommended["free_cn_ppm_avg"].median(),
        "recommended_free_cn_p75": recommended["free_cn_ppm_avg"].quantile(0.75),
        "recommended_recovery_p50": recommended["recovery_au_pct"].median(),
        "recommended_complexed_p50": recommended["complexed_cn_gpl_avg"].median(),
        "recommended_quality_score_p50": recommended["quality_score"].median(),
    }

# Apply recommendation logic row by row
analogue_results = []
for _, r in guide_df.iterrows():
    analogue_results.append(
        get_recommendation_from_analogues(
            r,
            analogue_pool,
            n_analogues=30,
            top_fraction=0.4,
        )
    )

analogue_df = pd.DataFrame(analogue_results)
guide_df = pd.concat([guide_df.reset_index(drop=True), analogue_df.reset_index(drop=True)], axis=1)

# -----------------------------------------------------------------------------
# 4. MERGE REGIME BANDS INTO DAILY TABLE
# -----------------------------------------------------------------------------
guide_df = guide_df.merge(
	regime_bands.reset_index(),
	on="cu_regime",
	how="left"
)

# -----------------------------------------------------------------------------
# 4.5. RECOMMENDATION GAP METRICS
# -----------------------------------------------------------------------------
guide_df["cn_efficiency_score"] = np.where(
    guide_df["specific_nacn_kgpt"] > 0,
    guide_df["recovery_au_pct"] / guide_df["specific_nacn_kgpt"],
    np.nan
)

guide_df["complexation_ratio"] = np.where(
    guide_df["free_cn_ppm_avg"] > 0,
    guide_df["complexed_cn_gpl_avg"] / (guide_df["free_cn_ppm_avg"] / 1000.0),
    np.nan
)

guide_df["delta_specific_vs_recommended"] = (
    guide_df["specific_nacn_kgpt"] - guide_df["recommended_specific_p50"]
)

guide_df["delta_free_cn_vs_recommended"] = (
    guide_df["free_cn_ppm_avg"] - guide_df["recommended_free_cn_p50"]
)

guide_df["delta_recovery_vs_recommended"] = (
    guide_df["recovery_au_pct"] - guide_df["recommended_recovery_p50"]
)

guide_df["delta_complexed_vs_recommended"] = (
    guide_df["complexed_cn_gpl_avg"] - guide_df["recommended_complexed_p50"]
)

# -----------------------------------------------------------------------------
# 5. BUILD DAILY GUIDANCE FIELDS
# -----------------------------------------------------------------------------
guide_df["expected_nacn_tpd_band_regime"] = (
	guide_df["nacn_tpd_p25"].round(1).astype(str)
	+ " to "
	+ guide_df["nacn_tpd_p75"].round(1).astype(str)
)

guide_df["expected_specific_nacn_band_regime"] = (
	guide_df["specific_nacn_p25"].round(2).astype(str)
	+ " to "
	+ guide_df["specific_nacn_p75"].round(2).astype(str)
)

guide_df["expected_free_cn_band_regime"] = np.where(
	guide_df["free_cn_p25"].notna() & guide_df["free_cn_p75"].notna(),
	guide_df["free_cn_p25"].round(0).astype("Int64").astype(str)
	+ " to "
	+ guide_df["free_cn_p75"].round(0).astype("Int64").astype(str),
	np.nan
)

guide_df["expected_nacn_tpd_band_analog"] = np.where(
	guide_df["analogue_count"] > 0,
	guide_df["analog_nacn_tpd_p25"].round(1).astype(str)
	+ " to "
	+ guide_df["analog_nacn_tpd_p75"].round(1).astype(str),
	np.nan
)

guide_df["expected_specific_nacn_band_analog"] = np.where(
	guide_df["analogue_count"] > 0,
	guide_df["analog_specific_p25"].round(2).astype(str)
	+ " to "
	+ guide_df["analog_specific_p75"].round(2).astype(str),
	np.nan
)

guide_df["expected_free_cn_band_analog"] = np.where(
	guide_df["analogue_count"] > 0,
	guide_df["analog_free_cn_p25"].round(0).astype("Int64").astype(str)
	+ " to "
	+ guide_df["analog_free_cn_p75"].round(0).astype("Int64").astype(str),
	np.nan
)

# -----------------------------------------------------------------------------
# 6. HIGH-LEVEL GUIDANCE COMMENT
# -----------------------------------------------------------------------------
def build_guidance_comment(row):
	comments = []

	if row["cu_regime"] == "High Cu":
		comments.append("High-Cu regime: expect materially higher cyanide demand and stronger complexation risk.")
	elif row["cu_regime"] == "Medium Cu":
		comments.append("Medium-Cu regime: operate near historical median band and monitor Cu trend closely.")
	else:
		comments.append("Low-Cu regime: lower reagent band may be adequate if free CN remains stable.")

	if row["flag_low_free_cn"]:
		comments.append("Free CN is in the lower historical range.")
	if row["flag_high_complexed_cn"]:
		comments.append("Complexed/WAD cyanide is elevated.")
	if row["flag_high_specific_nacn"]:
		comments.append("Specific NaCN is already in the upper historical range.")
	if row["flag_low_recovery"]:
		comments.append("Recovery is in the lower historical range.")

	if row["flag_high_cu"] and row["flag_low_free_cn"]:
		comments.append("Extra NaCN may be required to maintain free CN, but efficiency should be checked carefully.")
	if row["flag_high_cu"] and row["flag_high_complexed_cn"]:
		comments.append("A meaningful share of added cyanide may be reporting to copper-related complexes.")

	return " ".join(comments)

guide_df["guidance_comment"] = guide_df.apply(build_guidance_comment, axis=1)

# -----------------------------------------------------------------------------
# 7. SIMPLE GUIDANCE STATUS
# -----------------------------------------------------------------------------
def classify_status(row):
	score = 0
	score += int(bool(row["flag_high_cu"]))
	score += int(bool(row["flag_low_free_cn"]))
	score += int(bool(row["flag_high_complexed_cn"]))
	score += int(bool(row["flag_high_specific_nacn"]))
	score += int(bool(row["flag_low_recovery"]))

	if score >= 4:
		return "Critical"
	elif score >= 2:
		return "Watch"
	return "Normal"

guide_df["guidance_status"] = guide_df.apply(classify_status, axis=1)

# -----------------------------------------------------------------------------
# 8. FINAL DAILY GUIDANCE TABLE
# -----------------------------------------------------------------------------
daily_guidance_table = guide_df[[
	"date",
	"cu_regime",
	"guidance_status",
	"throughput_tpd",
	"cu_feed_ppm",
	"cu_solution_ppm_avg",
	"free_cn_ppm_avg",
	"wad_gpl_avg",
	"complexed_cn_gpl_avg",
	"specific_nacn_kgpt",
	"nacn_consumption_tpd",
	"recovery_au_pct",
	"expected_nacn_tpd_band_regime",
	"expected_specific_nacn_band_regime",
	"expected_free_cn_band_regime",
	"analogue_count",
	"expected_nacn_tpd_band_analog",
	"expected_specific_nacn_band_analog",
	"expected_free_cn_band_analog",
	"guidance_comment",
]].copy()

print("Daily guidance table preview:")
print(daily_guidance_table.tail(20))

# -----------------------------------------------------------------------------
# 9. TODAY-LIKE / LATEST-DAY GUIDANCE SNAPSHOT
# -----------------------------------------------------------------------------
latest_guidance = daily_guidance_table.tail(1).copy()

print("\nLatest day guidance snapshot:")
print(latest_guidance.T)

# -----------------------------------------------------------------------------
# 10. SUMMARY BY STATUS
# -----------------------------------------------------------------------------
status_summary = (
	daily_guidance_table.groupby(["cu_regime", "guidance_status"])
	.agg(
		n_days=("date", "count"),
		mean_solution_cu_ppm=("cu_solution_ppm_avg", "mean"),
		mean_specific_nacn=("specific_nacn_kgpt", "mean"),
		mean_free_cn=("free_cn_ppm_avg", "mean"),
		mean_complexed_cn=("complexed_cn_gpl_avg", "mean"),
		mean_recovery=("recovery_au_pct", "mean"),
	)
	.round(2)
)

print("\nGuidance status summary:")
print(status_summary)

# -----------------------------------------------------------------------------
# 11. EXPORT
# -----------------------------------------------------------------------------
output_dir = Path("la_coipa_diagnostics_outputs")
output_dir.mkdir(exist_ok=True)

daily_guidance_table.to_csv(output_dir / "daily_reagent_guidance_table.csv", index=False)
status_summary.to_csv(output_dir / "daily_reagent_guidance_status_summary.csv")
latest_guidance.to_csv(output_dir / "latest_day_guidance_snapshot.csv", index=False)

print(f"\nDaily guidance outputs saved to: {output_dir.resolve()}")


# =============================================================================
# INCREMENTAL REAGENT RESPONSE ANALYSIS
# =============================================================================

# -----------------------------------------------------------------------------
# 1. PREPARE DATA
# -----------------------------------------------------------------------------
resp_df = dfo.copy().sort_values("date").reset_index(drop=True)

# Copper regimes
resp_df["cu_regime"] = pd.qcut(
	resp_df["cu_solution_ppm_avg"],
	q=3,
	labels=["Low Cu", "Medium Cu", "High Cu"],
	duplicates="drop"
)

# NaCN operating bands
resp_df["nacn_band"] = pd.qcut(
	resp_df["specific_nacn_kgpt"],
	q=3,
	labels=["Low NaCN", "Medium NaCN", "High NaCN"],
	duplicates="drop"
)

# -----------------------------------------------------------------------------
# 2. SIMPLE RESPONSE BY Cu REGIME AND NaCN BAND
# -----------------------------------------------------------------------------
response_summary = (
	resp_df.groupby(["cu_regime", "nacn_band"], observed=False)
	.agg(
		n_days=("date", "count"),
		mean_solution_cu_ppm=("cu_solution_ppm_avg", "mean"),
		mean_specific_nacn_kgpt=("specific_nacn_kgpt", "mean"),
		mean_nacn_tpd=("nacn_consumption_tpd", "mean"),
		mean_free_cn_ppm=("free_cn_ppm_avg", "mean"),
		mean_wad_gpl=("wad_gpl_avg", "mean"),
		mean_complexed_cn_gpl=("complexed_cn_gpl_avg", "mean"),
		mean_recovery_pct=("recovery_au_pct", "mean"),
		mean_ph=("ph_tk_8_s", "mean"),
		mean_do=("do_avg", "mean"),
	)
	.round(2)
)

print("Response summary by Cu regime and NaCN band:")
print(response_summary)

# -----------------------------------------------------------------------------
# 3. HIGH-Cu REGIME FOCUS
# -----------------------------------------------------------------------------
high_cu_df = resp_df[resp_df["cu_regime"] == "High Cu"].copy()

if len(high_cu_df) > 0:
	high_cu_response = (
		high_cu_df.groupby("nacn_band", observed=False)
		.agg(
			n_days=("date", "count"),
			mean_specific_nacn_kgpt=("specific_nacn_kgpt", "mean"),
			mean_nacn_tpd=("nacn_consumption_tpd", "mean"),
			mean_solution_cu_ppm=("cu_solution_ppm_avg", "mean"),
			mean_free_cn_ppm=("free_cn_ppm_avg", "mean"),
			mean_wad_gpl=("wad_gpl_avg", "mean"),
			mean_complexed_cn_gpl=("complexed_cn_gpl_avg", "mean"),
			mean_recovery_pct=("recovery_au_pct", "mean"),
		)
		.round(2)
	)
	print("\nHigh-Cu regime response:")
	print(high_cu_response)

	print("\nHigh-Cu regime correlations with specific NaCN:")
	for target in ["free_cn_ppm_avg", "wad_gpl_avg", "complexed_cn_gpl_avg", "recovery_au_pct"]:
		r = high_cu_df["specific_nacn_kgpt"].corr(high_cu_df[target])
		print(f"  specific NaCN vs {target}: r = {r:.3f}")

# -----------------------------------------------------------------------------
# 4. COMPARABLE-DAY ANALYSIS
# -----------------------------------------------------------------------------
# The goal here is not to prove causality.
# It is to test whether, on roughly similar days, higher NaCN is associated
# with better free CN or recovery, or whether it mostly tracks difficult chemistry.
# -----------------------------------------------------------------------------

# Build comparable-day bins
resp_df["cu_bin5"] = pd.qcut(resp_df["cu_solution_ppm_avg"], q=5, duplicates="drop")
resp_df["tp_bin5"] = pd.qcut(resp_df["throughput_tpd"], q=5, duplicates="drop")
resp_df["ph_bin3"] = pd.qcut(resp_df["ph_tk_8_s"], q=3, duplicates="drop")
resp_df["grade_bin3"] = pd.qcut(resp_df["au_feed_gpt"], q=3, duplicates="drop")

comparable_rows = []

for key, sub in resp_df.groupby(["cu_bin5", "tp_bin5", "ph_bin3", "grade_bin3"], observed=False):
	sub = sub.dropna(subset=[
		"specific_nacn_kgpt",
		"free_cn_ppm_avg",
		"wad_gpl_avg",
		"complexed_cn_gpl_avg",
		"recovery_au_pct",
		"cu_solution_ppm_avg",
	]).copy()

	if len(sub) < 8:
		continue

	low_cut = sub["specific_nacn_kgpt"].quantile(1/3)
	high_cut = sub["specific_nacn_kgpt"].quantile(2/3)

	low_sub = sub[sub["specific_nacn_kgpt"] <= low_cut].copy()
	high_sub = sub[sub["specific_nacn_kgpt"] >= high_cut].copy()

	if len(low_sub) < 2 or len(high_sub) < 2:
		continue

	comparable_rows.append({
		"cell_key": str(key),
		"n_total": len(sub),
		"mean_solution_cu_ppm": sub["cu_solution_ppm_avg"].mean(),
		"delta_specific_nacn_kgpt": high_sub["specific_nacn_kgpt"].mean() - low_sub["specific_nacn_kgpt"].mean(),
		"delta_free_cn_ppm": high_sub["free_cn_ppm_avg"].mean() - low_sub["free_cn_ppm_avg"].mean(),
		"delta_wad_gpl": high_sub["wad_gpl_avg"].mean() - low_sub["wad_gpl_avg"].mean(),
		"delta_complexed_cn_gpl": high_sub["complexed_cn_gpl_avg"].mean() - low_sub["complexed_cn_gpl_avg"].mean(),
		"delta_recovery_pct": high_sub["recovery_au_pct"].mean() - low_sub["recovery_au_pct"].mean(),
	})

comparable_day_results = pd.DataFrame(comparable_rows)

print("\nComparable-day incremental response results:")
print(comparable_day_results)

if len(comparable_day_results) > 0:
	comparable_summary = pd.Series({
		"n_comparable_cells": len(comparable_day_results),
		"mean_delta_specific_nacn_kgpt": comparable_day_results["delta_specific_nacn_kgpt"].mean(),
		"median_delta_specific_nacn_kgpt": comparable_day_results["delta_specific_nacn_kgpt"].median(),
		"mean_delta_free_cn_ppm": comparable_day_results["delta_free_cn_ppm"].mean(),
		"median_delta_free_cn_ppm": comparable_day_results["delta_free_cn_ppm"].median(),
		"mean_delta_wad_gpl": comparable_day_results["delta_wad_gpl"].mean(),
		"median_delta_wad_gpl": comparable_day_results["delta_wad_gpl"].median(),
		"mean_delta_complexed_cn_gpl": comparable_day_results["delta_complexed_cn_gpl"].mean(),
		"median_delta_complexed_cn_gpl": comparable_day_results["delta_complexed_cn_gpl"].median(),
		"mean_delta_recovery_pct": comparable_day_results["delta_recovery_pct"].mean(),
		"median_delta_recovery_pct": comparable_day_results["delta_recovery_pct"].median(),
	}).round(3)

	print("\nComparable-day incremental summary:")
	print(comparable_summary)
else:
	comparable_summary = pd.Series(dtype=float)
	print("\nNo comparable-day cells met the minimum sample requirement.")

# -----------------------------------------------------------------------------
# 5. RESPONSE EFFICIENCY FLAGS
# -----------------------------------------------------------------------------
# These are practical diagnostic flags rather than strict statistical claims.
# -----------------------------------------------------------------------------
resp_df["flag_high_nacn_low_free"] = (
	(resp_df["specific_nacn_kgpt"] >= resp_df["specific_nacn_kgpt"].quantile(0.75)) &
	(resp_df["free_cn_ppm_avg"] <= resp_df["free_cn_ppm_avg"].quantile(0.25))
)

resp_df["flag_high_nacn_low_recovery"] = (
	(resp_df["specific_nacn_kgpt"] >= resp_df["specific_nacn_kgpt"].quantile(0.75)) &
	(resp_df["recovery_au_pct"] <= resp_df["recovery_au_pct"].quantile(0.25))
)

resp_df["flag_high_nacn_high_complexed"] = (
	(resp_df["specific_nacn_kgpt"] >= resp_df["specific_nacn_kgpt"].quantile(0.75)) &
	(resp_df["complexed_cn_gpl_avg"] >= resp_df["complexed_cn_gpl_avg"].quantile(0.75))
)

flag_summary = pd.Series({
	"n_high_nacn_low_free": int(resp_df["flag_high_nacn_low_free"].sum()),
	"n_high_nacn_low_recovery": int(resp_df["flag_high_nacn_low_recovery"].sum()),
	"n_high_nacn_high_complexed": int(resp_df["flag_high_nacn_high_complexed"].sum()),
}).astype(int)

print("\nResponse efficiency flag summary:")
print(flag_summary)

flagged_response_days = resp_df[
	resp_df[
		[
			"flag_high_nacn_low_free",
			"flag_high_nacn_low_recovery",
			"flag_high_nacn_high_complexed",
		]
	].any(axis=1)
][[
	"date",
	"cu_regime",
	"throughput_tpd",
	"cu_solution_ppm_avg",
	"specific_nacn_kgpt",
	"nacn_consumption_tpd",
	"free_cn_ppm_avg",
	"wad_gpl_avg",
	"complexed_cn_gpl_avg",
	"recovery_au_pct",
	"flag_high_nacn_low_free",
	"flag_high_nacn_low_recovery",
	"flag_high_nacn_high_complexed",
]].copy()

print("\nFlagged response days preview:")
print(flagged_response_days.head(20))

# -----------------------------------------------------------------------------
# 6. PRACTICAL INTERPRETATION TABLE
# -----------------------------------------------------------------------------
interpretation_rows = []

# Overall response pattern
for regime in ["Low Cu", "Medium Cu", "High Cu"]:
	sub = resp_df[resp_df["cu_regime"] == regime].copy()
	if len(sub) < 10:
		continue

	r_free = sub["specific_nacn_kgpt"].corr(sub["free_cn_ppm_avg"])
	r_comp = sub["specific_nacn_kgpt"].corr(sub["complexed_cn_gpl_avg"])
	r_rec = sub["specific_nacn_kgpt"].corr(sub["recovery_au_pct"])

	interpretation_rows.append({
		"cu_regime": regime,
		"n_days": len(sub),
		"corr_specific_vs_free_cn": round(r_free, 3),
		"corr_specific_vs_complexed_cn": round(r_comp, 3),
		"corr_specific_vs_recovery": round(r_rec, 3),
		"interpretation": (
			"Higher NaCN appears to coincide more with difficult chemistry than with improved outcome."
			if (pd.notna(r_free) and r_free <= 0) and (pd.notna(r_rec) and r_rec <= 0.10)
			else "Higher NaCN may still be associated with improved support variables in this regime."
		)
	})

response_interpretation = pd.DataFrame(interpretation_rows)

print("\nResponse interpretation by regime:")
print(response_interpretation)

# -----------------------------------------------------------------------------
# 7. EXPORT
# -----------------------------------------------------------------------------
output_dir = Path("la_coipa_diagnostics_outputs")
output_dir.mkdir(exist_ok=True)

response_summary.to_csv(output_dir / "incremental_response_summary.csv")
flagged_response_days.to_csv(output_dir / "incremental_response_flagged_days.csv", index=False)
response_interpretation.to_csv(output_dir / "incremental_response_interpretation.csv", index=False)

if len(comparable_day_results) > 0:
	comparable_day_results.to_csv(output_dir / "incremental_response_comparable_days.csv", index=False)
	comparable_summary.to_csv(output_dir / "incremental_response_comparable_summary.csv")

print(f"\nIncremental response outputs saved to: {output_dir.resolve()}")

# =============================================================================
# BUILD A CLEAN V2 MODELLING TABLE
# Purpose:
# 1. Create QA flags rather than silently dropping bad rows
# 2. Separate "raw diagnostics" from "model-ready" data
# 3. Build leakage-safe features for recommendation + forecasting
# =============================================================================

# -----------------------------------------------------------------------------
# 1. START FROM OPERATING DAYS
# -----------------------------------------------------------------------------
model_v2 = dfo.copy().sort_values("date").reset_index(drop=True)

# -----------------------------------------------------------------------------
# 2. BASIC QA / PLAUSIBILITY FLAGS
# -----------------------------------------------------------------------------
# Separate impossible rows from rows that are just less reliable.
# Do not let derived mass-balance diagnostics wipe out the modelling table.
# -----------------------------------------------------------------------------

core_required_cols = [
	"throughput_tpd",
	"nacn_consumption_tpd",
	"specific_nacn_kgpt",
	"cu_solution_ppm_avg",
	"free_cn_ppm_avg",
	"wad_gpl_avg",
	"recovery_au_pct",
]

model_v2["flag_missing_core"] = model_v2[core_required_cols].isna().any(axis=1)
model_v2["flag_nonpositive_throughput"] = model_v2["throughput_tpd"] <= 0

# Near-idle / abnormal throughput days can distort RT and modelling
model_v2["flag_low_throughput_for_model"] = model_v2["throughput_tpd"] < 500

model_v2["flag_negative_nacn"] = model_v2["nacn_consumption_tpd"] < 0
model_v2["flag_negative_specific_nacn"] = model_v2["specific_nacn_kgpt"] < 0
model_v2["flag_negative_free_cn"] = model_v2["free_cn_ppm_avg"] < 0
model_v2["flag_negative_wad"] = model_v2["wad_gpl_avg"] < 0

# Treat missing pH separately from out-of-range pH
model_v2["flag_missing_ph_tk1"] = model_v2["ph_tk_1_s"].isna()
model_v2["flag_missing_ph_tk8"] = model_v2["ph_tk_8_s"].isna()

model_v2["flag_bad_ph_tk1"] = (
	model_v2["ph_tk_1_s"].notna() &
	~model_v2["ph_tk_1_s"].between(7, 13, inclusive="both")
)
model_v2["flag_bad_ph_tk8"] = (
	model_v2["ph_tk_8_s"].notna() &
	~model_v2["ph_tk_8_s"].between(7, 13, inclusive="both")
)

# Allow tiny floating-point noise around zero / 100
model_v2["recovery_au_pct"] = model_v2["recovery_au_pct"].clip(lower=0, upper=100)
model_v2["flag_bad_recovery"] = (
	model_v2["recovery_au_pct"].notna() &
	~model_v2["recovery_au_pct"].between(-0.01, 100.01, inclusive="both")
)

model_v2["flag_free_gt_wad"] = (model_v2["free_cn_ppm_avg"] / 1000) > model_v2["wad_gpl_avg"]
model_v2["flag_negative_complexed"] = model_v2["complexed_cn_gpl_avg"] < 0

# Residence-time outliers caused by very low throughput
model_v2["flag_extreme_rt_est"] = model_v2["rt_hours_est"] > 200

# Keep derived accountability flags as diagnostics only
if "wad_cn_accountability_pct" in model_v2.columns:
	model_v2["flag_high_wad_accountability"] = model_v2["wad_cn_accountability_pct"] > 100
else:
	model_v2["flag_high_wad_accountability"] = False

if "cu_solution_fraction_pct_est" in model_v2.columns:
	model_v2["flag_high_cu_solution_fraction"] = model_v2["cu_solution_fraction_pct_est"] > 100
else:
	model_v2["flag_high_cu_solution_fraction"] = False

# -----------------------------------------------------------------------------
# 3. MASS BALANCE DIAGNOSTIC FLAGS
# Keep these as diagnostics only, not absolute truth
# -----------------------------------------------------------------------------
if "wad_cn_accountability_pct" in model_v2.columns:
	model_v2["flag_high_wad_accountability"] = model_v2["wad_cn_accountability_pct"] > 100
else:
	model_v2["flag_high_wad_accountability"] = False

if "cu_solution_fraction_pct_est" in model_v2.columns:
	model_v2["flag_high_cu_solution_fraction"] = model_v2["cu_solution_fraction_pct_est"] > 100
else:
	model_v2["flag_high_cu_solution_fraction"] = False

# -----------------------------------------------------------------------------
# 4. COMBINE QA FLAGS
# -----------------------------------------------------------------------------
qa_flag_cols = [c for c in model_v2.columns if c.startswith("flag_")]

model_v2["n_qa_flags"] = model_v2[qa_flag_cols].sum(axis=1)
model_v2["flag_any_qa_issue"] = model_v2["n_qa_flags"] > 0

def classify_qa_status(row):
	# Hard exclusions: impossible or incomplete for modelling
	if row["flag_missing_core"]:
		return "exclude"
	if row["flag_nonpositive_throughput"]:
		return "exclude"
	if row["flag_negative_nacn"] or row["flag_negative_specific_nacn"]:
		return "exclude"
	if row["flag_negative_free_cn"] or row["flag_negative_wad"]:
		return "exclude"
	if row["flag_free_gt_wad"] or row["flag_negative_complexed"]:
		return "exclude"
	if row["flag_bad_ph_tk1"] or row["flag_bad_ph_tk8"]:
		return "exclude"
	if row["flag_bad_recovery"]:
		return "exclude"

	# Review rows: usable for diagnostics, but weaker for forecasting
	if row["flag_low_throughput_for_model"]:
		return "review"
	if row["flag_missing_ph_tk1"] or row["flag_missing_ph_tk8"]:
		return "review"
	if row["flag_extreme_rt_est"]:
		return "review"

	# Accountability flags remain diagnostic only
	return "ok"

model_v2["qa_status"] = model_v2.apply(classify_qa_status, axis=1)

# -----------------------------------------------------------------------------
# 5. CREATE CLEAN MODEL DATASET
# -----------------------------------------------------------------------------
# "ok" = clean enough for primary modelling
# "review" = may still be useful for recommendation context / diagnostics
# -----------------------------------------------------------------------------
model_clean = model_v2[model_v2["qa_status"] == "ok"].copy().reset_index(drop=True)
model_review = model_v2[model_v2["qa_status"].isin(["ok", "review"])].copy().reset_index(drop=True)

print("QA status counts:")
print(model_v2["qa_status"].value_counts(dropna=False))

print("\nExcluded rows preview:")
print(
	model_v2.loc[model_v2["qa_status"] == "exclude", ["date", "qa_status", "n_qa_flags"] + qa_flag_cols]
	.head(20)
)

# -----------------------------------------------------------------------------
# 6. BUILD MODELLING TARGETS
# -----------------------------------------------------------------------------
# Keep targets distinct:
# - reagent addition
# - free CN achievement
# - complexation burden
# - metallurgical outcome
# -----------------------------------------------------------------------------
for df_ in [model_clean, model_review]:
	df_["target_nacn_tpd"] = df_["nacn_consumption_tpd"]
	df_["target_specific_nacn"] = df_["specific_nacn_kgpt"]
	df_["target_free_cn"] = df_["free_cn_ppm_avg"]
	df_["target_complexed_cn"] = df_["complexed_cn_gpl_avg"]
	df_["target_recovery"] = df_["recovery_au_pct"]

# -----------------------------------------------------------------------------
# 7. LEAKAGE-SAFE FEATURES
# Important:
# Use only same-day variables that would plausibly be known at decision time,
# plus lagged / rolling history.
# -----------------------------------------------------------------------------
base_current_features = [
	"throughput_tpd",
	"au_feed_gpt",
	"ag_feed_gpt",
	"cu_feed_ppm",
	"tailings_moisture_pct",
]

# Same-day chemistry features that may or may not be available before dosing.
# Keep them separate so we can test both cases.
same_day_chem_features = [
	"cu_solution_ppm_avg",
	"free_cn_ppm_avg",
	"wad_gpl_avg",
	"complexed_cn_gpl_avg",
	"do_avg",
	"ph_tk_1_s",
	"ph_tk_8_s",
]

lag_feature_sources = [
	"nacn_consumption_tpd",
	"specific_nacn_kgpt",
	"cu_solution_ppm_avg",
	"cu_feed_ppm",
	"free_cn_ppm_avg",
	"wad_gpl_avg",
	"complexed_cn_gpl_avg",
	"recovery_au_pct",
	"do_avg",
	"ph_tk_8_s",
	"throughput_tpd",
	"au_feed_gpt",
	"ag_feed_gpt",
]

rolling_feature_sources = [
	"nacn_consumption_tpd",
	"specific_nacn_kgpt",
	"cu_solution_ppm_avg",
	"free_cn_ppm_avg",
	"wad_gpl_avg",
	"complexed_cn_gpl_avg",
	"recovery_au_pct",
]

def add_time_features(df_in: pd.DataFrame) -> pd.DataFrame:
	df_out = df_in.copy()

	for col in lag_feature_sources:
		if col in df_out.columns:
			for lag in [1, 2, 3, 7]:
				df_out[f"{col}_lag{lag}"] = df_out[col].shift(lag)

	for col in rolling_feature_sources:
		if col in df_out.columns:
			df_out[f"{col}_roll3"] = df_out[col].shift(1).rolling(3, min_periods=1).mean()
			df_out[f"{col}_roll7"] = df_out[col].shift(1).rolling(7, min_periods=1).mean()

	df_out["day_of_week"] = df_out["date"].dt.dayofweek
	df_out["month"] = df_out["date"].dt.month
	df_out["day_of_year"] = df_out["date"].dt.dayofyear

	return df_out

model_clean = add_time_features(model_clean)
model_review = add_time_features(model_review)

# -----------------------------------------------------------------------------
# 8. DEFINE FEATURE SETS FOR DIFFERENT USE CASES
# -----------------------------------------------------------------------------
# A. Pre-dose forecast: only variables known before/at shift start
# B. Same-day guidance: can include current chemistry / assay context
# -----------------------------------------------------------------------------
predose_features = []
for c in (
	base_current_features
	+ [f"{col}_lag1" for col in lag_feature_sources if col in model_clean.columns]
	+ [f"{col}_lag2" for col in lag_feature_sources if col in model_clean.columns]
	+ [f"{col}_lag3" for col in lag_feature_sources if col in model_clean.columns]
	+ [f"{col}_roll3" for col in rolling_feature_sources if col in model_clean.columns]
	+ [f"{col}_roll7" for col in rolling_feature_sources if col in model_clean.columns]
	+ ["day_of_week", "month", "day_of_year"]
):
	if c in model_clean.columns and c not in predose_features:
		predose_features.append(c)

same_day_guidance_features = []
for c in predose_features + same_day_chem_features:
	if c in model_clean.columns and c not in same_day_guidance_features:
		same_day_guidance_features.append(c)

# -----------------------------------------------------------------------------
# 9. REBUILD COPPER REGIMES ON CLEAN DATA
# -----------------------------------------------------------------------------
cu_non_null = model_clean["cu_solution_ppm_avg"].dropna()

if cu_non_null.nunique() < 2:
	model_clean["cu_regime"] = pd.Series(pd.NA, index=model_clean.index, dtype="object")
else:
	_, cu_bins = pd.qcut(
		cu_non_null,
		q=3,
		retbins=True,
		duplicates="drop"
	)

	n_bins = len(cu_bins) - 1
	cu_labels = ["Low Cu", "Medium Cu", "High Cu"][:n_bins]

	if n_bins < 1:
		model_clean["cu_regime"] = pd.Series(pd.NA, index=model_clean.index, dtype="object")
	else:
		model_clean["cu_regime"] = pd.qcut(
			model_clean["cu_solution_ppm_avg"],
			q=3,
			labels=cu_labels,
			duplicates="drop"
		)

# Operating state clustering for diagnostic use
cluster_features = [
	"cu_solution_ppm_avg",
	"free_cn_ppm_avg",
	"wad_gpl_avg",
	"complexed_cn_gpl_avg",
	"specific_nacn_kgpt",
	"recovery_au_pct",
	"do_avg",
	"ph_tk_8_s",
]
cluster_features = [c for c in cluster_features if c in model_clean.columns]

cluster_base = model_clean[cluster_features].dropna().copy()

if len(cluster_base) >= 20:
	from sklearn.preprocessing import StandardScaler
	from sklearn.cluster import KMeans

	scaler = StandardScaler()
	X_cluster = scaler.fit_transform(cluster_base)

	kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
	cluster_labels = kmeans.fit_predict(X_cluster)

	cluster_base["operating_state"] = cluster_labels
	model_clean.loc[cluster_base.index, "operating_state"] = cluster_labels
else:
	model_clean["operating_state"] = np.nan

# -----------------------------------------------------------------------------
# 10. BUILD RECOMMENDATION BANDS ON CLEAN DATA
# -----------------------------------------------------------------------------
recommendation_bands = (
	model_clean.groupby("cu_regime", observed=False)
	.agg(
		n_days=("date", "count"),
		cu_solution_ppm_avg=("cu_solution_ppm_avg", "mean"),
		specific_nacn_p25=("specific_nacn_kgpt", lambda x: x.quantile(0.25)),
		specific_nacn_p50=("specific_nacn_kgpt", "median"),
		specific_nacn_p75=("specific_nacn_kgpt", lambda x: x.quantile(0.75)),
		nacn_tpd_p25=("nacn_consumption_tpd", lambda x: x.quantile(0.25)),
		nacn_tpd_p50=("nacn_consumption_tpd", "median"),
		nacn_tpd_p75=("nacn_consumption_tpd", lambda x: x.quantile(0.75)),
		free_cn_p25=("free_cn_ppm_avg", lambda x: x.quantile(0.25)),
		free_cn_p50=("free_cn_ppm_avg", "median"),
		free_cn_p75=("free_cn_ppm_avg", lambda x: x.quantile(0.75)),
		complexed_cn_p25=("complexed_cn_gpl_avg", lambda x: x.quantile(0.25)),
		complexed_cn_p50=("complexed_cn_gpl_avg", "median"),
		complexed_cn_p75=("complexed_cn_gpl_avg", lambda x: x.quantile(0.75)),
		recovery_p25=("recovery_au_pct", lambda x: x.quantile(0.25)),
		recovery_p50=("recovery_au_pct", "median"),
		recovery_p75=("recovery_au_pct", lambda x: x.quantile(0.75)),
	)
	.round(3)
)

print("\nRecommendation bands:")
print(recommendation_bands)

# -----------------------------------------------------------------------------
# 11. ANALOGUE POOL FOR RECOMMENDATION ENGINE
# Use clean+review if desired later, but start with clean only.
# -----------------------------------------------------------------------------
analogue_features_v2 = [
	"cu_solution_ppm_avg",
	"throughput_tpd",
	"free_cn_ppm_avg",
	"wad_gpl_avg",
	"ph_tk_8_s",
	"do_avg",
]
analogue_features_v2 = [c for c in analogue_features_v2 if c in model_clean.columns]

analogue_output_cols = [
	"analogue_count",
	"analog_specific_p25",
	"analog_specific_p50",
	"analog_specific_p75",
	"analog_nacn_tpd_p25",
	"analog_nacn_tpd_p50",
	"analog_nacn_tpd_p75",
	"analog_free_cn_p25",
	"analog_free_cn_p50",
	"analog_free_cn_p75",
	"analog_complexed_p50",
	"analog_recovery_p50",
]

analogue_pool_v2 = model_clean[
	["date", "cu_regime", "target_nacn_tpd", "target_specific_nacn", "target_free_cn", "target_complexed_cn", "target_recovery"]
	+ analogue_features_v2
].dropna().copy()

if len(model_clean) == 0 or len(analogue_features_v2) == 0 or len(analogue_pool_v2) == 0:
	for col in analogue_output_cols:
		model_clean[col] = np.nan
	model_clean["analogue_count"] = 0

else:
	feature_means_v2 = analogue_pool_v2[analogue_features_v2].mean()
	feature_stds_v2 = analogue_pool_v2[analogue_features_v2].std().replace(0, np.nan)

	for c in analogue_features_v2:
		analogue_pool_v2[f"{c}_z"] = (analogue_pool_v2[c] - feature_means_v2[c]) / feature_stds_v2[c]

	def get_analogue_guidance_v2(row, pool, feature_means, feature_stds, features, n_analogues=20):
		sub = pool[pool["date"] < row["date"]].copy()
		if len(sub) < max(10, n_analogues):
			sub = pool.copy()

		same_regime = sub[sub["cu_regime"] == row["cu_regime"]].copy()
		if len(same_regime) >= max(10, n_analogues):
			sub = same_regime

		current = {}
		for c in features:
			if c not in row.index or pd.isna(row[c]) or pd.isna(feature_stds[c]) or feature_stds[c] == 0:
				return {
					"analogue_count": 0,
					"analog_specific_p25": np.nan,
					"analog_specific_p50": np.nan,
					"analog_specific_p75": np.nan,
					"analog_nacn_tpd_p25": np.nan,
					"analog_nacn_tpd_p50": np.nan,
					"analog_nacn_tpd_p75": np.nan,
					"analog_free_cn_p25": np.nan,
					"analog_free_cn_p50": np.nan,
					"analog_free_cn_p75": np.nan,
					"analog_complexed_p50": np.nan,
					"analog_recovery_p50": np.nan,
				}
			current[c] = (row[c] - feature_means[c]) / feature_stds[c]

		dist = np.zeros(len(sub))
		for c in features:
			dist += (sub[f"{c}_z"].values - current[c]) ** 2

		sub = sub.copy()
		sub["distance"] = np.sqrt(dist)
		sub = sub.sort_values("distance").head(n_analogues)

		return {
			"analogue_count": len(sub),
			"analog_specific_p25": sub["target_specific_nacn"].quantile(0.25),
			"analog_specific_p50": sub["target_specific_nacn"].median(),
			"analog_specific_p75": sub["target_specific_nacn"].quantile(0.75),
			"analog_nacn_tpd_p25": sub["target_nacn_tpd"].quantile(0.25),
			"analog_nacn_tpd_p50": sub["target_nacn_tpd"].median(),
			"analog_nacn_tpd_p75": sub["target_nacn_tpd"].quantile(0.75),
			"analog_free_cn_p25": sub["target_free_cn"].quantile(0.25),
			"analog_free_cn_p50": sub["target_free_cn"].median(),
			"analog_free_cn_p75": sub["target_free_cn"].quantile(0.75),
			"analog_complexed_p50": sub["target_complexed_cn"].median(),
			"analog_recovery_p50": sub["target_recovery"].median(),
		}

	analogue_results_v2 = []
	for _, row in model_clean.iterrows():
		analogue_results_v2.append(
			get_analogue_guidance_v2(
				row=row,
				pool=analogue_pool_v2,
				feature_means=feature_means_v2,
				feature_stds=feature_stds_v2,
				features=analogue_features_v2,
				n_analogues=20,
			)
		)

	analogue_results_v2 = pd.DataFrame(analogue_results_v2, columns=analogue_output_cols)
	model_clean = pd.concat(
		[model_clean.reset_index(drop=True), analogue_results_v2.reset_index(drop=True)],
		axis=1
	)

# -----------------------------------------------------------------------------
# 12. SIMPLE STATUS / INEFFICIENCY FLAGS FOR RECOMMENDATION ENGINE
# -----------------------------------------------------------------------------
model_clean["flag_high_cu_regime"] = model_clean["cu_regime"] == "High Cu"
model_clean["flag_low_free_cn_vs_clean"] = model_clean["free_cn_ppm_avg"] < model_clean["free_cn_ppm_avg"].quantile(0.25)
model_clean["flag_high_complexed_vs_clean"] = model_clean["complexed_cn_gpl_avg"] > model_clean["complexed_cn_gpl_avg"].quantile(0.75)
model_clean["flag_high_specific_nacn_vs_clean"] = model_clean["specific_nacn_kgpt"] > model_clean["specific_nacn_kgpt"].quantile(0.75)
model_clean["flag_low_recovery_vs_clean"] = model_clean["recovery_au_pct"] < model_clean["recovery_au_pct"].quantile(0.25)

def classify_guidance_status_v2(row):
	score = (
		int(bool(row["flag_high_cu_regime"])) +
		int(bool(row["flag_low_free_cn_vs_clean"])) +
		int(bool(row["flag_high_complexed_vs_clean"])) +
		int(bool(row["flag_high_specific_nacn_vs_clean"])) +
		int(bool(row["flag_low_recovery_vs_clean"]))
	)
	if score >= 4:
		return "Critical"
	elif score >= 2:
		return "Watch"
	return "Normal"

model_clean["guidance_status_v2"] = model_clean.apply(classify_guidance_status_v2, axis=1)

# -----------------------------------------------------------------------------
# 13. BUILD EXPORT TABLES
# -----------------------------------------------------------------------------
model_v2_summary = pd.Series({
	"n_rows_raw_operating": len(model_v2),
	"n_rows_ok": (model_v2["qa_status"] == "ok").sum(),
	"n_rows_review": (model_v2["qa_status"] == "review").sum(),
	"n_rows_exclude": (model_v2["qa_status"] == "exclude").sum(),
	"pct_rows_ok": 100 * (model_v2["qa_status"] == "ok").mean(),
	"pct_rows_review": 100 * (model_v2["qa_status"] == "review").mean(),
	"pct_rows_exclude": 100 * (model_v2["qa_status"] == "exclude").mean(),
}).round(2)

print("\nV2 modelling table summary:")
print(model_v2_summary)

guidance_preview_v2 = model_clean[
	[
		"date",
		"cu_regime",
		"guidance_status_v2",
		"throughput_tpd",
		"cu_solution_ppm_avg",
		"specific_nacn_kgpt",
		"free_cn_ppm_avg",
		"complexed_cn_gpl_avg",
		"recovery_au_pct",
		"analogue_count",
		"analog_specific_p25",
		"analog_specific_p50",
		"analog_specific_p75",
		"analog_free_cn_p50",
		"analog_complexed_p50",
		"analog_recovery_p50",
	]
].copy()

print("\nGuidance preview v2:")
print(guidance_preview_v2.tail(20))

# -----------------------------------------------------------------------------
# 14. EXPORT
# -----------------------------------------------------------------------------
output_dir = Path("la_coipa_diagnostics_outputs")
output_dir.mkdir(exist_ok=True)

model_v2.to_csv(output_dir / "model_v2_raw_with_qa_flags.csv", index=False)
model_clean.to_csv(output_dir / "model_v2_clean_for_modelling.csv", index=False)
model_review.to_csv(output_dir / "model_v2_ok_plus_review.csv", index=False)
recommendation_bands.to_csv(output_dir / "model_v2_recommendation_bands.csv")
guidance_preview_v2.to_csv(output_dir / "model_v2_guidance_preview.csv", index=False)
model_v2_summary.to_csv(output_dir / "model_v2_summary.csv")

pd.Series(predose_features, name="predose_features").to_csv(
	output_dir / "model_v2_predose_features.csv", index=False
)
pd.Series(same_day_guidance_features, name="same_day_guidance_features").to_csv(
	output_dir / "model_v2_same_day_guidance_features.csv", index=False
)

print(f"\nV2 modelling outputs saved to: {output_dir.resolve()}")

# =============================================================================
# FREE-CN SUPPORT GUIDANCE TABLE
# =============================================================================

# -----------------------------------------------------------------------------
# 1. PREPARE DATA
# -----------------------------------------------------------------------------
support_df = dfo.copy().sort_values("date").reset_index(drop=True)

# Cu regimes
support_df["cu_regime"] = pd.qcut(
	support_df["cu_solution_ppm_avg"],
	q=3,
	labels=["Low Cu", "Medium Cu", "High Cu"],
	duplicates="drop"
)

# Free CN regime bands within each Cu regime
def assign_free_cn_status(group):
	group = group.copy()
	q25 = group["free_cn_ppm_avg"].quantile(0.25)
	q75 = group["free_cn_ppm_avg"].quantile(0.75)

	def classify(x):
		if pd.isna(x):
			return np.nan
		if x < q25:
			return "Low Free CN"
		elif x > q75:
			return "High Free CN"
		return "Normal Free CN"

	group["free_cn_status"] = group["free_cn_ppm_avg"].apply(classify)
	group["free_cn_q25_regime"] = q25
	group["free_cn_q75_regime"] = q75
	return group

support_df = (
	support_df.groupby("cu_regime", group_keys=False, observed=False)
	.apply(assign_free_cn_status)
	.reset_index(drop=True)
)

# Additional context flags
support_df["flag_high_complexed"] = support_df["complexed_cn_gpl_avg"] >= support_df["complexed_cn_gpl_avg"].quantile(0.75)
support_df["flag_high_specific_nacn"] = support_df["specific_nacn_kgpt"] >= support_df["specific_nacn_kgpt"].quantile(0.75)
support_df["flag_low_recovery"] = support_df["recovery_au_pct"] <= support_df["recovery_au_pct"].quantile(0.25)
support_df["flag_high_cu"] = support_df["cu_regime"] == "High Cu"

# Rolling context
for col in ["cu_solution_ppm_avg", "free_cn_ppm_avg", "specific_nacn_kgpt", "wad_gpl_avg", "complexed_cn_gpl_avg"]:
	support_df[f"{col}_roll3"] = support_df[col].rolling(3, min_periods=1).mean()

# -----------------------------------------------------------------------------
# 2. REGIME-LEVEL SUPPORT BANDS
# -----------------------------------------------------------------------------
support_bands = (
	support_df.groupby("cu_regime", observed=False)
	.agg(
		n_days=("date", "count"),
		mean_solution_cu_ppm=("cu_solution_ppm_avg", "mean"),

		free_cn_p25=("free_cn_ppm_avg", lambda x: x.quantile(0.25)),
		free_cn_p50=("free_cn_ppm_avg", "median"),
		free_cn_p75=("free_cn_ppm_avg", lambda x: x.quantile(0.75)),

		nacn_tpd_p25=("nacn_consumption_tpd", lambda x: x.quantile(0.25)),
		nacn_tpd_p50=("nacn_consumption_tpd", "median"),
		nacn_tpd_p75=("nacn_consumption_tpd", lambda x: x.quantile(0.75)),

		specific_p25=("specific_nacn_kgpt", lambda x: x.quantile(0.25)),
		specific_p50=("specific_nacn_kgpt", "median"),
		specific_p75=("specific_nacn_kgpt", lambda x: x.quantile(0.75)),

		wad_p50=("wad_gpl_avg", "median"),
		complexed_p50=("complexed_cn_gpl_avg", "median"),
		recovery_p50=("recovery_au_pct", "median"),
	)
	.round(2)
)

print("Support bands by Cu regime:")
print(support_bands)

# -----------------------------------------------------------------------------
# 3. DEFINE "SUPPORTED" HISTORICAL DAYS
# -----------------------------------------------------------------------------
# A day is treated as historically "supported" if:
# - Free CN is not in the bottom quartile of its Cu regime
# - Recovery is not in the bottom quartile overall
# This is a simple pragmatic filter, not a causal claim.
# -----------------------------------------------------------------------------
recovery_q25 = support_df["recovery_au_pct"].quantile(0.25)

support_df["is_supported_day"] = (
	(support_df["free_cn_status"] != "Low Free CN") &
	(support_df["recovery_au_pct"] > recovery_q25)
)

print("\nSupported-day counts:")
print(support_df["is_supported_day"].value_counts(dropna=False))

# -----------------------------------------------------------------------------
# 4. ANALOGUE SUPPORT FUNCTION
# -----------------------------------------------------------------------------
support_features = [
	"cu_solution_ppm_avg",
	"throughput_tpd",
	"ph_tk_8_s",
	"do_avg",
	"au_feed_gpt",
	"free_cn_ppm_avg_roll3",
	"wad_gpl_avg",
]

support_pool = support_df[
	[
		"date",
		"cu_regime",
		"is_supported_day",
		"nacn_consumption_tpd",
		"specific_nacn_kgpt",
		"free_cn_ppm_avg",
		"complexed_cn_gpl_avg",
		"recovery_au_pct",
	] + support_features
].dropna().copy()

feature_means = support_pool[support_features].mean()
feature_stds = support_pool[support_features].std().replace(0, np.nan)

for c in support_features:
	support_pool[f"{c}_z"] = (support_pool[c] - feature_means[c]) / feature_stds[c]

def get_support_guidance(row, pool, n_analogues=20):
	sub = pool[pool["date"] < row["date"]].copy()
	if len(sub) < max(10, n_analogues):
		sub = pool.copy()

	# Prefer same Cu regime
	same_regime = sub[sub["cu_regime"] == row["cu_regime"]].copy()
	if len(same_regime) >= max(10, n_analogues):
		sub = same_regime

	# Strongly prefer historically supported days
	supported = sub[sub["is_supported_day"]].copy()
	if len(supported) >= max(8, int(n_analogues * 0.6)):
		sub = supported

	current = {}
	for c in support_features:
		if pd.isna(row[c]) or pd.isna(feature_stds[c]) or feature_stds[c] == 0:
			return {
				"support_analogue_count": 0,
				"support_nacn_tpd_p25": np.nan,
				"support_nacn_tpd_p50": np.nan,
				"support_nacn_tpd_p75": np.nan,
				"support_specific_p25": np.nan,
				"support_specific_p50": np.nan,
				"support_specific_p75": np.nan,
				"support_free_cn_p50": np.nan,
				"support_recovery_p50": np.nan,
			}
		current[c] = (row[c] - feature_means[c]) / feature_stds[c]

	dist = np.zeros(len(sub))
	for c in support_features:
		dist += (sub[f"{c}_z"].values - current[c]) ** 2

	sub = sub.copy()
	sub["distance"] = np.sqrt(dist)
	sub = sub.sort_values("distance").head(n_analogues)

	return {
		"support_analogue_count": len(sub),
		"support_nacn_tpd_p25": sub["nacn_consumption_tpd"].quantile(0.25),
		"support_nacn_tpd_p50": sub["nacn_consumption_tpd"].median(),
		"support_nacn_tpd_p75": sub["nacn_consumption_tpd"].quantile(0.75),
		"support_specific_p25": sub["specific_nacn_kgpt"].quantile(0.25),
		"support_specific_p50": sub["specific_nacn_kgpt"].median(),
		"support_specific_p75": sub["specific_nacn_kgpt"].quantile(0.75),
		"support_free_cn_p50": sub["free_cn_ppm_avg"].median(),
		"support_recovery_p50": sub["recovery_au_pct"].median(),
	}

support_results = []
for _, r in support_df.iterrows():
	support_results.append(get_support_guidance(r, support_pool, n_analogues=20))

support_results_df = pd.DataFrame(support_results)
support_df = pd.concat(
	[support_df.reset_index(drop=True), support_results_df.reset_index(drop=True)],
	axis=1
)

# -----------------------------------------------------------------------------
# 5. CURRENT SUPPORT GAP
# -----------------------------------------------------------------------------
# Compare current conditions to the analogue-supported range
# -----------------------------------------------------------------------------
support_df["support_gap_specific_vs_p50"] = support_df["specific_nacn_kgpt"] - support_df["support_specific_p50"]
support_df["support_gap_free_cn_vs_p50"] = support_df["free_cn_ppm_avg"] - support_df["support_free_cn_p50"]

def classify_support_position(row):
	if pd.isna(row["support_specific_p25"]) or pd.isna(row["support_specific_p75"]):
		return "Insufficient analogue data"

	if row["specific_nacn_kgpt"] < row["support_specific_p25"] and row["free_cn_status"] == "Low Free CN":
		return "Potentially under-dosed"
	if row["specific_nacn_kgpt"] > row["support_specific_p75"] and row["flag_high_complexed"]:
		return "High dose / high complexation risk"
	if row["free_cn_status"] == "Normal Free CN":
		return "Within historical support range"
	if row["free_cn_status"] == "High Free CN" and row["specific_nacn_kgpt"] > row["support_specific_p50"]:
		return "Possibly over-supported"
	return "Watch"

support_df["support_position"] = support_df.apply(classify_support_position, axis=1)

# -----------------------------------------------------------------------------
# 6. BUILD TEXT BANDS
# -----------------------------------------------------------------------------
def make_band(a, b, decimals=1):
	if pd.isna(a) or pd.isna(b):
		return np.nan
	return f"{a:.{decimals}f} to {b:.{decimals}f}"

support_df["support_specific_nacn_band"] = support_df.apply(
    lambda r: make_band(r["support_specific_p25"], r["support_specific_p75"], decimals=2),
    axis=1,
)
support_df["support_nacn_tpd_band"] = support_df.apply(
    lambda r: make_band(r["support_nacn_tpd_p25"], r["support_nacn_tpd_p75"], decimals=1),
    axis=1,
)
support_df["regime_free_cn_band"] = support_df.apply(
    lambda r: make_band(r["free_cn_q25_regime"], r["free_cn_q75_regime"], decimals=0),
    axis=1,
)

# -----------------------------------------------------------------------------
# 7. PLAIN-ENGLISH SUPPORT COMMENT
# -----------------------------------------------------------------------------
def build_support_comment(row):
	comments = []

	if row["cu_regime"] == "High Cu":
		comments.append("High-Cu regime: maintaining free CN is likely to be difficult.")
	elif row["cu_regime"] == "Medium Cu":
		comments.append("Medium-Cu regime: free CN support should be monitored closely.")
	else:
		comments.append("Low-Cu regime: free CN is generally easier to support.")

	if row["support_position"] == "Potentially under-dosed":
		comments.append("Current specific NaCN is below the historical supported range while free CN is low.")
	elif row["support_position"] == "High dose / high complexation risk":
		comments.append("Current dosing is already high relative to supported analogues and complexation risk is elevated.")
	elif row["support_position"] == "Possibly over-supported":
		comments.append("Free CN is strong relative to regime history and dosing may be above what was typically required.")
	elif row["support_position"] == "Within historical support range":
		comments.append("Current dosing and free CN sit within the normal historical support envelope.")
	else:
		comments.append("Conditions should be watched; the day is not a clean fit to the historical support envelope.")

	if row["flag_high_complexed"]:
		comments.append("Complexed/WAD cyanide is elevated.")
	if row["flag_low_recovery"]:
		comments.append("Recovery is currently in the lower historical range.")
	if row["support_analogue_count"] < 10:
		comments.append("Analogue support is based on a limited sample.")

	return " ".join(comments)

support_df["support_comment"] = support_df.apply(build_support_comment, axis=1)

# -----------------------------------------------------------------------------
# 8. FINAL FREE-CN SUPPORT TABLE
# -----------------------------------------------------------------------------
free_cn_support_table = support_df[
	[
		"date",
		"cu_regime",
		"free_cn_status",
		"support_position",
		"throughput_tpd",
		"cu_solution_ppm_avg",
		"free_cn_ppm_avg",
		"wad_gpl_avg",
		"complexed_cn_gpl_avg",
		"specific_nacn_kgpt",
		"nacn_consumption_tpd",
		"recovery_au_pct",
		"regime_free_cn_band",
		"support_analogue_count",
		"support_specific_nacn_band",
		"support_nacn_tpd_band",
		"support_free_cn_p50",
		"support_recovery_p50",
		"support_comment",
	]
].copy()

print("\nFree-CN support guidance table preview:")
print(free_cn_support_table.tail(20))

# -----------------------------------------------------------------------------
# 9. LATEST-DAY SNAPSHOT
# -----------------------------------------------------------------------------
latest_support_snapshot = free_cn_support_table.tail(1).copy()

print("\nLatest free-CN support snapshot:")
print(latest_support_snapshot.T)

# -----------------------------------------------------------------------------
# 10. SUPPORT POSITION SUMMARY
# -----------------------------------------------------------------------------
support_position_summary = (
	free_cn_support_table.groupby(["cu_regime", "support_position"])
	.agg(
		n_days=("date", "count"),
		mean_solution_cu_ppm=("cu_solution_ppm_avg", "mean"),
		mean_specific_nacn=("specific_nacn_kgpt", "mean"),
		mean_free_cn=("free_cn_ppm_avg", "mean"),
		mean_complexed_cn=("complexed_cn_gpl_avg", "mean"),
		mean_recovery=("recovery_au_pct", "mean"),
	)
	.round(2)
)

print("\nSupport position summary:")
print(support_position_summary)

# -----------------------------------------------------------------------------
# 11. OPTIONAL: DAYS MOST CLEARLY UNDER-SUPPORTED
# -----------------------------------------------------------------------------
under_supported_days = free_cn_support_table[
	free_cn_support_table["support_position"] == "Potentially under-dosed"
].copy().sort_values(["cu_solution_ppm_avg", "free_cn_ppm_avg"], ascending=[False, True])

print("\nPotentially under-supported days preview:")
print(under_supported_days.head(20))

# -----------------------------------------------------------------------------
# 12. OPTIONAL: DAYS WITH HIGH COMPLEXATION RISK
# -----------------------------------------------------------------------------
complexation_risk_days = free_cn_support_table[
	free_cn_support_table["support_position"] == "High dose / high complexation risk"
].copy().sort_values(["complexed_cn_gpl_avg", "specific_nacn_kgpt"], ascending=[False, False])

print("\nHigh complexation-risk days preview:")
print(complexation_risk_days.head(20))

# -----------------------------------------------------------------------------
# 13. EXPORT
# -----------------------------------------------------------------------------
output_dir = Path("la_coipa_diagnostics_outputs")
output_dir.mkdir(exist_ok=True)

free_cn_support_table.to_csv(output_dir / "free_cn_support_guidance_table.csv", index=False)
latest_support_snapshot.to_csv(output_dir / "latest_free_cn_support_snapshot.csv", index=False)
support_position_summary.to_csv(output_dir / "free_cn_support_position_summary.csv")
under_supported_days.to_csv(output_dir / "free_cn_potentially_under_supported_days.csv", index=False)
complexation_risk_days.to_csv(output_dir / "free_cn_high_complexation_risk_days.csv", index=False)

print(f"\nFree-CN support outputs saved to: {output_dir.resolve()}")


n_days_total                                 456
n_days_operating                             421
date_min                     2025-01-01 00:00:00
date_max                     2026-04-01 00:00:00
throughput_tpd_mean                 11769.798866
nacn_consumption_tpd_mean              24.417401
specific_nacn_kgpt_mean                 2.661847
cu_feed_ppm_mean                      862.923459
cu_solution_ppm_avg_mean              2139.09554
free_cn_ppm_avg_mean                  457.330174
wad_gpl_avg_mean                        7.063712
complexed_cn_gpl_avg_mean               6.606382
recovery_au_pct_mean                   75.153347
dtype: object
      days  throughput_tpd_mean  nacn_tpd_mean  specific_nacn_kgpt_mean  \
year                                                                      
2025   351             11444.17          24.48                     2.42   
2026    70             13402.58          24.08                     3.87   

      cu_feed_ppm_mean  cu_solution_ppm_avg_mean

C:\Users\expg\AppData\Local\Temp\ipykernel_17744\502972788.py:475: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Average tank progression values:


,Au (ppm),Ag (ppm),Cu (ppm),Free CN (ppm),WAD (g/L),pH
TK1-E,1.727,10.207,1848.259,477.433,6.297,11.789
TK1-S,1.825,12.230,1934.173,460.811,6.654,11.733
TK6-S,1.877,16.861,2006.251,481.801,6.799,11.632
TK8-S,1.720,20.127,2030.994,450.909,6.786,11.576



EXECUTIVE SUMMARY
                                                                                        finding                                                                                                    evidence                                                                                                                                            implication
                   Dissolved copper is a stronger indicator of cyanide demand than feed copper.                                                r(solution Cu, NaCN t/d) = 0.65; r(feed Cu, NaCN t/d) = 0.51 Circuit chemistry appears more sensitive to dissolved copper than to feed assay alone. Solution chemistry should be monitored directly where possible.
Higher dissolved copper is strongly associated with more cyanide tied up in WAD/complexed form.                                                                         r(solution Cu, complexed CN) = 0.99                           A large share of added cyanide may be rep

...

C:\Users\expg\AppData\Local\Temp\ipykernel_17744\502972788.py:5131: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.




Comparable-day incremental response results:
                                            cell_key  n_total  \
0  (Interval(3757.406, 6570.0, closed='right'), I...       11   
1  (Interval(3757.406, 6570.0, closed='right'), I...        8   

   mean_solution_cu_ppm  delta_specific_nacn_kgpt  delta_free_cn_ppm  \
0           4195.765097                  2.649175        -337.075223   
1           4454.843750                  2.134861        -220.739583   

   delta_wad_gpl  delta_complexed_cn_gpl  delta_recovery_pct  
0      -2.311496               -1.974420           -0.666106  
1      -0.427515               -0.206775           -5.779265  

Comparable-day incremental summary:
n_comparable_cells                   2.000
mean_delta_specific_nacn_kgpt        2.392
median_delta_specific_nacn_kgpt      2.392
mean_delta_free_cn_ppm            -278.907
median_delta_free_cn_ppm          -278.907
mean_delta_wad_gpl                  -1.370
median_delta_wad_gpl                -1.370
mean_delta_com

C:\Users\expg\AppData\Local\Temp\ipykernel_17744\502972788.py:6011: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



Support bands by Cu regime:
           n_days  mean_solution_cu_ppm  free_cn_p25  free_cn_p50  \
cu_regime                                                           
Low Cu        131                732.19       417.19       531.62   
Medium Cu     131               1722.26       337.14       435.06   
High Cu       132               3949.02       184.67       272.36   

           free_cn_p75  nacn_tpd_p25  nacn_tpd_p50  nacn_tpd_p75  \
cu_regime                                                          
Low Cu          672.14          6.81         12.08         14.78   
Medium Cu       601.62         13.41         22.40         31.21   
High Cu         412.03         23.96         37.46         54.61   

           specific_p25  specific_p50  specific_p75  wad_p50  complexed_p50  \
cu_regime                                                                     
Low Cu             0.76          1.07          1.44     3.31           2.85   
Medium Cu          1.37          2.05          2

C:\Users\expg\AppData\Local\Temp\ipykernel_17744\502972788.py:6289: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

